# GP03 Frozen Evaluation

Inspect the frozen paired evaluation and accepted transport settings.

This source copy comes from **GP03_Held_Out_Evaluation_v1_1.ipynb**. It keeps the study code and removes historical cell outputs and platform metadata. Original receipts and candidate files stay with the sealed run archives. See the [evidence index](../docs/evidence-index.md) for recorded outcomes and reproduction requirements.

Run the public companion for the self-contained CPU checks. Running these historical cells requires the relevant Kaggle inputs and the earlier evidence files.

# GP03 held-out evaluation

Wave 1 executes the first five frozen B/C pairs in order. Wave 2 executes the remaining four pairs after the first wave is archived. Each arm uses a fresh official agent sandbox, seed 42 and a shared 15-minute cap. The local model server is started once per wave and shut down when the wave ends or holds.

1. Use **L4 ×4**, **Internet Off**, the same competition input, organizer wheelhouse and Gemma 4 31B QAT model. Make the accepted `GP03_Official_Worker_Acceptance_20261109T093704Z_4c374696.zip` and the original GP02B Rich 2943 ZIP available in `/kaggle/working` or as mounted inputs.
2. Keep `RUN_MODE = "EVAL_GPU"` and `WAVE = 1`. Enter your current **usable L4 wall minutes**. A fresh wave 1 reserves 195 wall minutes, including 45 minutes of overhead. Use your observed quota-to-wall-time conversion.
3. Run All and share the newest wave ZIP. Both arms of five pairs should be archived. The wave report separates recorded attempts, deadline captures and infrastructure holds.

The same notebook supports `RUN_MODE = "GRADE_CPU"` in a CPU session after wave evidence is available. Set `EVALUATION_ZIPS` to the latest sealed ZIP for each wave you want to grade. Protected verification runs in fresh evaluator sandboxes and exports only counts and provenance.

### Revision 1.1

The transport now supplies the frozen 2,048-token limit to ADK summary requests that omit it. Start Wave 1 fresh with `RESUME_WAVE_ZIP = None` and `RETRY_INFRA = []`. The v1.0 archive is retained as infrastructure evidence. Revision 1.1 creates its own checkpoint directory.


## 1. Session settings

In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
import json
from pathlib import Path
import subprocess
import sys
import uuid

RUN_MODE = "EVAL_GPU"  # Set to GRADE_CPU later in a CPU session.
WAVE = 1
L4_WALL_MINUTES_AVAILABLE = None  # Enter usable wall minutes after checking your Kaggle balance.
WORK_DIR = Path("/kaggle/working")
COMPETITION_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WHEELHOUSE_DIR = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
OFFICIAL_ACCEPTANCE_ZIP = None  # Auto-detect the accepted GP03 ZIP.
ORIGINAL_GP02B_ZIP = None  # Auto-detect the original GP02B Rich 2943 ZIP.
WAVE_1_EVIDENCE_ZIP = None  # Required only for wave 2.
RESUME_WAVE_ZIP = None  # Restore the same wave from its sealed evidence in a new session.
RETRY_INFRA = []  # Use an exact task:arm key only after an approved infrastructure hold.
EVALUATION_ZIPS = []  # Used in GRADE_CPU mode.
ALLOW_OFFLINE_WHEEL_INSTALL = True
AUTO_DOWNLOAD_ZIP = True

tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
GP03_SOURCE = WORK_DIR / (".gp03_eval_source_" + tag)
GP03_SOURCE.mkdir(parents=True, exist_ok=False)


## 2. Frozen evaluation manifest

In [ ]:
GP03_FREEZE_TEXT = r'''{
  "accepted_source_pins": {
    "adapter_recipe_sha256": "ac01106525c4d97767b950cf565de028526f9934e93375e2d6ab981b0863bbca",
    "eligible_pool_sha256": "cba7da409788164e4f559f74d325088333c033db57b2739adf8b86955d68b541",
    "frozen_submission_archive_sha256": "9d0021324097f386a13f88bbce1e5344afa3e2a3244a75c28b7791d9efcd0b3e",
    "paired_cpu_evidence_sha256": "e702d7c67556a324614a979dab96e0143762a1c54b95b8ffe810cf0eb7f7b7c9",
    "paired_cpu_run": "GP03_Paired_Runner_CPU_20261109T081452Z_b22e3ff3",
    "source_sha256": {
      "gp03_frozen_agent_parts.py": "b7df4deee768e6ca7eecbf69eeced2922016052a8d75ed0c6c59a2a710cebb36",
      "gp03_frozen_helpers.py": "64fe34f476b0ea171f87b3d15a65c6ecd26112d151cc90f7c9e0291f664acaba",
      "gp03_official_driver.py": "a85079287955c932602369c0fe44a458d13b9d06da1ee7e7481b93755cc15491",
      "gp03_official_hook.py": "8dec5c6d070da4b87003bfb899259417228ae23f6a6ebdf6d86b6f165b85b4c9",
      "gp03_official_worker.py": "7559d5ff106950515d90fbdb7400877b023c91b17232a345566a996cfd4ac9c0",
      "gp03_organizer_runtime.py": "959b928bd43ae1bf1b28899f7e72ca4707da44bbf436c2f3c6163c6e018c8c90",
      "gp03_public_runtime.py": "e8af8493b9784bfbc6ee306a3ef0d1baea55d026f405f079eccb6f2d212b0210",
      "gp03_seed_proxy.py": "14a8c55ba03ddc1a802b9697991e754c9d0032fd07030177889f833c2b1829b0",
      "tool_gate.py": "e40d94ed74de4fdc7814162c4b83a88cf1f9b18f718b40bf4de8fd6d9cdeeb5b"
    }
  },
  "allocation": {
    "insufficient_balance_policy": "Pause or resume later with the same frozen assignments.",
    "last_reported_quota_hours": 12,
    "live_balance_status": "ENTER_AT_LAUNCH",
    "observed_quota_hours_per_l4_wall_hour": 2,
    "planning_l4_wall_minutes": 360
  },
  "analysis": {
    "applicable_deadline_capture": "Grade and report separately from verified delivery.",
    "bootstrap": {
      "draws": 10000,
      "interval": 0.95,
      "seed": 20261108,
      "stratification": "repository",
      "unit": "pair"
    },
    "empty_or_proven_nonapplicable_candidate": 0,
    "infrastructure_and_missing": "Unresolved. Report complete pairs and assigned-cohort bounds.",
    "no_discordance": "Report one-sided 95% discordance upper bound under independent task groups.",
    "primary": "Difference in independently protected-verified repair proportions C minus B in matched evaluable pairs",
    "secondary": [
      "verified delivery",
      "public gate status",
      "call and wall-time cost",
      "natural pass-to-edit exposure",
      "observed regression"
    ]
  },
  "arms": {
    "B": "trusted final-state submission gate",
    "C": "same gate plus proactive rechecks and feedback after qualifying pass invalidation"
  },
  "budget": {
    "context_tokens": 32768,
    "evaluation_worker_minutes": 270,
    "final_reserve_seconds": 60,
    "max_model_calls": 60,
    "max_output_tokens": 2048,
    "max_tool_calls": 60,
    "ordinary_seconds": 840,
    "overhead_reserve_minutes": 90,
    "reserve_minutes_per_wave": 45,
    "seed": 42,
    "worker_seconds": 900
  },
  "cohort": {
    "cpu_holds": [
      {
        "base_commit": "36fe3f7ca9becca4777861d5e6e625f5a4a37545",
        "reason": "NO_ISSUE_LINKED_TEST",
        "repo": "Textualize/rich",
        "snapshot_bytes": 96523552,
        "status": "CPU_HOLD",
        "task_id": "rich_3944"
      },
      {
        "base_commit": "740ec2787b25e59f7aa9e3df1db09d4489c22718",
        "reason": "OPTIONAL_DEPENDENCIES_UNAVAILABLE",
        "repo": "fastapi/fastapi",
        "snapshot_bytes": 226787652,
        "status": "CPU_HOLD",
        "task_id": "fastapi_14303"
      },
      {
        "base_commit": "0b4d494192de489701d3a2e32acef8fb5d3f042e",
        "reason": "OPTIONAL_DEPENDENCIES_UNAVAILABLE",
        "repo": "psf/requests",
        "snapshot_bytes": 36265394,
        "status": "CPU_HOLD",
        "task_id": "requests_6592"
      },
      {
        "base_commit": "2809954800af15b4027364cebf82390fe23dc194",
        "reason": "NO_ISSUE_LINKED_TEST",
        "repo": "Textualize/rich",
        "snapshot_bytes": 95103715,
        "status": "CPU_HOLD",
        "task_id": "rich_3772"
      }
    ],
    "eligible": 32,
    "eligible_outside_quota_prefix": [
      "fastapi_14258",
      "requests_7309",
      "rich_3470",
      "fastapi_14482",
      "requests_7433",
      "rich_3506",
      "fastapi_14794",
      "requests_7505",
      "rich_3278",
      "fastapi_14458",
      "requests_6644",
      "rich_3953",
      "fastapi_14463",
      "requests_7315",
      "rich_3535",
      "fastapi_15745",
      "requests_6589",
      "rich_3067",
      "fastapi_15763",
      "requests_7328",
      "rich_4070",
      "fastapi_14266",
      "rich_3006"
    ],
    "provisional_candidates": 36,
    "selected": 9
  },
  "effective_server_command": [
    "/usr/bin/python3",
    "-m",
    "vllm.entrypoints.openai.api_server",
    "--model",
    "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2",
    "--host",
    "127.0.0.1",
    "--port",
    "8000",
    "--max-model-len",
    "32768",
    "--dtype",
    "bfloat16",
    "--gpu-memory-utilization",
    "0.9",
    "--tensor-parallel-size",
    "4",
    "--enable-auto-tool-choice",
    "--tool-call-parser",
    "gemma4",
    "--reasoning-parser",
    "gemma4",
    "--reasoning-config",
    "{\"reasoning_start_str\": \"<|channel>\", \"reasoning_end_str\": \"<channel|>\"}",
    "--no-scheduler-reserve-full-isl",
    "--enable-lora",
    "--max-loras",
    "2",
    "--max-lora-rank",
    "8",
    "--lora-modules",
    "main_lora=@frozen_submission/adapters/main_lora",
    "tool_lora=@frozen_submission/adapters/tool_lora",
    "--hf-overrides",
    "{\"architectures\": [\"Gemma4ForCausalLM\"], \"model_type\": \"gemma4\"}"
  ],
  "environment_sha256": "0f53e69268c9834ce589f44b8adf8a27263e8daab2ff90f5e37a333ffa34d772",
  "evaluation_frozen": true,
  "frozen_submission_file_sha256": {
    "adapters/main_lora/adapter_config.json": "75a46da2db7f3c70442e5c728f64059aff52cf64174cee7aeb3a4ec37f6e78fb",
    "adapters/main_lora/adapter_model.safetensors": "dcbedd989af34f5201a39606e0bd4014d351a29162dd96da418782cbbe487ad9",
    "adapters/tool_lora/adapter_config.json": "75a46da2db7f3c70442e5c728f64059aff52cf64174cee7aeb3a4ec37f6e78fb",
    "adapters/tool_lora/adapter_model.safetensors": "dcbedd989af34f5201a39606e0bd4014d351a29162dd96da418782cbbe487ad9",
    "agent.yaml": "68ea9b980ced41266ca1d3e6755742ba0b03254629a24808714b37793476ae33",
    "configs/google_original_sampling.yaml": "3dab0b2506dae34ce92fef7b380d6073729104b23fe2cf66d44ac1d2f213aa6a",
    "configs/sampling.yaml": "7791f472b075c047a93fc9840a89e1d22b67b3a30c9653a1eb442c37a8771c58",
    "eval_config.yaml": "ae4f2b4834dbd27d9c200bc754f23287814dfeb6eff5874876caf6a6878dc293",
    "google_original_agent.yaml": "c7fbbbbdc44778419be8e9f53a85800eeb17d0cf151e0846c46e107401b72034",
    "google_original_eval_config.yaml": "adb486b67535fa59b427b79d44dafc131176f1f1964d184e16046ed520ede02d",
    "prompts/analyzer.md": "c762b062032cc8463015e589ad8b4f0e17296062e1b0ff1e78712eef9f5e2d43",
    "prompts/google_original_system.md": "f1cbf7943ee9687382321b1dfd9cd88581d608c1c8b8e6ea61a763bd1a3234af",
    "prompts/investigation_protocol.md": "a3d1715f37043ee0d32a6ea89a44e9c8f91f9d6b38ebeaf3224594addbace8b4",
    "prompts/system.md": "1f51f10ffa2368603d5a9be34dd3b27f69582097684c333f24160d0a79c2d732",
    "prompts/system_custom_base.md": "05847a49d613c0906381bc922b7dad8bc5122887c243130bf4694cf4e77df4ad",
    "prompts/tools_code_graph.md": "68ebc934263198ddd8f42991cc161a90501473a99d545d5f1d60590ef2093b6b",
    "prompts/tools_code_retrieval.md": "1d66b3771fd4147e468b0d4b127bb587af67b9b5194128dc4103d8adc8db31a8",
    "prompts/tools_execution.md": "e32d28dc835b197122739b76bd0e6113b4cc866d9b99e7878eaab6416c538948",
    "prompts/tools_file_operations.md": "5097b3541ac597f12a061a118b9f5bf8c72f2dcdc7f88812514da96cf1124f92",
    "prompts/tools_testing.md": "87e0550756c2c75fe1ad1420e710266134e3dee3a5027847fd70cddbedecef82",
    "sub_agents/code_analyzer.yaml": "7a764798e36cc68aa38900256245e4aed969b5439b992c7da4105b6ebcb7e62b"
  },
  "frozen_utc": "2026-11-09T11:10:52+00:00",
  "grader": {
    "command_seconds": 300,
    "prior_development_grade_sha256": "627fccdb163dccac8ad320a2427a79c44c62040d16e78d3450a17d2715fdb0f8",
    "runtime_versions": {
      "adk-eval-core": "0.1.0",
      "adk-submission": "0.2.13",
      "google-adk": "1.36.1",
      "httpx": "0.28.1",
      "pydantic": "2.12.3",
      "pytest": "8.4.2",
      "starlette": "0.52.1",
      "swegemma": "0.2.11"
    },
    "source": "Pinned organizer verifier in a fresh evaluator-only CPU sandbox",
    "verify_task_parameters": [
      "docker",
      "config",
      "task",
      "snapshot_path",
      "base_snapshot_path",
      "patch_path",
      "fast_path",
      "agent_patch",
      "agent_error",
      "trace",
      "start_time"
    ],
    "worker_seconds": 360
  },
  "graph_inventory": {
    "fastapi_12942": {
      "edges": 2062,
      "nodes": 3314
    },
    "fastapi_14964": {
      "edges": 2332,
      "nodes": 4115
    },
    "fastapi_15023": {
      "edges": 2390,
      "nodes": 4193
    },
    "requests_7205": {
      "edges": 1826,
      "nodes": 732
    },
    "requests_7427": {
      "edges": 1876,
      "nodes": 753
    },
    "requests_7502": {
      "edges": 1884,
      "nodes": 758
    },
    "rich_2943": {
      "edges": 5556,
      "nodes": 1949
    },
    "rich_3043": {
      "edges": 5486,
      "nodes": 1926
    },
    "rich_3718": {
      "edges": 5668,
      "nodes": 1984
    },
    "rich_4077": {
      "edges": 5776,
      "nodes": 2017
    }
  },
  "hardware": [
    "NVIDIA L4",
    "NVIDIA L4",
    "NVIDIA L4",
    "NVIDIA L4"
  ],
  "held_out_model_calls_at_freeze": 17,
  "model": {
    "files": {
      "chat_template.jinja": {
        "bytes": 16934,
        "sha256": "94899c0f917d93f6fe81c95744d1e8ddab2d21d39228d2e4aec1fb2a25bff413"
      },
      "config.json": {
        "bytes": 18711,
        "sha256": "b100d85e571c25b688e82919d819253063a1defb0e6cee3b9b1c1fbad99bd0c2"
      },
      "generation_config.json": {
        "bytes": 208,
        "sha256": "70110aa886fa7ee25ce47346d102dd29ca5f967500490b4362fc72068e1db70a"
      },
      "model.safetensors": {
        "bytes": 23265352448,
        "sha256": "1b9b1d622a93f02c0d33f98e502f233b5d707443af6ddc464ed0bf5498506c20"
      },
      "processor_config.json": {
        "bytes": 1689,
        "sha256": "32bdf45d2ad4cc29a0822ddd157a182de76644f0419a6228d151495256e9813c"
      },
      "tokenizer.json": {
        "bytes": 32169626,
        "sha256": "cc8d3a0ce36466ccc1278bf987df5f71db1719b9ca6b4118264f45cb627bfe0f"
      },
      "tokenizer_config.json": {
        "bytes": 2741,
        "sha256": "a284d1243b62be31faa9c13e1c28cece940c4abaa7bd9ad87b94f61b40687200"
      }
    },
    "mount": "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2"
  },
  "observed_preflight_seconds": 412.018,
  "observed_startup_seconds": 287.966,
  "official_acceptance": {
    "executed_notebook_sha256": "f41cc2c64f5f9e42ef035592837f61cd670db2ef5393308b156a463f9fd2b232",
    "name": "GP03_Official_Worker_Acceptance_20261109T093704Z_4c374696.zip",
    "sha256": "798d3005be6d519fb2c2449ecf204562c6621ab63e7d66586881d3ef06e8a943"
  },
  "official_sources": {
    "agent_runner": "4d7eccabd203c26f7c39e882684d2fd851f383f6a5c76c4a75b0e559cd5d6a18",
    "context": "272168834495f424dac54cfcc3d470c1b8e42b75a1125e7501586eb5f4fb73c0",
    "subprocess_sandbox": "a26e9265d93f0387dd0a820ea451d1c530884bebca41583738f3ab2cc31df1f2",
    "verifier": "99e2a5fdab2be2b5e0388de29080da9d6e7003cf8b53299cda790464f4ecd2aa"
  },
  "organizer_runtime": {
    "model_calls": 0,
    "status": "RUNTIME_MATCHED",
    "versions": {
      "adk-eval-core": "0.1.0",
      "adk-submission": "0.2.13",
      "compressed-tensors": "0.15.0.1",
      "google-adk": "1.36.1",
      "safetensors": "0.8.0",
      "swegemma": "0.2.11",
      "tokenizers": "0.22.2",
      "torch": "2.10.0+cu128",
      "transformers": "5.13.1",
      "vllm": "0.19.1"
    },
    "wheel_pins": [
      {
        "name": "adk_eval_core-0.1.0-py3-none-any.whl",
        "sha256": "e614e1b2fe8d886a634871e4c57f9887e1cc66e8c27300f8503a43c6e3ca82f3"
      },
      {
        "name": "adk_submission-0.2.13-py3-none-any.whl",
        "sha256": "f156f0bd84580ccb9334726d114d7a5481f16d3a6e2887cc4d276ced0ef8d99d"
      },
      {
        "name": "swegemma-0.2.11-py3-none-any.whl",
        "sha256": "9ad876581f3acde8ec3c369858f0834e2b9f41a3cffce3ca42f09cb090aea9d3"
      },
      {
        "name": "google_adk-1.36.1-py3-none-any.whl",
        "sha256": "1a2f6868c509e3151fb0de3575a7d18b45c338be86f420924dad74e7193631a0"
      },
      {
        "name": "vllm-0.19.1-cp38-abi3-manylinux_2_31_x86_64.whl",
        "sha256": "73c69bafca24fbdad44690c44d6c6446514351e6f3cc9a6057cba05e66051310"
      },
      {
        "name": "transformers-5.13.1-py3-none-any.whl",
        "sha256": "53f0ea8aa397e29244c2377ba981bcaf0c87adcf44fbdd447ef6306522afcacd"
      },
      {
        "name": "tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
        "sha256": "369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67"
      },
      {
        "name": "compressed_tensors-0.15.0.1-py3-none-any.whl",
        "sha256": "e1b1f322e82e475715e242bad46925a304ea8e5c98b5055a15b8eb22fb6bfea9"
      },
      {
        "name": "safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",
        "sha256": "f216afd4ba743e744c86cbd08e7185efa8749e4c905f14e4e977d65576466cb0"
      }
    ]
  },
  "pairs": [
    {
      "arm_order": [
        "B",
        "C"
      ],
      "base_commit": "bc06e4296d588f39f38c7ac2fc96d3913c565962",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/fastapi_14964/CPU_Check_Manifest.json",
      "cpu_check_sha256": "30a05a807d13a9cbf8cb03f6a13f8b55e5c0ea8e5b5194654d8fe9507eac743d",
      "cpu_receipt_origin": "v1.0_retained",
      "cpu_receipt_sha256": "f5aebd1a3823212bb17b7e06b00ef59ef1f594c8de86f4cc82796a923d06e783",
      "expected_collected": 1,
      "has_snapshot_links": false,
      "native_head": "47da909b715240cfe7dc2b04d2994c56c9153b52",
      "native_tree": "cac1a745f81c2ac504ce548da1a4106cc53a5e23",
      "official_manifest_sha256": "30a05a807d13a9cbf8cb03f6a13f8b55e5c0ea8e5b5194654d8fe9507eac743d",
      "pair_index": 1,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_orjson_response_class.py",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        1,
        1,
        0,
        0,
        0
      ],
      "rank_sha256": "0b1c850408e946a63c547c1ae99ce1cfad9505a10ae7ebf51960f3a725266f75",
      "repo": "fastapi/fastapi",
      "seed": 42,
      "snapshot_bytes": 271065089,
      "snapshot_sha256": "f09059d29d925ac0b5ed96ddd8ee164e656cdf164d8ae88a53bae3ff783659e8",
      "task_id": "fastapi_14964",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "037915e5cbb3a2e6c89049373ad88d2a69fb1fe9cae9189105d1d2a8ccd2e6c9"
    },
    {
      "arm_order": [
        "C",
        "B"
      ],
      "base_commit": "661970d171d9c3e12e4c789c4768db647d8c4da0",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/requests_7502/CPU_Check_Manifest.json",
      "cpu_check_sha256": "c6a3ae1855099875f96cfa57f9b5f31a0007b6f7699d8ec3aec2fa064dd2a2cc",
      "cpu_receipt_origin": "v1.1_retry",
      "cpu_receipt_sha256": "93edc64af99aaf183eed4eb79a0198c5b4d646cfd1c57bca0423c72edf2ac57a",
      "expected_collected": 1,
      "has_snapshot_links": true,
      "native_head": "ced47d0b985f67110197b488df11f54e4d355eca",
      "native_tree": "ea72ddd13e602fe8f9d9a5546d1be5f64b46046c",
      "official_manifest_sha256": "c6a3ae1855099875f96cfa57f9b5f31a0007b6f7699d8ec3aec2fa064dd2a2cc",
      "pair_index": 2,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_requests.py::TestRequests::test_basicauth_encodes_byte_strings",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        1,
        1,
        0,
        0,
        0
      ],
      "rank_sha256": "04e9442de4385c144ad4dfa09f1633861723f1211563255f7b9f38956450f8a4",
      "repo": "psf/requests",
      "seed": 42,
      "snapshot_bytes": 37463957,
      "snapshot_sha256": "a7da726a17f54ea4d43d1a21e3e9f3e95bfa2787a845a18e2d30a88fcc1f53fb",
      "task_id": "requests_7502",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "bb45bd1f05182d16e002ac7c38d4b87b2390dbd22a0eb2d5d91b7a89467ca1e7"
    },
    {
      "arm_order": [
        "B",
        "C"
      ],
      "base_commit": "7d36119f8378b36129502c3281a7faffef837e02",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/rich_3718/CPU_Check_Manifest.json",
      "cpu_check_sha256": "7fad2c43829f54b41af538bda13a4c1ad54d06c4e18aac0b40df212598734cb1",
      "cpu_receipt_origin": "v1.1_retry",
      "cpu_receipt_sha256": "dabf2c77ced00da25bdeceb8a876510e67295d1ec2e7c7c7c1fc730a04e4037c",
      "expected_collected": 11,
      "has_snapshot_links": false,
      "native_head": "fef719c6fa6ea0305898a10ca7244b9da200a6cb",
      "native_tree": "e1652a57c5e0929cc91da1f8333834d9ae898515",
      "official_manifest_sha256": "7fad2c43829f54b41af538bda13a4c1ad54d06c4e18aac0b40df212598734cb1",
      "pair_index": 3,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_panel.py",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        11,
        11,
        0,
        0,
        0
      ],
      "rank_sha256": "2a13e31c5f90e29d0762c259ff078c17571643f6db97b63f6ade61ef1cf500bf",
      "repo": "Textualize/rich",
      "seed": 42,
      "snapshot_bytes": 95464718,
      "snapshot_sha256": "f66ac24cfee77115da4b15dfad5e5adbab07faedb53a62693bf16b75c2750eb6",
      "task_id": "rich_3718",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "94e0250a860aeebd85519124551ff9bad512e42b628020ef677696dd5763be13"
    },
    {
      "arm_order": [
        "C",
        "B"
      ],
      "base_commit": "27c0f7e75fc0ce26ca75295d455eb36f0ca033d3",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/fastapi_12942/CPU_Check_Manifest.json",
      "cpu_check_sha256": "b2ceca01de32d22080a3e5ae9b7f6772bc15d5f3871e0535142870e3e9b8efb8",
      "cpu_receipt_origin": "v1.0_retained",
      "cpu_receipt_sha256": "a9f795e8af84f2d11e8bc8d70bc747d5d2166ee8e81747790561dd72cc9e56ab",
      "expected_collected": 38,
      "has_snapshot_links": false,
      "native_head": "8c426cdec82445a3f69c8af6ae991ab26d10ea80",
      "native_tree": "09e60c5d118d3d01fdcc6fdbb970c4b44ef22451",
      "official_manifest_sha256": "b2ceca01de32d22080a3e5ae9b7f6772bc15d5f3871e0535142870e3e9b8efb8",
      "pair_index": 4,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_response_model_as_return_annotation.py",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        38,
        38,
        0,
        0,
        0
      ],
      "rank_sha256": "0c84af5514ba9d8d279adc9713c0966642d5978893e576561125dcf276821336",
      "repo": "fastapi/fastapi",
      "seed": 42,
      "snapshot_bytes": 218214164,
      "snapshot_sha256": "68f9786237a6b5cc811190e95f8d45045d850aa09f77cb8be71e6bfd8e900a2a",
      "task_id": "fastapi_12942",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "5d985207da57cb3a75e9f317b0dc86e48e2dd64e9fb0e89b1afaa04a2e96a0dc"
    },
    {
      "arm_order": [
        "B",
        "C"
      ],
      "base_commit": "1b40fdd004bfc8ba5301bcf8a6908264e9b6b877",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/requests_7205/CPU_Check_Manifest.json",
      "cpu_check_sha256": "ac5b3accf4b666a01fe937fe459a4146d1f9e0c8fffe9eeee416aba8a293039b",
      "cpu_receipt_origin": "v1.1_retry",
      "cpu_receipt_sha256": "7285909ade50c8e6533cb3eb87e640a99141b6c7f6db8d04006b5b155bdde251",
      "expected_collected": 1,
      "has_snapshot_links": true,
      "native_head": "443f0b73558781b7dac0e304ad268a30ee35aaeb",
      "native_tree": "683d44befe0d3519469011c35e46f332eebbc010",
      "official_manifest_sha256": "ac5b3accf4b666a01fe937fe459a4146d1f9e0c8fffe9eeee416aba8a293039b",
      "pair_index": 5,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_requests.py::TestRequests::test_proxy_auth_empty_pass",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        1,
        1,
        0,
        0,
        0
      ],
      "rank_sha256": "0a3fd9629109362be6725a7ea800d14b08d4904a587634ec0494501afa5aa08c",
      "repo": "psf/requests",
      "seed": 42,
      "snapshot_bytes": 37402474,
      "snapshot_sha256": "42dc3d360f842a5c7a66d0008ac05447fcd807dc33ee8cc18d2f2d4876283106",
      "task_id": "requests_7205",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "bdb677433a9a52af01c19bc0d5ccd4feccf6e292ab062a206d1e6a143c06cf90"
    },
    {
      "arm_order": [
        "C",
        "B"
      ],
      "base_commit": "58ac15120fe254886c10d6253667733c83604cb4",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/rich_4077/CPU_Check_Manifest.json",
      "cpu_check_sha256": "202833bc7ea98a1ff967c2480884d15d67bae2adcf682302c58af265f2fde382",
      "cpu_receipt_origin": "v1.1_retry",
      "cpu_receipt_sha256": "58d6eb45f64a93c28491f55510d6df8b4a4f744d0dbc5571fe80de4d8ee58e8e",
      "expected_collected": 3,
      "has_snapshot_links": false,
      "native_head": "11cdcf5c9e81f42d4e9b2e3981bc8f67ed44b18e",
      "native_tree": "bc6ede75faeaee74089fa8f3b91fa5f5e3ac3138",
      "official_manifest_sha256": "202833bc7ea98a1ff967c2480884d15d67bae2adcf682302c58af265f2fde382",
      "pair_index": 6,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_file_proxy.py",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        3,
        3,
        0,
        0,
        0
      ],
      "rank_sha256": "3f0898754663f089d61c6483eda14c56ec4d04eeebda696f098e0b116d1cc39e",
      "repo": "Textualize/rich",
      "seed": 42,
      "snapshot_bytes": 97076310,
      "snapshot_sha256": "96aab6cf7a51b313a5e203b27c7bc4c557b044eaa65f0120fe0c27357fa6ed4f",
      "task_id": "rich_4077",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "37d8ba9baaa9e6ee5d8c252785ce3d3f293e5629b6914045d96dda2f9014c189"
    },
    {
      "arm_order": [
        "B",
        "C"
      ],
      "base_commit": "c0836dc1b7f4ba91e9563c8c4ccd0ebe655bc914",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/fastapi_15023/CPU_Check_Manifest.json",
      "cpu_check_sha256": "a0924d89fb59c4f42653c5d6a52a0a102faf5590ad18cdb322b627f1c907d47b",
      "cpu_receipt_origin": "v1.0_retained",
      "cpu_receipt_sha256": "67d5a51c6a0800c841c6edf7350fd275aef0311e05ef1bf243a6b124fb264609",
      "expected_collected": 7,
      "has_snapshot_links": false,
      "native_head": "f7f72e123b1f425f7aa98a13920a3413856077cf",
      "native_tree": "e677615e75916874f814d2b2499f6efbb22aa00a",
      "official_manifest_sha256": "a0924d89fb59c4f42653c5d6a52a0a102faf5590ad18cdb322b627f1c907d47b",
      "pair_index": 7,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_dependency_after_yield_streaming.py",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        7,
        7,
        0,
        0,
        0
      ],
      "rank_sha256": "1f0a7967c5bceccdcc4f9c8262d571fb6cab317ee2dd029039a33bc8fefc39f9",
      "repo": "fastapi/fastapi",
      "seed": 42,
      "snapshot_bytes": 275792435,
      "snapshot_sha256": "b3873cfe6c959e13ef6d655648ab1cf4931d096af69debf70550ecfdf4146ccf",
      "task_id": "fastapi_15023",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "400e698783ffc697ddd567c0bdd03f9c78c06459a36fd2e0a0115dd6edb3142c"
    },
    {
      "arm_order": [
        "C",
        "B"
      ],
      "base_commit": "b684dcb9bbf3aa557d1238e72062c4a29737dd1c",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/requests_7427/CPU_Check_Manifest.json",
      "cpu_check_sha256": "fecce775e5605870467133259043c81e60ea0039319ddbd30b2f80860b8a2db8",
      "cpu_receipt_origin": "v1.1_retry",
      "cpu_receipt_sha256": "c3017eaa1270aaf077bdedd5e1f256d6ccc486dbc475e84fa38a8397b3528988",
      "expected_collected": 2,
      "has_snapshot_links": true,
      "native_head": "6cd34441464a253437ac6d7846379fd4f945b97b",
      "native_tree": "f89ba9281369ca35eaad0f27853941e129836be2",
      "official_manifest_sha256": "fecce775e5605870467133259043c81e60ea0039319ddbd30b2f80860b8a2db8",
      "pair_index": 8,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_requests.py::TestRequests::test_proxy_authorization_not_appended_to_https_request",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        2,
        2,
        0,
        0,
        0
      ],
      "rank_sha256": "0a4bb2e29eee1a6c909d019abe71822ac222153ad3672536c87c1b9982f364f5",
      "repo": "psf/requests",
      "seed": 42,
      "snapshot_bytes": 37176965,
      "snapshot_sha256": "d8f2092187969feb4f4b32b501682d76c09e27173371b4673c1daceef72f2e04",
      "task_id": "requests_7427",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "ed429cd7e97b0c39c7a39b15e283537d8c1cfa346e67fe819eb8aa37a654a91c"
    },
    {
      "arm_order": [
        "B",
        "C"
      ],
      "base_commit": "8a616de8e25db3c93abaaa220c493d80878de69d",
      "baseline_outcome": "PASS",
      "cpu_check_member": "tasks/rich_3043/CPU_Check_Manifest.json",
      "cpu_check_sha256": "ec7ff0096a38d1d9999f65c3f6ad0e303597365593e5287b691c514b0c13eba9",
      "cpu_receipt_origin": "v1.1_retry",
      "cpu_receipt_sha256": "113364c86485926628f62eb4f50fcd7bf753686959218b25da606cf3dd05615b",
      "expected_collected": 1,
      "has_snapshot_links": false,
      "native_head": "c41239b93ac0e9af2f4e3890dfb807f942b31f89",
      "native_tree": "c2ab870ce396f066664ea9de5678ee71ca24dd92",
      "official_manifest_sha256": "ec7ff0096a38d1d9999f65c3f6ad0e303597365593e5287b691c514b0c13eba9",
      "pair_index": 9,
      "public_check_argv": [
        "/usr/bin/python3",
        "-m",
        "pytest",
        "tests/test_console.py::test_export_html",
        "-q",
        "--tb=short"
      ],
      "public_counts": [
        1,
        1,
        0,
        0,
        0
      ],
      "rank_sha256": "4b5b2a65bb1e9464b6570821066d4291aeb6dd7e4be135c3d7ff298317bb869e",
      "repo": "Textualize/rich",
      "seed": 42,
      "snapshot_bytes": 90818810,
      "snapshot_sha256": "71edf847e2907fcde33ce124f6d57871448c8b55aba059cda75bad05b7653d2d",
      "task_id": "rich_3043",
      "tool_signatures_sha256": "aa62fee12145ba966727620f7b8877b0640052809900ac17743dbf1074ca099a",
      "visible_manifest_sha256": "c16b607989941f6dc9b8f1fc757f2228472ed9a32df2d1c81e381b3a60b9bf48"
    }
  ],
  "replacement": {
    "maximum_per_arm": 1,
    "permitted": "Documented upstream transport infrastructure failure with confirmed cleanup, identical freeze and available reserve.",
    "retention": "Keep the original record and use the one valid replacement in the primary analysis."
  },
  "revision": "1.1",
  "schema": "gp03.held-out-evaluation-freeze.v1",
  "seed_probe": {
    "client_seed": null,
    "event": "VLLM_REQUEST",
    "forwarded_seed": 42,
    "max_output_tokens": 2048,
    "model": "main_lora",
    "request_bytes": 19823,
    "request_sha256": "14839f54c63c5d5cfa70de19f1fd82aec6e16c5f4eaf6513a8ad7d3ecd241240",
    "stream": false
  },
  "server_config": {
    "default_chat_template_kwargs": null,
    "dtype": "bfloat16",
    "enable_auto_tool_choice": true,
    "enable_lora": true,
    "extra_args": [
      "--hf-overrides",
      "{\"architectures\": [\"Gemma4ForCausalLM\"], \"model_type\": \"gemma4\"}"
    ],
    "gpu_memory_utilization": 0.9,
    "health_check_interval": 1.0,
    "host": "127.0.0.1",
    "kv_cache_dtype": null,
    "limit_mm_per_prompt": null,
    "max_lora_rank": 4,
    "max_loras": 2,
    "max_model_len": 32768,
    "model": "/kaggle/input/models/google/gemma-4/other/gemma-4-31b-it-qat-w4a16-ct/2",
    "nvidia_lib_dir": "/usr/local/nvidia/lib64",
    "port": 8000,
    "reasoning_config": null,
    "reasoning_parser": "gemma4",
    "scheduler_reserve_full_isl": false,
    "speculative_config": null,
    "startup_timeout": 1200,
    "tensor_parallel_size": 4,
    "tool_call_parser": "gemma4"
  },
  "source_sha256": {
    "gp03_eval_capture.py": "e9a1e131d3797e108827431cd3d33febb6e3cda25f043786cdc0733995baf206",
    "gp03_eval_freeze.py": "67cb33808f8f657d4dd638847f78594d3eb0d8a4657819af03fbc816b0ad6972",
    "gp03_eval_grade.py": "dc18bfd39df4f5d3df20e143765498be6ed79b298f6baf6650941874011b2ddf",
    "gp03_eval_run.py": "e90e9faeeb74d11118bc63fd680336ff267ad1fea23464a93c1f04c02aed06f0",
    "gp03_eval_score.py": "338928c0df926a616c1ce59f452e3cca8182264c2899649d7a6b0df5875250b6",
    "gp03_frozen_agent_parts.py": "b7df4deee768e6ca7eecbf69eeced2922016052a8d75ed0c6c59a2a710cebb36",
    "gp03_frozen_helpers.py": "64fe34f476b0ea171f87b3d15a65c6ecd26112d151cc90f7c9e0291f664acaba",
    "gp03_grade_runtime.py": "7476368558230512b8169d4af0efe0dbe45dfda51d62b2dc69f098600a5f9af5",
    "gp03_official_driver.py": "a85079287955c932602369c0fe44a458d13b9d06da1ee7e7481b93755cc15491",
    "gp03_official_hook.py": "d629d71ea95dc3210fb2e971dbb7e08c54f0431b6f54e14d0e137c123dbc7188",
    "gp03_official_worker.py": "d26dd1c100c0aed8840262622800887a25fbe569ff497f9a2bfd05dddbe4b026",
    "gp03_organizer_runtime.py": "959b928bd43ae1bf1b28899f7e72ca4707da44bbf436c2f3c6163c6e018c8c90",
    "gp03_private_grader.py": "c51ac126f1ab402302bca4f13801e043a15c32ab105e2121763a8c8ab4b4e426",
    "gp03_public_runtime.py": "e8af8493b9784bfbc6ee306a3ef0d1baea55d026f405f079eccb6f2d212b0210",
    "gp03_seed_proxy.py": "64b47ceb3572a6cae7b1278b31c3a28246acc9e8f12955d2c281ac1be38dbe61",
    "tool_gate.py": "e40d94ed74de4fdc7814162c4b83a88cf1f9b18f718b40bf4de8fd6d9cdeeb5b"
  },
  "tasks_sha256": "e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6",
  "transport_amendment": {
    "correction": "Apply the frozen 2048-token cap when both client cap fields are missing. Reject incompatible explicit caps.",
    "disposition": "Archive the interrupted attempt separately as infrastructure evidence. Start Wave 1 with fresh B and C sandboxes under revision 1.1.",
    "excluded_archive_sha256": "01d8da5864b378cb6fded6f695a3a796d9a506aeb9532d445d8d251535a0b75c",
    "excluded_attempt_candidate_sha256": "da3ebc1c952aa851f398f2790c3747db9181639d1a03dc0e56f352caafd3a07e",
    "excluded_attempt_forwarded_model_calls": 17,
    "excluded_attempt_id": "285da7be533d_fastapi_14964_B_01",
    "failure": "ADK conversation compaction emitted no client token cap. The gateway blocked it before vLLM.",
    "parent_freeze_sha256": "285da7be533d2b36b6e914ef722535021271a7d0ed132a19d6269f2eb3b1ee5e",
    "protected_grades_seen_at_amendment": 0
  },
  "waves": {
    "1": {
      "initial_launch_l4_wall_minutes": 195,
      "pair_positions": [
        1,
        2,
        3,
        4,
        5
      ]
    },
    "2": {
      "initial_launch_l4_wall_minutes": 165,
      "pair_positions": [
        6,
        7,
        8,
        9
      ]
    }
  }
}
'''
GP03_FREEZE_SHA256 = '28b958752eaca3034eaed5d05ce1069f0f356681ca7c731a825e93a44e5268d4'


In [ ]:
if sha256(GP03_FREEZE_TEXT.encode()).hexdigest() != GP03_FREEZE_SHA256:
    raise RuntimeError("The frozen evaluation manifest was edited")
GP03_FREEZE = GP03_SOURCE / "GP03_Evaluation_Freeze_v1_1.json"
GP03_FREEZE.write_bytes(GP03_FREEZE_TEXT.encode())
GP03_SOURCE_PINS = json.loads(GP03_FREEZE_TEXT)["source_sha256"]

def write_gp03_source(line, cell):
    name = line.strip()
    data = (cell.rstrip() + "\n").encode()
    if name not in GP03_SOURCE_PINS or sha256(data).hexdigest() != GP03_SOURCE_PINS[name]:
        raise RuntimeError(f"Frozen GP03 source changed for {name}")
    path = GP03_SOURCE / name
    if path.is_file() and path.read_bytes() == data:
        return
    with path.open("xb") as stream:
        stream.write(data)

get_ipython().register_magic_function(write_gp03_source, magic_kind="cell", magic_name="gp03_source")


## 3. gp03_public_runtime.py

In [ ]:
%%gp03_source gp03_public_runtime.py
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path, PurePosixPath
from typing import Mapping, Sequence
import ast
import csv
import fcntl
import hashlib
import json
import os
import platform
import posixpath
import re
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
import time
import uuid
import xml.etree.ElementTree as ET
import zipfile
from dataclasses import asdict
from typing import Callable

COMPETITION_DIR = Path("/kaggle/input/competitions/gemma-4-developer-agent")
WORK_DIR = Path("/kaggle/working")
CPU_EVIDENCE_ZIP = None  # Set an absolute path only if the original ZIP has been renamed.
AUTO_DOWNLOAD_ZIP = True

from collections.abc import Mapping
import functools
import inspect
import select
import shlex

PUBLIC_ADAPTER_EVIDENCE_ZIP = None  # Set an absolute path only if the original ZIP has been renamed.

GP03_PAIR_RECIPE_SHA256 = 'abac8f45bd12386f79f9059e7b9eb207fe8ae55276788cb494a3cba87de60a27'

ADAPTER_RECIPE_SHA256 = 'ac01106525c4d97767b950cf565de028526f9934e93375e2d6ab981b0863bbca'

ELIGIBLE_POOL_SHA256 = 'cba7da409788164e4f559f74d325088333c033db57b2739adf8b86955d68b541'

GP03_CPU_EVIDENCE = {'cpu_environment_sha256': '0f53e69268c9834ce589f44b8adf8a27263e8daab2ff90f5e37a333ffa34d772',
 'filename': 'GP03_CPU_Qualification_20261108T225859Z_ca48fb6a.zip',
 'gp00_split_sha256': '27b50b7f46c510c55b7b7bb264aba729abba0cc9e53647e596690c94cd79bb09',
 'notebook_recipe_sha256': '0ae4bc06d88271e9b2ccd3c4fcdae182d9b58b07743eb7130cf7daa317a0f750',
 'qualification_run_sha256': '085458dbf8f424ea23dcc57914e4c284c1bcaaf55ac1f8e5c56d678c8717cff9',
 'sha256': '166d33d3e2426c26831b8473c2098ffc6cf990072165d492fe4926784b03cce2',
 'tasks_sha256': 'e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6'}

GP03_PROPOSED_PAIRS = [{'pair_index': 1,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'fastapi_14964',
  'repo': 'fastapi/fastapi',
  'base_commit': 'bc06e4296d588f39f38c7ac2fc96d3913c565962',
  'snapshot_bytes': 271065089,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.0_retained',
  'cpu_check_member': 'tasks/fastapi_14964/CPU_Check_Manifest.json',
  'cpu_check_sha256': '30a05a807d13a9cbf8cb03f6a13f8b55e5c0ea8e5b5194654d8fe9507eac743d',
  'cpu_receipt_sha256': 'f5aebd1a3823212bb17b7e06b00ef59ef1f594c8de86f4cc82796a923d06e783',
  'snapshot_sha256': 'f09059d29d925ac0b5ed96ddd8ee164e656cdf164d8ae88a53bae3ff783659e8',
  'native_head': '47da909b715240cfe7dc2b04d2994c56c9153b52',
  'native_tree': 'cac1a745f81c2ac504ce548da1a4106cc53a5e23',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_orjson_response_class.py',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': False,
  'rank_sha256': '0b1c850408e946a63c547c1ae99ce1cfad9505a10ae7ebf51960f3a725266f75'},
 {'pair_index': 2,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'requests_7502',
  'repo': 'psf/requests',
  'base_commit': '661970d171d9c3e12e4c789c4768db647d8c4da0',
  'snapshot_bytes': 37463957,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/requests_7502/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'c6a3ae1855099875f96cfa57f9b5f31a0007b6f7699d8ec3aec2fa064dd2a2cc',
  'cpu_receipt_sha256': '93edc64af99aaf183eed4eb79a0198c5b4d646cfd1c57bca0423c72edf2ac57a',
  'snapshot_sha256': 'a7da726a17f54ea4d43d1a21e3e9f3e95bfa2787a845a18e2d30a88fcc1f53fb',
  'native_head': 'ced47d0b985f67110197b488df11f54e4d355eca',
  'native_tree': 'ea72ddd13e602fe8f9d9a5546d1be5f64b46046c',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_requests.py::TestRequests::test_basicauth_encodes_byte_strings',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': True,
  'rank_sha256': '04e9442de4385c144ad4dfa09f1633861723f1211563255f7b9f38956450f8a4'},
 {'pair_index': 3,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'rich_3718',
  'repo': 'Textualize/rich',
  'base_commit': '7d36119f8378b36129502c3281a7faffef837e02',
  'snapshot_bytes': 95464718,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/rich_3718/CPU_Check_Manifest.json',
  'cpu_check_sha256': '7fad2c43829f54b41af538bda13a4c1ad54d06c4e18aac0b40df212598734cb1',
  'cpu_receipt_sha256': 'dabf2c77ced00da25bdeceb8a876510e67295d1ec2e7c7c7c1fc730a04e4037c',
  'snapshot_sha256': 'f66ac24cfee77115da4b15dfad5e5adbab07faedb53a62693bf16b75c2750eb6',
  'native_head': 'fef719c6fa6ea0305898a10ca7244b9da200a6cb',
  'native_tree': 'e1652a57c5e0929cc91da1f8333834d9ae898515',
  'public_check_argv': ['/usr/bin/python3', '-m', 'pytest', 'tests/test_panel.py', '-q', '--tb=short'],
  'expected_collected': 11,
  'has_snapshot_links': False,
  'rank_sha256': '2a13e31c5f90e29d0762c259ff078c17571643f6db97b63f6ade61ef1cf500bf'},
 {'pair_index': 4,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'fastapi_12942',
  'repo': 'fastapi/fastapi',
  'base_commit': '27c0f7e75fc0ce26ca75295d455eb36f0ca033d3',
  'snapshot_bytes': 218214164,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.0_retained',
  'cpu_check_member': 'tasks/fastapi_12942/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'b2ceca01de32d22080a3e5ae9b7f6772bc15d5f3871e0535142870e3e9b8efb8',
  'cpu_receipt_sha256': 'a9f795e8af84f2d11e8bc8d70bc747d5d2166ee8e81747790561dd72cc9e56ab',
  'snapshot_sha256': '68f9786237a6b5cc811190e95f8d45045d850aa09f77cb8be71e6bfd8e900a2a',
  'native_head': '8c426cdec82445a3f69c8af6ae991ab26d10ea80',
  'native_tree': '09e60c5d118d3d01fdcc6fdbb970c4b44ef22451',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_response_model_as_return_annotation.py',
                        '-q',
                        '--tb=short'],
  'expected_collected': 38,
  'has_snapshot_links': False,
  'rank_sha256': '0c84af5514ba9d8d279adc9713c0966642d5978893e576561125dcf276821336'},
 {'pair_index': 5,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'requests_7205',
  'repo': 'psf/requests',
  'base_commit': '1b40fdd004bfc8ba5301bcf8a6908264e9b6b877',
  'snapshot_bytes': 37402474,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/requests_7205/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'ac5b3accf4b666a01fe937fe459a4146d1f9e0c8fffe9eeee416aba8a293039b',
  'cpu_receipt_sha256': '7285909ade50c8e6533cb3eb87e640a99141b6c7f6db8d04006b5b155bdde251',
  'snapshot_sha256': '42dc3d360f842a5c7a66d0008ac05447fcd807dc33ee8cc18d2f2d4876283106',
  'native_head': '443f0b73558781b7dac0e304ad268a30ee35aaeb',
  'native_tree': '683d44befe0d3519469011c35e46f332eebbc010',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_requests.py::TestRequests::test_proxy_auth_empty_pass',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': True,
  'rank_sha256': '0a3fd9629109362be6725a7ea800d14b08d4904a587634ec0494501afa5aa08c'},
 {'pair_index': 6,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'rich_4077',
  'repo': 'Textualize/rich',
  'base_commit': '58ac15120fe254886c10d6253667733c83604cb4',
  'snapshot_bytes': 97076310,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/rich_4077/CPU_Check_Manifest.json',
  'cpu_check_sha256': '202833bc7ea98a1ff967c2480884d15d67bae2adcf682302c58af265f2fde382',
  'cpu_receipt_sha256': '58d6eb45f64a93c28491f55510d6df8b4a4f744d0dbc5571fe80de4d8ee58e8e',
  'snapshot_sha256': '96aab6cf7a51b313a5e203b27c7bc4c557b044eaa65f0120fe0c27357fa6ed4f',
  'native_head': '11cdcf5c9e81f42d4e9b2e3981bc8f67ed44b18e',
  'native_tree': 'bc6ede75faeaee74089fa8f3b91fa5f5e3ac3138',
  'public_check_argv': ['/usr/bin/python3', '-m', 'pytest', 'tests/test_file_proxy.py', '-q', '--tb=short'],
  'expected_collected': 3,
  'has_snapshot_links': False,
  'rank_sha256': '3f0898754663f089d61c6483eda14c56ec4d04eeebda696f098e0b116d1cc39e'},
 {'pair_index': 7,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'fastapi_15023',
  'repo': 'fastapi/fastapi',
  'base_commit': 'c0836dc1b7f4ba91e9563c8c4ccd0ebe655bc914',
  'snapshot_bytes': 275792435,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.0_retained',
  'cpu_check_member': 'tasks/fastapi_15023/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'a0924d89fb59c4f42653c5d6a52a0a102faf5590ad18cdb322b627f1c907d47b',
  'cpu_receipt_sha256': '67d5a51c6a0800c841c6edf7350fd275aef0311e05ef1bf243a6b124fb264609',
  'snapshot_sha256': 'b3873cfe6c959e13ef6d655648ab1cf4931d096af69debf70550ecfdf4146ccf',
  'native_head': 'f7f72e123b1f425f7aa98a13920a3413856077cf',
  'native_tree': 'e677615e75916874f814d2b2499f6efbb22aa00a',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_dependency_after_yield_streaming.py',
                        '-q',
                        '--tb=short'],
  'expected_collected': 7,
  'has_snapshot_links': False,
  'rank_sha256': '1f0a7967c5bceccdcc4f9c8262d571fb6cab317ee2dd029039a33bc8fefc39f9'},
 {'pair_index': 8,
  'seed': 42,
  'arm_order': ['C', 'B'],
  'task_id': 'requests_7427',
  'repo': 'psf/requests',
  'base_commit': 'b684dcb9bbf3aa557d1238e72062c4a29737dd1c',
  'snapshot_bytes': 37176965,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/requests_7427/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'fecce775e5605870467133259043c81e60ea0039319ddbd30b2f80860b8a2db8',
  'cpu_receipt_sha256': 'c3017eaa1270aaf077bdedd5e1f256d6ccc486dbc475e84fa38a8397b3528988',
  'snapshot_sha256': 'd8f2092187969feb4f4b32b501682d76c09e27173371b4673c1daceef72f2e04',
  'native_head': '6cd34441464a253437ac6d7846379fd4f945b97b',
  'native_tree': 'f89ba9281369ca35eaad0f27853941e129836be2',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_requests.py::TestRequests::test_proxy_authorization_not_appended_to_https_request',
                        '-q',
                        '--tb=short'],
  'expected_collected': 2,
  'has_snapshot_links': True,
  'rank_sha256': '0a4bb2e29eee1a6c909d019abe71822ac222153ad3672536c87c1b9982f364f5'},
 {'pair_index': 9,
  'seed': 42,
  'arm_order': ['B', 'C'],
  'task_id': 'rich_3043',
  'repo': 'Textualize/rich',
  'base_commit': '8a616de8e25db3c93abaaa220c493d80878de69d',
  'snapshot_bytes': 90818810,
  'baseline_outcome': 'PASS',
  'cpu_receipt_origin': 'v1.1_retry',
  'cpu_check_member': 'tasks/rich_3043/CPU_Check_Manifest.json',
  'cpu_check_sha256': 'ec7ff0096a38d1d9999f65c3f6ad0e303597365593e5287b691c514b0c13eba9',
  'cpu_receipt_sha256': '113364c86485926628f62eb4f50fcd7bf753686959218b25da606cf3dd05615b',
  'snapshot_sha256': '71edf847e2907fcde33ce124f6d57871448c8b55aba059cda75bad05b7653d2d',
  'native_head': 'c41239b93ac0e9af2f4e3890dfb807f942b31f89',
  'native_tree': 'c2ab870ce396f066664ea9de5678ee71ca24dd92',
  'public_check_argv': ['/usr/bin/python3',
                        '-m',
                        'pytest',
                        'tests/test_console.py::test_export_html',
                        '-q',
                        '--tb=short'],
  'expected_collected': 1,
  'has_snapshot_links': False,
  'rank_sha256': '4b5b2a65bb1e9464b6570821066d4291aeb6dd7e4be135c3d7ff298317bb869e'}]

GP03_DEVELOPMENT = [
    ('fastapi_15589', 'fastapi/fastapi', '22b02e26f9e8c7e32bd8266e2b0ebe8bb3a0db2b'),
    ('rich_4079', 'Textualize/rich', '19c67b9a3479841e9133bea94607c89ee931d3fc'),
    ('fastapi_14479', 'fastapi/fastapi', '2b212ddd7604891d42e2ffbafe764c334230bdb7'),
    ('fastapi_5624', 'fastapi/fastapi', 'c6487ed632056e450d844846a1b63be551a3cbc6'),
    ('httpx_3672', 'encode/httpx', '4acf5c2c37714cc63b5cf71b3e284fca83c90311'),
    ('requests_6757', 'psf/requests', '0e322af87745eff34caffe4df68456ebc20d9068'),
    ('rich_2943', 'Textualize/rich', '610fd75c433c3dfcd9c296e8b91363979127b9fa'),
    ('rich_3130', 'Textualize/rich', '720800e6930d85ad027b1e9bd0cbb96b5e994ce3'),
]

GP03_ADAPTER_EVIDENCE_NAME = 'GP03_Public_Adapter_Acceptance_20261109T002001Z_1c73a0b4.zip'
GP03_ADAPTER_EVIDENCE_SHA256 = '76a13ad7a352f9fa4b80142d866e3679a234060195a0643b1b71b6ececb61416'

POLICY = {
    "version": "GP03_CPU_Qualification_v1.1",
    "selector_rule": "public_issue_to_tracked_test_paths_and_symbols_v1",
    "max_selectors": 5,
    "large_file_test_functions": 50,
    "max_collected_nodes": 200,
    "snapshot_bytes_expanded_cap": 6 * 1024**3,
    "task_wall_seconds": 300,
    "collect_wall_seconds": 55,
    "baseline_wall_seconds": 40,
    "final_reserve_seconds": 60,
    "optional_import_fallback": [
        "dirty_equals", "email_validator", "httpbin", "inline_snapshot", "multipart", "orjson",
        "pytest_asyncio", "pytest_httpbin", "pytest_mock", "trustme", "ujson", "uvicorn",
    ],
    "optional_fixture_fallback": ["httpbin", "httpbin_secure"],
}

PACKAGE_LAYOUTS = {
    "fastapi/fastapi": ("fastapi", ("fastapi", "src/fastapi")),
    "Textualize/rich": ("rich", ("rich", "src/rich")),
    "psf/requests": ("requests", ("requests", "src/requests")),
}

EXTRA_DEPENDENCIES = (
    "attrs", "brotli", "charset-normalizer", "cryptography", "dirty-equals",
    "flask", "httpbin", "pytest-httpbin", "pytest-mock", "requests-mock",
    "simplejson", "socksio", "trustme", "uvicorn", "werkzeug",
)

OFFLINE_FLAGS = {
    "CUDA_VISIBLE_DEVICES": "",
    "GIT_CONFIG_NOSYSTEM": "1",
    "GIT_CONFIG_GLOBAL": os.devnull,
    "GIT_OPTIONAL_LOCKS": "0",
    "GIT_TERMINAL_PROMPT": "0",
    "HF_HUB_OFFLINE": "1",
    "PIP_CONFIG_FILE": os.devnull,
    "PIP_NO_INDEX": "1",
    "PIP_NO_INPUT": "1",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTHONHASHSEED": "0",
    "PYTHONNOUSERSITE": "1",
    "TRANSFORMERS_OFFLINE": "1",
}

PUBLIC_TEST = re.compile(r"tests/(?:[A-Za-z0-9_-]+/)*[A-Za-z0-9_-]+\.py")

VALID_ID = re.compile(r"[A-Za-z0-9_]+")

GIT_ID = re.compile(r"[0-9a-f]{40}")

class CPUHold(RuntimeError):
    def __init__(self, reason, detail):
        super().__init__(detail)
        self.reason = reason

TEST_DEPENDENCIES = (
    'pytest', 'pluggy', 'iniconfig', 'packaging', 'pytest-asyncio', 'pytest-cov',
    'pytest-timeout', 'coverage', 'typing-extensions', 'pydantic', 'pydantic-core',
    'typing-inspection', 'annotated-types', 'annotated-doc', 'starlette', 'anyio',
    'httpx', 'httpcore', 'h11', 'h2', 'hpack', 'hyperframe', 'idna', 'sniffio',
    'certifi', 'requests', 'urllib3', 'python-multipart', 'email-validator',
    'markdown-it-py', 'mdurl', 'pygments', 'exceptiongroup', 'trio',
    'fastapi-cli', 'jinja2', 'orjson', 'ujson', 'pyyaml',
)

def canonical_hash(obj: object) -> str:
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=True,
                                      separators=(",", ":"), default=str).encode()).hexdigest()

def environment_fingerprint(config: Mapping[str, object]) -> str:
    """Hash explicit environment identity supplied by the trusted harness.

    In live integration, include installed wheel/source hashes and relevant runtime
    settings, not just package names, and refresh after environment modifications.
    """
    if not config:
        raise ValueError("Nonempty trusted environment manifest required")
    return canonical_hash(dict(config))

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(262144), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(data, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    temporary.replace(path)

def _cancel_process_group(proc: subprocess.Popen, grace: float = 0.15) -> None:
    """Terminate a process GROUP even if its leader has already exited.

    A parent exiting after SIGTERM does not prove children have exited. All
    checks run in their own POSIX session. After a brief grace, SIGKILL is sent
    to the *group*, including remaining descendants, even if leader.poll() != None.
    """
    if os.name == "posix":
        pgid = proc.pid  # subprocess.Popen(start_new_session=True)
        try:
            os.killpg(pgid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=max(grace, 0.02))
        except subprocess.TimeoutExpired:
            pass
        # Crucial: don't return merely because the group LEADER exited.
        # An orphan/daemonized child in this process group may still be alive.
        time.sleep(min(0.04, max(grace, 0.0)))
        try:
            os.killpg(pgid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        try:
            proc.wait(timeout=2)
        except subprocess.TimeoutExpired:
            pass
    else:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=grace)
            except subprocess.TimeoutExpired:
                proc.kill()
                proc.wait(timeout=2)

def load_gp03_inputs(competition, candidates, development, release_sha):
    """Use issue text and metadata from the pinned release, with an explicit projection."""
    if not (competition / "tasks.jsonl").is_file():
        raise CPUHold("TASK_RELEASE_MISSING", f"Mount the Developer Agent input at {competition}")
    if not (competition / "snapshots").is_dir():
        raise CPUHold("SNAPSHOT_MOUNT_MISSING", f"Expected {competition / 'snapshots'}")
    if sha256_file(competition / "tasks.jsonl") != release_sha:
        raise CPUHold("TASK_RELEASE_MISMATCH", "tasks.jsonl SHA-256 differs from the GP00 release")
    index = {r["task_id"]: r for r in candidates}
    groups = {(r["repo"], r["base_commit"]) for r in candidates}
    dev_ids = {r[0] for r in development}
    dev_groups = {(r[1], r[2]) for r in development}
    if len(index) != len(candidates) or len(groups) != len(candidates):
        raise CPUHold("COHORT_DUPLICATE", "Candidate IDs and repository/commit groups must be unique")
    if dev_ids.intersection(index) or dev_groups.intersection(groups):
        raise CPUHold("COHORT_OVERLAP", "Candidate set intersects the development set")
    for task, row in index.items():
        if (not VALID_ID.fullmatch(task) or row["repo"] not in PACKAGE_LAYOUTS
                or not GIT_ID.fullmatch(row["base_commit"]) or row["snapshot_bytes"] <= 0):
            raise CPUHold("COHORT_INVALID", f"Invalid public identity for {task}")

    issues = {}
    with (competition / "tasks.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            raw = json.loads(line)
            task = raw.get("instance_id")
            if task in index:
                pin = index[task]
                if task in issues or (raw.get("repo"), raw.get("base_commit")) != (
                    pin["repo"], pin["base_commit"]
                ):
                    raise CPUHold("TASK_IDENTITY_MISMATCH", f"Release identity differs for {task}")
                issue = raw.get("problem_statement")
                if not isinstance(issue, str) or not issue.strip():
                    raise CPUHold("PUBLIC_ISSUE_MISSING", f"Public issue description missing for {task}")
                issues[task] = issue
            del raw
    if set(issues) != set(index):
        raise CPUHold("CANDIDATE_MISSING_FROM_RELEASE", "A selected GP00 case is missing")
    return issues

def environment_profile():
    if os.name != "posix" or sys.version_info < (3, 12):
        raise CPUHold("RUNTIME_UNSUPPORTED", "Use the Kaggle Linux CPU image with Python 3.12 or later")
    try:
        import pytest
    except ImportError as exc:
        raise CPUHold("PYTEST_UNAVAILABLE", "Use the organizer CPU image with pytest available") from exc
    if shutil.which("git") is None:
        raise CPUHold("GIT_UNAVAILABLE", "The published snapshots require Git")
    packages = {}
    for name in sorted(set(TEST_DEPENDENCIES) | set(EXTRA_DEPENDENCIES)):
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError:
            packages[name] = None
        else:
            record = dist.read_text("RECORD") or dist.read_text("METADATA") or ""
            packages[name] = {"version": dist.version, "record_sha256": hashlib.sha256(record.encode()).hexdigest()}
    return {
        "python_version": platform.python_version(),
        "python_executable": sys.executable,
        "python_executable_sha256": sha256_file(Path(sys.executable).resolve()),
        "machine": platform.machine(),
        "pytest_version": pytest.__version__,
        "test_dependencies": packages,
        "flags": OFFLINE_FLAGS,
    }

def command_env(root, temporary, capture=None, package=None):
    home = temporary / "home"
    home.mkdir(exist_ok=True)
    paths = [root, root / "src"]
    if capture is not None:
        paths.append(temporary)
    env = {
        "PATH": os.environ.get("PATH", os.defpath), "HOME": str(home),
        "LANG": "C.UTF-8", "LC_ALL": "C.UTF-8", "TZ": "UTC",
        "TMPDIR": str(temporary), "TMP": str(temporary), "TEMP": str(temporary),
        "PYTHONPATH": os.pathsep.join(map(str, paths)), **OFFLINE_FLAGS,
    }
    if capture is not None:
        env.update(GP03_CAPTURE_PATH=str(capture), GP03_PUBLIC_PACKAGE=package)
    return env

def bounded_command(argv, root, env, output, timeout):
    output.mkdir(parents=True, exist_ok=False)
    started = time.monotonic()
    with (output / "stdout.txt").open("wb") as stdout, (output / "stderr.txt").open("wb") as stderr:
        proc = subprocess.Popen(
            argv, cwd=root, env=env, stdout=stdout, stderr=stderr, start_new_session=True,
        )
        timed_out = False
        try:
            if tuple(argv[1:3]) == ('-m', 'pytest'):
                try:
                    register_trusted_child(proc, os.environ, tuple(argv))
                except (LiveSourceHold, OSError) as exc:
                    raise CPUHold('TRUSTED_PYTEST_REGISTRATION', str(exc)) from exc
            proc.wait(timeout=max(0.01, timeout))
        except subprocess.TimeoutExpired:
            timed_out = True
        finally:
            try:
                os.killpg(proc.pid, 0)
            except ProcessLookupError:
                pass
            else:
                _cancel_process_group(proc)
    return {
        "argv": list(argv), "exit_code": proc.returncode, "timed_out": timed_out,
        "seconds": round(time.monotonic() - started, 3),
    }

def remaining(deadline, phase_cap):
    seconds = deadline - time.monotonic()
    if seconds <= 0:
        raise CPUHold("TASK_DEADLINE", "The CPU task wall clock expired")
    return min(seconds, phase_cap)

EXCLUDED_DIRS = frozenset({
    ".git", ".venv", "venv", "__pycache__", ".pytest_cache", ".mypy_cache",
    ".ruff_cache", ".tox", ".nox", ".cache", "node_modules", ".ipynb_checkpoints",
    ".gp02c_results", "__ledger_output__", "htmlcov", ".coverage-data",
})

EXCLUDED_NAMES = frozenset({".coverage", ".DS_Store", "coverage.xml"})

EXCLUDED_SUFFIXES = frozenset({".pyc", ".pyo"})

PYTEST_CONFIG_NAMES = (
    "pytest.ini", ".pytest.ini", "pyproject.toml", "setup.cfg", "tox.ini",
    "pytest.toml", ".pytest.toml",
)

RICH_OPTIONAL_TEST_DEFINITIONS = (
    "conftest.py", "tests/conftest.py", "tests/__init__.py",
    *PYTEST_CONFIG_NAMES, *(f"tests/{name}" for name in PYTEST_CONFIG_NAMES),
)

@dataclass(frozen=True)
class Snapshot:
    sha256: str
    files: int
    entries: tuple[tuple[str, str, str], ...]

class WorkspaceFingerprinter:
    """Conservative all-files policy, except a fixed generated-artifact exclusion list.

    Includes untracked source and config; file additions, deletions and symlink
    targets affect the fingerprint. NEVER walk an evaluator/protected workspace.
    """
    def __init__(self, root: str | Path):
        self.root = Path(root).resolve(strict=True)
        if not self.root.is_dir():
            raise ValueError("Workspace root must be a directory")

    def snapshot(self) -> Snapshot:
        entries: list[tuple[str, str, str]] = []
        for base, dirs, files in os.walk(self.root, followlinks=False):
            basedir = Path(base)
            # Symlink directories are recorded as links, never dereferenced.
            for name in list(dirs):
                full = basedir / name
                if name in EXCLUDED_DIRS:
                    dirs.remove(name)
                elif full.is_symlink():
                    dirs.remove(name)
                    rel = full.relative_to(self.root).as_posix()
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
            for name in files:
                if name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                    continue
                full = basedir / name
                rel = full.relative_to(self.root).as_posix()
                if full.is_symlink():
                    entries.append((rel, "L", hashlib.sha256(os.readlink(full).encode()).hexdigest()))
                elif full.is_file():
                    h = hashlib.sha256()
                    with full.open("rb") as f:
                        for block in iter(lambda: f.read(256 * 1024), b""):
                            h.update(block)
                    entries.append((rel, "F", h.hexdigest()))
        entries.sort()
        return Snapshot(canonical_hash(entries), len(entries), tuple(entries))

def definition_files_fingerprint(root: str | Path, relative_files: Sequence[str], *,
                                 allow_missing: bool = False) -> str:
    """Bind a selected visible test to its current test/config file bytes.

    File deletions, symlinks, path traversal and non-files fail closed. Caller
    supplies only approved visible test/config paths; NEVER protected tests.
    """
    root = Path(root).resolve(strict=True)
    if not relative_files or len(relative_files) != len(set(relative_files)):
        raise ValueError("Nonempty unique visible test definition paths required")
    entries = []
    for name in sorted(relative_files):
        relative = Path(name)
        if (not name or relative.is_absolute() or any(x in {"..", "."} for x in relative.parts)
            or relative.as_posix() != name):
            raise ValueError("Invalid or noncanonical visible definition path")
        path = root / relative
        if path.is_symlink() or not path.resolve().is_relative_to(root):
            raise ValueError("Visible definition unsafe or symlinked")
        if not path.is_file():
            if (allow_missing and not path.exists() and
                (name in RICH_OPTIONAL_TEST_DEFINITIONS or relative.name == "conftest.py")):
                entries.append((name, 'ABSENT_AT_FREEZE'))
                continue
            raise ValueError("Visible definition missing or unsafe")
        entries.append((name, hashlib.sha256(path.read_bytes()).hexdigest()))
    return canonical_hash(entries)

CONFIG_FILES = ('setup.py', 'setup.cfg', 'pytest.ini', '.pytest.ini', 'pytest.toml',
                '.pytest.toml', 'pyproject.toml', 'tox.ini', 'conftest.py')

HERMETIC_INI = """[pytest]
addopts = -p no:anyio
norecursedirs = .* build dist venv
python_classes = Test* *Test
python_files = test_*.py *_test.py
filterwarnings =
    ignore::DeprecationWarning
    ignore::UserWarning
"""

HERMETIC_HEADER = '# Hermetic test discovery hook for SWE-gemma\n'

def snapshot_link_target(name, linkname, symbolic):
    if not linkname or PurePosixPath(linkname).is_absolute() or "\\" in linkname:
        raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Invalid target for {name}: {linkname[:180]}")
    base = name.parent.as_posix() if symbolic else "."
    target = PurePosixPath(posixpath.normpath(posixpath.join(base, linkname)))
    if ".." in target.parts or target == PurePosixPath("."):
        raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Target escapes or points to the archive root: {name}")
    return target

def install_snapshot_links(destination, links, regular):
    # Ordinary files are extracted first, so archive contents cannot be written through a link.
    for name, entry in links.items():
        path = destination / name
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists() or path.is_symlink():
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Link collides with a staged file or directory: {name}")
        if entry["kind"] == "hardlink":
            target = entry["target"]
            visited = {name}
            while target in links and links[target]["kind"] == "hardlink" and target not in visited:
                visited.add(target)
                target = links[target]["target"]
            if target not in regular:
                raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Hardlink lacks an in-archive regular target: {name}")
            os.link(destination / target, path, follow_symlinks=False)
    for name, entry in links.items():
        if entry["kind"] == "symlink":
            path = destination / name
            if path.exists() or path.is_symlink():
                raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Symbolic link collides with a staged path: {name}")
            path.symlink_to(entry["linkname"])
    for name, entry in links.items():
        path = destination / name
        try:
            target = path.resolve(strict=True)
        except (OSError, RuntimeError) as exc:
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Broken or cyclic archive link: {name}") from exc
        if not target.is_relative_to(destination) or target in path.parents:
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Archive link target escapes or cycles to an ancestor: {name}")

def stage_public_snapshot(archive, destination, repo, deadline):
    destination = destination.resolve()
    destination.mkdir()
    total = 0
    regular, links = set(), {}
    with tarfile.open(archive, "r|gz") as tar:
        for member in tar:
            name = PurePosixPath(member.name)
            if name.is_absolute() or ".." in name.parts or "\\" in member.name:
                raise CPUHold("UNSAFE_SNAPSHOT_MEMBER", f"Unsafe archive path: {member.name[:200]}")
            remaining(deadline, POLICY["task_wall_seconds"])
            if member.issym() or member.islnk():
                if name in links or name in regular or name == PurePosixPath("."):
                    raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Duplicate or invalid archive link: {name}")
                links[name] = {
                    "kind": "symlink" if member.issym() else "hardlink",
                    "target": snapshot_link_target(name, member.linkname, member.issym()),
                    "linkname": member.linkname,
                }
                continue
            if not (member.isfile() or member.isdir()) or name in links or name in regular:
                raise CPUHold("UNSAFE_SNAPSHOT_MEMBER", f"Invalid or duplicate archive member: {name} ({member.type!r})")
            total += member.size
            if total > POLICY["snapshot_bytes_expanded_cap"]:
                raise CPUHold("SNAPSHOT_SIZE_LIMIT", "Expanded archive exceeded its CPU staging limit")
            tar.extract(member, destination, filter="data")
            if member.isfile():
                regular.add(name)
    install_snapshot_links(destination, links, regular)

    package, layouts = PACKAGE_LAYOUTS[repo]
    candidates = [
        (item.parent, layout) for item in destination.rglob(".git") if item.is_dir()
        for layout in layouts if (item.parent / layout / "__init__.py").is_file()
    ]
    if len(candidates) != 1:
        raise CPUHold("SNAPSHOT_LAYOUT", f"Expected one Git checkout containing {package}")
    root, package_dir = candidates[0]
    link_inventory = []
    for name, entry in sorted(links.items()):
        path = destination / name
        target = path if entry["kind"] == "symlink" else destination / entry["target"]
        if path.is_relative_to(root) and not target.resolve(strict=True).is_relative_to(root):
            raise CPUHold("UNSAFE_SNAPSHOT_LINK", f"Link in the checkout points outside it: {name}")
        link_inventory.append({"path": name.as_posix(), "kind": entry["kind"],
                               "target": entry["target"].as_posix()})
    return root, package, package_dir, link_inventory

def git_bytes(root, env, deadline, *argv):
    result = subprocess.run(
        ("git", "-C", str(root), *argv), env=env, capture_output=True,
        timeout=remaining(deadline, 20), check=False,
    )
    if result.returncode:
        raise CPUHold("NATIVE_GIT_PROBE", "Git could not read the mounted checkout identity")
    return result.stdout

def prepare_pytest_config(root):
    """Match the organizer subprocess harness's baseline pytest setup."""
    ini = root / "pytest.ini"
    text = ini.read_text(errors="replace") if ini.is_file() else ""
    if "# Hermetic test discovery" not in text and "[pytest]" not in text:
        ini.write_text(HERMETIC_INI, encoding="utf-8")
    conftest = root / "conftest.py"
    old = conftest.read_text(errors="replace") if conftest.is_file() else ""
    if not any(f"# {kind} test discovery hook for SWE-gemma" in old for kind in ("Hermetic", "Standard")):
        clean = old.replace(HERMETIC_HEADER, "").strip()
        conftest.write_text(HERMETIC_HEADER + ("\n" + clean if clean else "") + "\n", encoding="utf-8")

def visible_definition_paths(root):
    paths = set(RICH_OPTIONAL_TEST_DEFINITIONS)
    paths.update(name for name in CONFIG_FILES if (root / name).is_file())
    for path in (root / "tests").rglob("*"):
        relative = path.relative_to(root)
        if (any(part in EXCLUDED_DIRS for part in relative.parts)
                or path.name in EXCLUDED_NAMES or path.suffix in EXCLUDED_SUFFIXES):
            continue
        if path.is_file():
            paths.add(relative.as_posix())
    return tuple(sorted(paths))

def visible_definition_binding(root, files):
    regular, links = [], {}
    for name in files:
        path = root / name
        if not path.is_symlink():
            regular.append(name)
            continue
        target = path.resolve(strict=True)
        if not target.is_relative_to(root) or not target.is_file():
            raise CPUHold("UNSAFE_VISIBLE_DEFINITION_LINK", f"Invalid linked public fixture: {name}")
        links[name] = {"link_text": os.readlink(path), "target": target.relative_to(root).as_posix(),
                       "target_sha256": sha256_file(target)}
    digest = definition_files_fingerprint(root, regular, allow_missing=True)
    if links:
        digest = canonical_hash({"regular_sha256": digest, "internal_fixture_links": links})
    return digest, links

IMPORT_PROBE = """
import importlib
from pathlib import Path
import sys
module = importlib.import_module(sys.argv[1])
origin = Path(module.__file__).resolve()
print(origin)
if not origin.is_relative_to(Path(sys.argv[2]).resolve()):
    raise RuntimeError("Production package import escaped the public snapshot")
"""

CAPTURE_PLUGIN = """import json
import os
from pathlib import Path
import sys

def pytest_collection_finish(session):
    missing = set()
    if os.environ["GP03_PUBLIC_PACKAGE"] == "requests":
        for item in session.items:
            fixture_info = getattr(item, "_fixtureinfo", None)
            if fixture_info is None:
                continue
            for name in {"httpbin", "httpbin_secure"}.intersection(item.fixturenames):
                definitions = fixture_info.name2fixturedefs.get(name) or ()
                # Requests may wrap the plugin fixture by requesting its own name.
                if not any(name not in definition.argnames for definition in definitions):
                    missing.add(name)
    capture = {
        "root": str(session.config.rootpath.resolve()),
        "nodeids": [item.nodeid for item in session.items],
        "item_files": [str(item.path.resolve()) for item in session.items],
        "missing_optional_fixtures": sorted(missing),
    }
    Path(os.environ["GP03_CAPTURE_PATH"]).write_text(json.dumps(capture), encoding="utf-8")

def pytest_sessionfinish(session, exitstatus):
    target = Path(os.environ["GP03_CAPTURE_PATH"])
    if target.is_file():
        capture = json.loads(target.read_text(encoding="utf-8"))
        package = sys.modules.get(os.environ["GP03_PUBLIC_PACKAGE"])
        capture["production_origin"] = getattr(package, "__file__", None)
        target.write_text(json.dumps(capture), encoding="utf-8")
"""

def diagnostic_text(output):
    fragments = []
    for name in ("stdout.txt", "stderr.txt"):
        path = output / name
        with path.open("rb") as stream:
            head = stream.read(32 * 1024)
            if path.stat().st_size > len(head):
                stream.seek(max(len(head), path.stat().st_size - 32 * 1024))
                head += b"\n" + stream.read()
        fragments.append(head.decode("utf-8", errors="replace"))
    return "\n".join(fragments)

def missing_imports(text):
    return set(re.findall(r"ModuleNotFoundError: No module named ['\"]([A-Za-z0-9_.]+)['\"]", text))

def read_node_capture(path, selector, root, package_dir):
    if not path.is_file():
        raise CPUHold("NODE_CAPTURE_MISSING", "pytest produced no independent node ID capture")
    capture = json.loads(path.read_text(encoding="utf-8"))
    nodes = capture.get("nodeids", [])
    if not nodes:
        raise CPUHold("ZERO_COLLECTION", "The selected public suite collected zero tests")
    if len(nodes) > POLICY["max_collected_nodes"]:
        raise CPUHold("COLLECTION_LIMIT", f"Collected {len(nodes)} nodes, exceeding the frozen CPU cap")
    pytest_root = Path(capture["root"]).resolve(strict=True)
    if not pytest_root.is_relative_to(root):
        raise CPUHold("COLLECTION_IDENTITY", "pytest root is outside the staged checkout")
    files = capture.get("item_files", [])
    if len(files) != len(nodes):
        raise CPUHold("COLLECTION_IDENTITY", "pytest omitted a collected file identity")
    canonical = []
    for node, item_file in zip(nodes, files):
        if not isinstance(node, str):
            raise CPUHold("COLLECTION_IDENTITY", "Invalid pytest node ID")
        filename, delimiter, suffix = node.partition("::")
        relative = PurePosixPath(filename)
        if not delimiter or not suffix or relative.is_absolute() or ".." in relative.parts or "\\" in filename:
            raise CPUHold("COLLECTION_IDENTITY", "Invalid relative pytest node path")
        path_from_node = pytest_root / relative
        if (not path_from_node.is_relative_to(root) or str(path_from_node.resolve(strict=True)) != item_file):
            raise CPUHold("COLLECTION_IDENTITY", "Node path disagrees with the file collected by pytest")
        canonical.append(path_from_node.relative_to(root).as_posix() + "::" + suffix)
    if "::" in selector:
        permitted = lambda item: item == selector or item.startswith(selector + "[")
    else:
        permitted = lambda item: item.startswith(selector + "::")
    if not all(permitted(n) for n in canonical) or len(canonical) != len(set(canonical)):
        raise CPUHold("COLLECTION_IDENTITY", "Canonical node identities differ from the selector")
    origin = capture.get("production_origin")
    if origin and not Path(origin).resolve().is_relative_to(root / package_dir):
        raise CPUHold("PACKAGE_ORIGIN_CHANGED", "pytest imported production code outside the snapshot")
    return canonical, capture["missing_optional_fixtures"]

def run_visible_phase(*, selector, root, package, package_dir, temporary, output,
                      deadline, baseline_sha, collect_only):
    argv = [sys.executable, "-m", "pytest", selector, "-q", "--tb=short",
            "-p", "no:cacheprovider", "-p", "_gp03_capture"]
    capture = output / "nodeids.json"
    if collect_only:
        argv.append("--collect-only")
        cap = POLICY["collect_wall_seconds"]
    else:
        argv.append(f"--junitxml={output / 'junit.xml'}")
        cap = POLICY["baseline_wall_seconds"]
    result = bounded_command(
        argv, root, command_env(root, temporary, capture, package),
        output, remaining(deadline, cap),
    )
    result["source_sha256_after"] = WorkspaceFingerprinter(root).snapshot().sha256
    write_json(output / "command.json", result)
    if result["source_sha256_after"] != baseline_sha:
        raise CPUHold("SOURCE_CHANGED", "The selected pytest command changed the staged source or test definitions")
    if result["timed_out"]:
        reason = "COLLECTION_TIMEOUT" if collect_only else "BASELINE_TIMEOUT"
        raise CPUHold(reason, "The selected public pytest command exceeded its CPU limit")
    if collect_only and result["exit_code"] == 5:
        raise CPUHold("ZERO_COLLECTION", "The selected public suite collected zero tests")
    return result, capture

def junit_counts(junit: Path | bytes) -> tuple[int, int, int, int, int]:
    """Extract actual testcase elements; never trust exit code or suite labels alone."""
    data = junit if isinstance(junit, bytes) else junit.read_bytes()
    root = ET.fromstring(data)
    cases = [el for el in root.iter() if el.tag.rsplit("}", 1)[-1] == "testcase"]
    collected = len(cases)
    fail = sum(any(c.tag.rsplit("}", 1)[-1] == "failure" for c in t) for t in cases)
    err = sum(any(c.tag.rsplit("}", 1)[-1] == "error" for c in t) for t in cases)
    skipped = sum(any(c.tag.rsplit("}", 1)[-1] == "skipped" for c in t) for t in cases)
    passed = collected - fail - err - skipped
    if passed < 0 or collected != fail + err + skipped + passed:
        raise ValueError("Contradictory JUnit testcase counts")
    return collected, passed, fail, err, skipped

def artifact_paths(folder, excluded=()):
    files = {}
    for path in sorted(folder.rglob("*")):
        relative = path.relative_to(folder).as_posix()
        if path.is_symlink():
            raise CPUHold("UNSAFE_EVIDENCE_PATH", f"Symlink in CPU evidence at {relative}")
        if path.is_file() and relative not in excluded:
            files[relative] = path
    return files

def seal_zip(output, files):
    """Write a new evidence archive and index every included file."""
    output = Path(output)
    index = {}
    temporary = output.with_suffix('.partial.zip')
    if output.exists() or temporary.exists():
        raise FileExistsError(f'Refusing to overwrite evidence {output}')
    try:
        with zipfile.ZipFile(temporary, 'x', compression=zipfile.ZIP_DEFLATED) as z:
            for name, path in sorted(files.items()):
                path = Path(path)
                if path.is_symlink() or not path.is_file():
                    raise ValueError(f'Unsafe evidence file {name}')
                index[name] = {'bytes': path.stat().st_size, 'sha256': sha256_file(path)}
                z.write(path, name)
            z.writestr('SHA256_MANIFEST.json', json.dumps(index, indent=2, sort_keys=True) + '\n')
        verify_zip(temporary)
        temporary.rename(output)
    finally:
        temporary.unlink(missing_ok=True)
    print(f'Evidence ZIP {output.name} | SHA-256 {sha256_file(output)}', flush=True)
    return output

def verify_zip(archive):
    with zipfile.ZipFile(archive) as z:
        names = z.namelist()
        index = json.loads(z.read('SHA256_MANIFEST.json'))
        if len(names) != len(set(names)) or set(names) != set(index) | {'SHA256_MANIFEST.json'}:
            raise RuntimeError('ZIP member set disagrees with the checksum manifest')
        for name, pin in index.items():
            path = Path(name)
            if path.is_absolute() or '..' in path.parts or z.getinfo(name).file_size != pin['bytes']:
                raise RuntimeError(f'Invalid archive member {name}')
            digest = hashlib.sha256()
            with z.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b''):
                    digest.update(block)
            if digest.hexdigest() != pin['sha256']:
                raise RuntimeError(f'Archive checksum mismatch for {name}')
    return index

def offer_zip_download(bundle, auto_download=True, embed_limit_mb=10):
    from pathlib import Path
    import base64
    import html
    import json
    import os
    from IPython.display import FileLink, HTML, Javascript, display

    bundle = Path(bundle).resolve()
    relative = os.path.relpath(bundle, Path.cwd())
    print('ZIP ready:', bundle)
    print('If the browser does not start a download, use the Download ZIP button or Kaggle Output panel.')
    display(FileLink(relative))
    if bundle.stat().st_size > embed_limit_mb * 1024 * 1024:
        print('This ZIP exceeds the inline-download size limit; use the file link or Output panel.')
        return {'automatic_download_requested': False, 'download_confirmed': False, 'reason': 'size_limit'}

    encoded = base64.b64encode(bundle.read_bytes()).decode('ascii')
    name = html.escape(bundle.name, quote=True)
    button = ('<a download="' + name + '" href="data:application/zip;base64,' + encoded + '" '
              'style="display:inline-block;padding:12px 20px;background:#174EA6;color:white;'
              'text-decoration:none;border-radius:6px;font-weight:600">Download ZIP</a>')
    display(HTML(button))
    if auto_download:
        script = '''(function () {
  try {
    const payload = __PAYLOAD__;
    const bytes = Uint8Array.from(atob(payload), function (c) { return c.charCodeAt(0); });
    const url = URL.createObjectURL(new Blob([bytes], {type: "application/zip"}));
    const link = document.createElement("a");
    link.href = url;
    link.download = __FILENAME__;
    link.style.display = "none";
    document.body.appendChild(link);
    link.click();
    link.remove();
    setTimeout(function () { URL.revokeObjectURL(url); }, 60000);
  } catch (error) {
    console.info("Automatic ZIP download was not available; use the Download ZIP button or Output panel.", error);
  }
})();'''.replace('__PAYLOAD__', json.dumps(encoded)).replace('__FILENAME__', json.dumps(bundle.name))
        display(Javascript(script))
        print('Automatic browser-download request offered. Notebook code cannot confirm that your PC saved it.')
    return {'automatic_download_requested': bool(auto_download), 'download_confirmed': False,
            'fallbacks': ['Download ZIP button', 'file link', 'Kaggle Output panel']}

# Reused from the pinned GP02C-04B source gp02c_ledger.py.
ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,120}$")

@dataclass(frozen=True)
class VisibleCheck:
    check_id: str
    command: tuple[str, ...]
    definition_sha256: str
    reserve_seconds: float = 0.0
    origin: str = "visible_repo"
    working_dir: str = "."
    test_selection: str = ""
    expected_collected: int | None = None
    pinned_environment_sha256: str | None = None
    definition_files: tuple[str, ...] = ()
    definition_files_sha256: str | None = None
    allow_missing_definition_files: bool = False

    def __post_init__(self):
        if not ID_RE.fullmatch(self.check_id):
            raise ValueError("Unsafe visible check identifier")
        if self.origin != "visible_repo":
            raise ValueError("Only trusted visible-repository checks are allowed")
        if not self.command or not all(isinstance(x, str) and x for x in self.command):
            raise ValueError("Nonempty command tokens required")
        if not re.fullmatch(r"[0-9a-f]{64}", self.definition_sha256):
            raise ValueError("Check definition must have SHA-256 identity")
        if self.reserve_seconds < 0:
            raise ValueError("Negative estimate")
        if (Path(self.working_dir).is_absolute() or
            any(x == ".." for x in Path(self.working_dir).parts)):
            raise ValueError("Working directory must be workspace-relative")
        if self.expected_collected is not None and self.expected_collected < 1:
            raise ValueError("Expected collection must be positive")
        if self.pinned_environment_sha256 is not None and not re.fullmatch(
            r"[0-9a-f]{64}", self.pinned_environment_sha256):
            raise ValueError("Bad pinned environment fingerprint")
        if bool(self.definition_files) != bool(self.definition_files_sha256):
            raise ValueError("Pinned visible definition files and digest must be supplied together")
        if self.definition_files_sha256 and not re.fullmatch(r"[0-9a-f]{64}", self.definition_files_sha256):
            raise ValueError("Bad pinned visible definition content fingerprint")
        for path in self.definition_files:
            rel = Path(path)
            if (not path or rel.is_absolute() or ".." in rel.parts or rel.as_posix() != path):
                raise ValueError("Unsafe visible definition filename")

    @property
    def command_sha256(self) -> str:
        return canonical_hash(self.command)

@dataclass(frozen=True)
class TestObservation:
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    elapsed_seconds: float
    stdout: str = ""
    stderr: str = ""
    timed_out: bool = False
    setup_error: bool = False
    tests_skipped: int | None = 0

    def __post_init__(self):
        if self.elapsed_seconds < 0:
            raise ValueError("Negative duration")
        for name in ("tests_collected", "tests_passed", "tests_failed", "tests_errored", "tests_skipped"):
            v = getattr(self, name)
            if v is not None and v < 0:
                raise ValueError(f"Negative test count: {name}")

    @property
    def observed_outcome(self) -> str:
        if self.timed_out or self.setup_error or self.exit_code is None:
            return "ERROR"
        counts = (self.tests_collected, self.tests_passed,
                  self.tests_failed, self.tests_errored, self.tests_skipped)
        if any(c is None for c in counts):
            return "ERROR"
        total, passed, failed, errored, skipped = counts
        if total < 1 or passed + failed + errored + skipped != total:
            return "ERROR"
        if errored > 0:
            return "ERROR"
        if failed > 0 and self.exit_code != 0:
            return "FAIL"
        if failed > 0 and self.exit_code == 0:
            return "ERROR"  # contradictory successful process
        if self.exit_code != 0:
            return "ERROR"  # e.g. missing dependency / collection failure
        if passed < 1:
            return "ERROR"  # all skipped / no executed positive check
        return "PASS"

# Reused from the pinned GP02C-04B source gp02c_live_adapter.py.
@dataclass
class RealDeadline:
    """One absolute clock per attempt. 840/900s defaults; no reset per check."""
    repair_cutoff_seconds: float = 840.0
    overall_cutoff_seconds: float = 900.0
    started_monotonic: float | None = None
    now: Callable[[], float] = time.monotonic

    def __post_init__(self):
        if not 0 < self.repair_cutoff_seconds < self.overall_cutoff_seconds:
            raise ValueError("Need 0 < repair cutoff < overall cutoff")
        if self.started_monotonic is None:
            self.started_monotonic = self.now()

    def remaining(self, phase: str) -> float:
        if phase not in {"ordinary", "proactive", "final"}:
            raise ValueError("Unrecognized execution phase")
        cap = self.overall_cutoff_seconds if phase == "final" else self.repair_cutoff_seconds
        return max(0.0, self.started_monotonic + cap - self.now())

    def elapsed(self) -> float:
        return self.now() - self.started_monotonic

def _read_head(path: Path, max_bytes: int = 6000) -> str:
    with path.open("rb") as f:
        return f.read(max_bytes).decode("utf-8", errors="replace")

def copy_gp03_workspace(source, destination):
    """Copy the public working tree and verify all links in the private copy."""
    source = source.resolve(strict=True)
    before = WorkspaceFingerprinter(source).snapshot()

    def ignore_generated(directory, names):
        ignored = set()
        for name in names:
            if Path(directory, name).is_dir():
                if name in EXCLUDED_DIRS:
                    ignored.add(name)
            elif name in EXCLUDED_NAMES or Path(name).suffix in EXCLUDED_SUFFIXES:
                ignored.add(name)
        return ignored

    shutil.copytree(source, destination, symlinks=True, ignore=ignore_generated)
    copied = WorkspaceFingerprinter(destination).snapshot()
    if copied.sha256 != before.sha256:
        raise CPUHold("ISOLATED_COPY_MISMATCH", "Public source fingerprint changed during copying")
    links = []
    for name, kind, _ in copied.entries:
        if kind != "L":
            continue
        link = destination / name
        target_text = os.readlink(link)
        snapshot_link_target(PurePosixPath(name), target_text, symbolic=True)
        try:
            target = link.resolve(strict=True)
        except (OSError, RuntimeError) as exc:
            raise CPUHold("UNSAFE_ISOLATED_LINK", f"Broken or cyclic public link {name}") from exc
        if (not target.is_relative_to(destination) or target == destination
                or (target.is_dir() and link.parent.is_relative_to(target))
                or not (target.is_dir() or target.is_file())):
            raise CPUHold("UNSAFE_ISOLATED_LINK", f"Unsafe public link {name}")
        links.append({"path": name, "target": target.relative_to(destination).as_posix()})
    return before.sha256, links


def gp03_check_from_manifest(manifest):
    fields = dict(manifest["check"])
    fields["command"] = tuple(fields["command"])
    fields["definition_files"] = tuple(fields["definition_files"])
    return VisibleCheck(**fields)


class GP03PublicPytestAdapter:
    """Execute one pinned public check and return the controller's TestObservation."""

    def __init__(self, *, source_root, manifest, environment_sha256, deadline, results_root):
        self.source_root = Path(source_root).resolve(strict=True)
        self.results_root = Path(results_root).resolve()
        if (self.results_root.is_relative_to(self.source_root)
                or self.source_root.is_relative_to(self.results_root)):
            raise ValueError("Evidence and source must be in separate directories")
        self.results_root.mkdir(parents=True, exist_ok=True)
        self.manifest = manifest
        self.check = gp03_check_from_manifest(manifest)
        self.environment_sha256 = environment_sha256
        self.deadline = deadline
        self.phase = "ordinary"
        self.executions = []

    def _validate_definition(self, check, root):
        selector = check.test_selection
        if (check != self.check or check.working_dir != "."
                or check.pinned_environment_sha256 != self.environment_sha256
                or check.command != (sys.executable, "-m", "pytest", selector, "-q", "--tb=short")
                or not PUBLIC_TEST.fullmatch(selector.partition("::")[0])):
            raise CPUHold("CHECK_PIN_MISMATCH", "The trusted public command differs from its pin")
        files = visible_definition_paths(root)
        if files != check.definition_files:
            raise CPUHold("VISIBLE_DEFINITION_SET_CHANGED", "The public test or configuration file set changed")
        definition_sha, fixture_links = visible_definition_binding(root, files)
        if (definition_sha != check.definition_files_sha256
                or fixture_links != self.manifest.get("definition_link_targets", {})):
            raise CPUHold("VISIBLE_DEFINITION_CHANGED", "The public test, configuration or fixture bytes changed")

    def __call__(self, check):
        started = time.monotonic()
        absolute_deadline = started + self.deadline.remaining(self.phase)
        folder = self.results_root / f"check_{len(self.executions) + 1:04d}"
        folder.mkdir()
        receipt = {"check_id": check.check_id, "phase": self.phase, "reason": None}
        phase = folder / "pytest"
        counts = (None,) * 5
        exit_code = None
        try:
            remaining(absolute_deadline, POLICY["baseline_wall_seconds"])
            with tempfile.TemporaryDirectory(prefix="gp03_isolated_", dir=self.results_root.parent) as work:
                temporary = Path(work).resolve()
                isolated = temporary / "checkout"
                source_sha, links = copy_gp03_workspace(self.source_root, isolated)
                receipt.update(source_sha256_before=source_sha, isolated_links=links)
                self._validate_definition(check, isolated)
                package = PACKAGE_LAYOUTS[self.manifest["repo"]][0]
                package_dir = self.manifest["production_root"].rstrip("/")
                (temporary / "_gp03_capture.py").write_text(CAPTURE_PLUGIN, encoding="utf-8")
                result, capture = run_visible_phase(
                    selector=check.test_selection, root=isolated, package=package, package_dir=package_dir,
                    temporary=temporary, output=phase, deadline=absolute_deadline,
                    baseline_sha=source_sha, collect_only=False,
                )
                exit_code = result["exit_code"]
                nodes, missing = read_node_capture(capture, check.test_selection, isolated, package_dir)
                junit = phase / "junit.xml"
                counts = junit_counts(junit)
                receipt.update(actual_nodeids_sha256=canonical_hash(nodes), junit_sha256=sha256_file(junit),
                               isolated_source_sha256_after=result["source_sha256_after"])
                total, passed, failed, errors, skipped = counts
                if missing or nodes != self.manifest["expected_nodeids"] or total != check.expected_collected:
                    raise CPUHold("CHECK_NODE_MISMATCH", "Public check collection differs from its frozen node IDs")
                if errors or total == skipped or (exit_code, bool(failed)) not in {(0, False), (1, True)}:
                    raise CPUHold("PUBLIC_PYTEST_ERROR", "JUnit and the pytest exit status do not show a complete check")
                if failed and missing_imports(diagnostic_text(phase)):
                    raise CPUHold("PUBLIC_PYTEST_DEPENDENCY", "A completed test raised a missing dependency error")
                remaining(absolute_deadline, 1)
        except CPUHold as exc:
            receipt.update(reason=exc.reason, detail=str(exc))
        except (OSError, ValueError, ET.ParseError) as exc:
            receipt.update(reason="PUBLIC_CHECK_ERROR", detail=f"{type(exc).__name__}: {exc}")
        finally:
            if "source_sha256_before" in receipt:
                after = WorkspaceFingerprinter(self.source_root).snapshot().sha256
                receipt["source_sha256_after"] = after
                if after != receipt["source_sha256_before"]:
                    receipt.update(reason="ORIGINAL_SOURCE_CHANGED", detail="Source changed while pytest ran")
        total, passed, failed, errors, skipped = counts
        stdout = _read_head(phase / "stdout.txt") if (phase / "stdout.txt").is_file() else ""
        stderr = _read_head(phase / "stderr.txt") if (phase / "stderr.txt").is_file() else ""
        if not receipt["reason"] and time.monotonic() >= absolute_deadline:
            receipt.update(reason="TASK_DEADLINE", detail="The public check exceeded its phase deadline")
        if receipt["reason"]:
            stderr += f"\n{receipt['reason']} | {receipt['detail']}"
        observation = TestObservation(
            exit_code, total, passed, failed, errors, round(time.monotonic() - started, 3),
            stdout=stdout, stderr=stderr,
            timed_out=receipt["reason"] in {"TASK_DEADLINE", "BASELINE_TIMEOUT"},
            setup_error=receipt["reason"] is not None, tests_skipped=skipped,
        )
        receipt.update(outcome=observation.observed_outcome, observation=asdict(observation))
        write_json(folder / "Public_Check_Receipt.json", receipt)
        self.executions.append(receipt)
        return observation

# Reused from the pinned GP02C-04B source gp02c_ledger.py.
ARMS = {"A", "B", "C"}

PROVENANCES = {"agent", "controller_recheck", "final_gate", "common_initial"}

SCHEMA = "gp02c.ledger.v1.3"

def utc_timestamp() -> str:
    return datetime.now(timezone.utc).isoformat()

@dataclass(frozen=True)
class VisibleManifest:
    checks: tuple[VisibleCheck, ...]
    strict: bool = False

    def __post_init__(self):
        ids = [c.check_id for c in self.checks]
        if not ids or len(ids) != len(set(ids)):
            raise ValueError("Manifest needs at least one unique selected visible check")
        if self.strict and any(c.expected_collected is None or
                               c.pinned_environment_sha256 is None or
                               not c.test_selection for c in self.checks):
            raise ValueError("Strict manifests must pin collection, environment, and selection")

    def get(self, check_id: str) -> VisibleCheck:
        for check in self.checks:
            if check.check_id == check_id:
                return check
        raise ValueError("Unselected or protected check is not allowed")

    @property
    def sha256(self) -> str:
        return canonical_hash([asdict(c) for c in self.checks])

@dataclass(frozen=True)
class Receipt:
    receipt_id: str
    run_id: str
    task_id: str
    arm: str
    seed: int
    check_id: str
    command: tuple[str, ...]
    command_definition_sha256: str
    command_sha256: str
    working_dir: str
    test_selection: str
    expected_collected: int | None
    workspace_revision_at_start: int
    workspace_revision_at_end: int
    workspace_sha256_at_start: str
    workspace_sha256_at_end: str
    environment_sha256: str
    timestamp_start: str
    timestamp_end: str
    exit_code: int | None
    tests_collected: int | None
    tests_passed: int | None
    tests_failed: int | None
    tests_errored: int | None
    tests_skipped: int | None
    qualification_error: str | None
    stdout_sha256: str
    stderr_sha256: str
    provenance: str
    observed_outcome: str
    elapsed_seconds: float
    timed_out: bool
    setup_error: bool
    test_completed_monotonic: float

@dataclass(frozen=True)
class ChangeEvent:
    action: str
    previous_revision: int
    new_revision: int
    changed: bool
    previous_sha256: str
    new_sha256: str
    edit_observed_monotonic: float

class EvidenceLedger:
    def __init__(self, root: str | Path, manifest: VisibleManifest,
                 *, run_id: str, task_id: str, arm: str, seed: int,
                 environment_sha256: str):
        if arm not in ARMS:
            raise ValueError("Unknown arm")
        for value in (run_id, task_id):
            if not ID_RE.fullmatch(value):
                raise ValueError("Unsafe run/task id")
        if not re.fullmatch(r"[0-9a-f]{64}", environment_sha256):
            raise ValueError("Expected environment SHA-256")
        self.fingerprinter = WorkspaceFingerprinter(root)
        self.manifest = manifest
        self.run_id = run_id
        self.task_id = task_id
        self.arm = arm
        self.seed = seed
        self.environment_sha256 = environment_sha256
        self.snapshot = self.fingerprinter.snapshot()
        self.revision = 0
        self.receipts: list[Receipt] = []
        self.changes: list[ChangeEvent] = []
        self.events: list[dict] = []
        self.unqualified_receipts: dict[str, str] = {}
        # Only a pass CURRENT AT THE INVALIDATING EDIT may start a regression pair.
        # Pending exposures are cleared on a later qualified check; old passes
        # cannot be resurrected after PASS -> FAIL -> edit -> FAIL.
        self.pending_regression_exposures: dict[str, dict] = {}

    def observe_workspace(self, action: str) -> ChangeEvent:
        new = self.fingerprinter.snapshot()
        old = self.snapshot
        before = self.revision
        changed = new.sha256 != old.sha256
        if changed:
            # Evaluate *prior to* advancing the source revision. The current
            # pass must be the latest qualified receipt for that check.
            for check in self.manifest.checks:
                prior = self.latest(check.check_id)
                if (prior is not None
                    and prior.observed_outcome == "PASS"
                    and prior.receipt_id not in self.unqualified_receipts
                    and prior.qualification_error is None
                    and prior.workspace_revision_at_start == self.revision
                    and prior.workspace_revision_at_end == self.revision
                    and prior.workspace_sha256_at_start == old.sha256
                    and prior.workspace_sha256_at_end == old.sha256
                    and prior.environment_sha256 == self.environment_sha256
                    and prior.command_definition_sha256 == check.definition_sha256
                    and (not check.definition_files_sha256 or
                         self._definition_matches(check))):
                    self.pending_regression_exposures.setdefault(check.check_id, {
                        "prior_receipt_id": prior.receipt_id,
                        "prior_revision": self.revision,
                        "edit_revision": self.revision + 1,
                        "edit_monotonic": time.monotonic(),
                        "prior_env": prior.environment_sha256,
                        "prior_command": prior.command_sha256,
                        "prior_definition": prior.command_definition_sha256,
                    })
            self.revision += 1
            self.snapshot = new
        ev = ChangeEvent(action, before, self.revision, changed, old.sha256,
                         new.sha256, time.monotonic())
        self.changes.append(ev)
        return ev

    def _definition_matches(self, check: VisibleCheck) -> bool:
        if not check.definition_files_sha256:
            return True  # Legacy CPU fixtures; GP02C-04 pilot requires pinned files.
        try:
            if check.check_id.startswith('dev_'):
                from gp02c_dev_registry import validate_check_definition
                return validate_check_definition(self.fingerprinter.root, check)
            return (definition_files_fingerprint(self.fingerprinter.root, check.definition_files,
                                               allow_missing=check.allow_missing_definition_files)
                    == check.definition_files_sha256)
        except (ValueError, OSError):
            return False

    def set_environment(self, environment_sha256: str) -> bool:
        if not re.fullmatch(r"[0-9a-f]{64}", environment_sha256):
            raise ValueError("Expected environment SHA-256")
        changed = self.environment_sha256 != environment_sha256
        self.environment_sha256 = environment_sha256
        if changed:
            self.events.append({"type": "environment_changed", "timestamp": utc_timestamp()})
        return changed

    def status(self, receipt: Receipt) -> str:
        check = self.manifest.get(receipt.check_id)
        self.observe_workspace("status_sync")  # detect out-of-band source changes
        if self.latest(check.check_id) is not receipt:
            return "STALE"
        if (receipt.workspace_revision_at_start != receipt.workspace_revision_at_end
            or receipt.workspace_sha256_at_start != receipt.workspace_sha256_at_end
            or receipt.workspace_revision_at_end != self.revision
            or receipt.workspace_sha256_at_end != self.snapshot.sha256
            or receipt.environment_sha256 != self.environment_sha256
            or receipt.command_definition_sha256 != check.definition_sha256
            or not self._definition_matches(check)
            or receipt.command_sha256 != check.command_sha256
            or receipt.working_dir != check.working_dir
            or receipt.test_selection != check.test_selection
            or (check.pinned_environment_sha256 is not None and
                check.pinned_environment_sha256 != self.environment_sha256)):

            return "STALE"
        if receipt.qualification_error or receipt.receipt_id in self.unqualified_receipts:
            return "ERROR"
        return {"PASS": "CURRENT_PASS", "FAIL": "CURRENT_FAIL"}.get(
            receipt.observed_outcome, "ERROR")

    def latest(self, check_id: str) -> Receipt | None:
        self.manifest.get(check_id)
        for receipt in reversed(self.receipts):
            if receipt.check_id == check_id:
                return receipt
        return None

    def current_status(self, check_id: str) -> str:
        r = self.latest(check_id)
        return self.status(r) if r else "NOT_EXECUTED"

    def selected_current_passes(self) -> tuple[str, ...]:
        return tuple(c.check_id for c in self.manifest.checks
                     if self.current_status(c.check_id) == "CURRENT_PASS")

    def record_check(self, check_id: str, provenance: str,
                     executor: Callable[[VisibleCheck], TestObservation], *,
                     receipt_qualifier: Callable[[Receipt], None] | None = None) -> Receipt:
        if provenance not in PROVENANCES:
            raise ValueError("Invalid provenance")
        check = self.manifest.get(check_id)
        self.observe_workspace("before_visible_check")
        start = self.snapshot
        rev_start = self.revision
        started = utc_timestamp()
        outcome = executor(check)
        if not isinstance(outcome, TestObservation):
            raise TypeError("Trusted visible executor must return TestObservation")
        ended = utc_timestamp()
        self.observe_workspace("after_visible_check")
        receipt = Receipt(
            receipt_id=f"visible-{len(self.receipts)+1:06d}",
            run_id=self.run_id, task_id=self.task_id, arm=self.arm, seed=self.seed,
            check_id=check.check_id, command=check.command,
            command_definition_sha256=check.definition_sha256,
            command_sha256=check.command_sha256,
            working_dir=check.working_dir, test_selection=check.test_selection,
            expected_collected=check.expected_collected,
            workspace_revision_at_start=rev_start,
            workspace_revision_at_end=self.revision,
            workspace_sha256_at_start=start.sha256,
            workspace_sha256_at_end=self.snapshot.sha256,
            environment_sha256=self.environment_sha256,
            timestamp_start=started, timestamp_end=ended,
            exit_code=outcome.exit_code, tests_collected=outcome.tests_collected,
            tests_passed=outcome.tests_passed, tests_failed=outcome.tests_failed,
            tests_errored=outcome.tests_errored,
            tests_skipped=outcome.tests_skipped,
            qualification_error=(
                "VISIBLE_DEFINITION_CHANGED" if not self._definition_matches(check) else
                "UNEXPECTED_COLLECTION" if check.expected_collected is not None
                and outcome.tests_collected != check.expected_collected else
                "ENVIRONMENT_MISMATCH" if check.pinned_environment_sha256 is not None
                and self.environment_sha256 != check.pinned_environment_sha256 else None),
            stdout_sha256=hashlib.sha256(outcome.stdout.encode()).hexdigest(),
            stderr_sha256=hashlib.sha256(outcome.stderr.encode()).hexdigest(),
            provenance=provenance, observed_outcome=outcome.observed_outcome,
            elapsed_seconds=outcome.elapsed_seconds, timed_out=outcome.timed_out,
            setup_error=outcome.setup_error,
            test_completed_monotonic=time.monotonic())
        # Raw observation is immutable. Qualification is completed BEFORE
        # deriving any observed-regression event.
        self.receipts.append(receipt)
        if receipt_qualifier is not None:
            receipt_qualifier(receipt)
        exposure = self.pending_regression_exposures.get(check_id)
        valid = (receipt.receipt_id not in self.unqualified_receipts
                 and receipt.qualification_error is None
                 and receipt.observed_outcome in {"PASS", "FAIL"}
                 and receipt.workspace_revision_at_start == receipt.workspace_revision_at_end
                 and receipt.workspace_sha256_at_start == receipt.workspace_sha256_at_end)
        if valid and exposure is not None:
            if (receipt.observed_outcome == "FAIL"
                and exposure["edit_revision"] <= receipt.workspace_revision_at_end
                and exposure["prior_env"] == receipt.environment_sha256
                and exposure["prior_command"] == receipt.command_sha256
                and exposure["prior_definition"] == receipt.command_definition_sha256):
                self.events.append({
                    "type": "OBSERVED_PASS_TO_FAIL_AFTER_EDIT",
                    "check_id": check_id,
                    "prior_receipt_id": exposure["prior_receipt_id"],
                    "failure_receipt_id": receipt.receipt_id,
                    "prior_revision": exposure["prior_revision"],
                    "failure_revision": receipt.workspace_revision_at_end,
                    "invalidating_edit_observed_monotonic": exposure["edit_monotonic"],
                    "failure_observed_monotonic": receipt.test_completed_monotonic,
                    "edit_to_detection_seconds": max(
                        0.0, receipt.test_completed_monotonic - exposure["edit_monotonic"]),
                })
            # A qualified result resolves the exposure. A later failure cannot
            # piggyback on an old pass after this check has already failed.
            self.pending_regression_exposures.pop(check_id, None)
        return receipt

    def disqualify_receipt(self, receipt: Receipt, reason: str) -> None:
        """Fail closed on controller-budget violations without mutating raw observations."""
        self.unqualified_receipts[receipt.receipt_id] = reason
        self.events.append({"type": "RECEIPT_UNQUALIFIED", "receipt_id": receipt.receipt_id,
                            "reason": reason})

    def export(self) -> dict:
        # Refresh revision *before* writing any status rows or reporting a hash.
        self.observe_workspace("export_preflight")
        return {
            "schema": SCHEMA, "run_id": self.run_id, "task_id": self.task_id,
            "arm": self.arm, "seed": self.seed, "manifest_sha256": self.manifest.sha256,
            "workspace_revision": self.revision, "workspace_sha256": self.snapshot.sha256,
            "environment_sha256": self.environment_sha256,
            "receipts": [{**asdict(r), "status": self.status(r),
                          "controller_disqualification": self.unqualified_receipts.get(r.receipt_id)}
                         for r in self.receipts],
            "changes": [asdict(e) for e in self.changes],
            "events": list(self.events),
        }

@dataclass
class SharedBudget:
    max_wall_seconds: float = 900.0
    max_tool_calls: int = 60
    max_model_calls: int = 60
    used_wall_seconds: float = 0.0
    used_tool_calls: int = 0
    used_model_calls: int = 0
    used_test_runs: int = 0
    final_reserve_seconds: float = 60.0

    @property
    def remaining_wall(self) -> float:
        return max(0.0, self.max_wall_seconds - self.used_wall_seconds)

    def can_run_check(self, check: VisibleCheck, *, protect_final_reserve: bool = False) -> bool:
        reserve = self.final_reserve_seconds if protect_final_reserve else 0.0
        return (self.used_tool_calls < self.max_tool_calls
                and self.remaining_wall > 0
                and self.remaining_wall >= check.reserve_seconds + reserve)

    @property
    def remaining_repair_wall(self) -> float:
        return max(0.0, self.max_wall_seconds - self.final_reserve_seconds - self.used_wall_seconds)

    @property
    def over_limit(self) -> bool:
        return (self.used_wall_seconds > self.max_wall_seconds
                or self.used_tool_calls > self.max_tool_calls
                or self.used_model_calls > self.max_model_calls)

    def charge_model(self, elapsed_seconds: float = 0.0) -> None:
        if self.used_model_calls >= self.max_model_calls:
            raise RuntimeError("Model budget exhausted")
        self.used_model_calls += 1
        self.used_wall_seconds += elapsed_seconds

    def charge_tool(self, elapsed_seconds: float = 0.0) -> None:
        if self.used_tool_calls >= self.max_tool_calls:
            raise RuntimeError("Tool budget exhausted")
        self.used_tool_calls += 1
        self.used_wall_seconds += elapsed_seconds

    def charge_visible_test(self, elapsed_seconds: float) -> None:
        # Executing a selected check also consumes one shared tool invocation.
        self.charge_tool(elapsed_seconds)
        self.used_test_runs += 1

    def summary(self) -> dict:
        return {"used_wall_seconds": self.used_wall_seconds,
                "remaining_wall_seconds": self.remaining_wall,
                "used_tool_calls": self.used_tool_calls,
                "used_model_calls": self.used_model_calls,
                "used_test_runs": self.used_test_runs}

@dataclass(frozen=True)
class GateDecision:
    accepted: bool
    status: str
    checks: dict[str, str]
    attempts: int
    accepted_patch_sha256: str | None = None
    accepted_workspace_sha256: str | None = None
    accepted_revision: int | None = None

class VerificationController:
    """Trusted wrapper prototype. Same gate implementation for B and C.

    Executor is a trusted *visible* check adapter; NEVER pass hidden evaluator
    results. This prototype uses deterministic duration accounting. Live GP02C-03
    must enforce wall time and tool/model counters on the actual worker.
    """
    def __init__(self, ledger: EvidenceLedger, budget: SharedBudget,
                 executor: Callable[[VisibleCheck], TestObservation],
                 production_patch_sha256: Callable[[], str],
                 *, max_rechecks_per_edit: int = 4,
                 max_rechecks_per_run: int = 12):
        self.ledger = ledger
        self.budget = budget
        self.executor = executor
        self.production_patch_sha256 = production_patch_sha256
        if max_rechecks_per_edit < 0 or max_rechecks_per_run < 0:
            raise ValueError("Negative auto-recheck limit")
        self.max_rechecks_per_edit = max_rechecks_per_edit
        self.max_rechecks_per_run = max_rechecks_per_run
        self.auto_rechecks_used = 0

    def _patch_digest(self) -> str:
        digest = self.production_patch_sha256()
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise ValueError("Trusted production-patch capture must return a SHA-256 digest")
        return digest

    def _qualify_receipt(self, receipt: Receipt, *, check_id: str,
                         provenance: str, allowed_seconds: float) -> None:
        """Qualify budget/deadline before deriving observed-regression metrics."""
        proactive = provenance == "controller_recheck"
        if (receipt.elapsed_seconds > allowed_seconds + 1e-9 or
            self.budget.over_limit or
            (proactive and self.budget.remaining_repair_wall <= 0 and
             self.budget.used_wall_seconds > self.budget.max_wall_seconds - self.budget.final_reserve_seconds + 1e-9)):
            self.ledger.disqualify_receipt(receipt, "ACTUAL_RUNTIME_EXCEEDED_ALLOWED_DEADLINE")
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                       "check_id": check_id, "provenance": provenance,
                                       "reason": "actual_runtime_overrun_requires_live_cancellation"})

    def _run_check(self, check_id: str, provenance: str) -> Receipt | None:
        check = self.ledger.manifest.get(check_id)
        proactive = provenance == "controller_recheck"
        if not self.budget.can_run_check(check, protect_final_reserve=proactive):
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET", "check_id": check_id,
                                       "provenance": provenance})
            return None
        # CPU contract only: the caller-supplied executor is synchronous and NOT
        # cancellable. It must be replaced in GP02C-03 by a hard-deadline trusted
        # subprocess adapter, using a monotonic 840/900-second cutoff.
        allowed_seconds = (self.budget.remaining_repair_wall if proactive
                           else self.budget.remaining_wall)
        def recorded_executor(c: VisibleCheck) -> TestObservation:
            try:
                result = self.executor(c)
            except Exception as e:
                result = TestObservation(None, None, None, None, None,
                                         elapsed_seconds=0.0, setup_error=True)
                self.ledger.events.append({"type": "executor_error", "check_id": check_id,
                                           "error_type": type(e).__name__})
            self.budget.charge_visible_test(result.elapsed_seconds)
            return result
        return self.ledger.record_check(
            check_id, provenance, recorded_executor,
            receipt_qualifier=lambda receipt: self._qualify_receipt(
                receipt, check_id=check_id, provenance=provenance,
                allowed_seconds=allowed_seconds))

    def observe_agent_action(self, kind: str, action: Callable[[], object],
                             *, elapsed_seconds: float = 0.0) -> dict:
        """Wrap any tool action: read/edit/write/shell; no mandatory read gate.

        Charge the agent action itself; C post-edit checks are additional charged
        calls. Exceptions preserve before/after snapshots and are re-raised.
        """
        if (self.budget.used_tool_calls >= self.budget.max_tool_calls
            or self.budget.remaining_wall <= (
                self.budget.final_reserve_seconds if self.ledger.arm in {"B", "C"} else 0)):
            raise RuntimeError("Shared agent-tool budget exhausted or reserved for final validation")
        self.ledger.observe_workspace(f"before_{kind}")
        previously_passing = self.ledger.selected_current_passes()
        prior_snapshot = self.ledger.snapshot
        prior_receipts = {
            check_id: self.ledger.latest(check_id).receipt_id for check_id in previously_passing
        }
        exc: BaseException | None = None
        started_action = time.monotonic()
        try:
            result = action()
        except BaseException as error:
            exc = error
            result = None
        finally:
            # Actual wall time is charged for a live tool (the earlier integration
            # incorrectly passed 0.0). Explicit synthetic estimates are retained
            # for historical deterministic tests but never used by the ADK bridge.
            actual_action_seconds = time.monotonic() - started_action
            billed_action_seconds = (elapsed_seconds if elapsed_seconds > 0
                                     else actual_action_seconds)
            self.budget.charge_tool(billed_action_seconds)
            change = self.ledger.observe_workspace(f"after_{kind}")
        if change.changed and previously_passing:
            # Shared passive exposure instrumentation, identical B/C. STALE is
            # evidence invalidation, NOT proof of a test regression.
            old = {entry[0]: entry[1:] for entry in prior_snapshot.entries}
            new = {entry[0]: entry[1:] for entry in self.ledger.snapshot.entries}
            self.ledger.events.append({
                'type':'QUALIFIED_PASS_INVALIDATED_BY_RELEVANT_EDIT',
                'workspace_sha256_before': change.previous_sha256,
                'prior_receipts': prior_receipts,
                'changed_paths': sorted(path for path in old.keys() | new.keys()
                                        if old.get(path) != new.get(path)),
                'tool':kind, 'check_ids':list(previously_passing),
                'revision_before':change.previous_revision,
                'revision_after':change.new_revision,
                'workspace_sha256':change.new_sha256,
                'elapsed_agent_tool_seconds':round(billed_action_seconds,6)})
        reruns: list[dict[str, str]] = []
        if change.changed and self.ledger.arm == "C":
            for position, check_id in enumerate(previously_passing):
                if (position >= self.max_rechecks_per_edit or
                    self.auto_rechecks_used >= self.max_rechecks_per_run):
                    status = "UNVERIFIED_RECHECK_LIMIT"
                    self.ledger.events.append({"type": status, "check_id": check_id})
                else:
                    self.auto_rechecks_used += 1
                    receipt = self._run_check(check_id, "controller_recheck")
                    status = ("UNVERIFIED_BUDGET" if receipt is None or
                              receipt.receipt_id in self.ledger.unqualified_receipts or
                              self.budget.over_limit else self.ledger.status(receipt))
                reruns.append({"check_id": check_id, "status": status})
        # A later recheck can mutate the worktree and stale earlier rechecks.
        # Derive the entire feedback message from one final observed state.
        self.ledger.observe_workspace("post_recheck_feedback_sync")
        for row in reruns:
            if row["status"] in {"CURRENT_PASS", "CURRENT_FAIL", "STALE", "ERROR"}:
                row["status"] = self.ledger.current_status(row["check_id"])
        feedback = {"changed": change.changed, "revision": self.ledger.revision,
                    "workspace_sha256": self.ledger.snapshot.sha256,
                    "stale_selected_checks": list(previously_passing) if change.changed else [],
                    "rechecks": reruns}
        if exc is not None:
            raise exc
        return {"result": result, "feedback": feedback}

    def agent_visible_feedback(self, result: dict) -> dict:
        """Bounded and exclusively derived from allowlisted visible-check IDs.

        Never includes test stdout, stderr, hidden tests, reference patch,
        tool outputs or the model context; the integration must preserve this.
        """
        feedback = result["feedback"]
        return {"schema": "gp02c.feedback.v1", "revision": feedback["revision"],
                "changed": feedback["changed"],
                "workspace_sha256": feedback["workspace_sha256"],
                "stale": feedback["stale_selected_checks"][:32],
                "rechecks": feedback["rechecks"][:32]}

    def final_gate(self) -> GateDecision:
        if self.ledger.arm == "A":
            raise ValueError("Ordinary baseline A has no mandatory final gate")
        # Same B/C gate. Each decision revalidates all selected statuses *after*
        # the last source sync and is bound to the workspace SHA, revision,
        # production patch SHA and frozen manifest SHA.
        self.ledger.observe_workspace("final_gate_entry")
        initial_patch = self._patch_digest()
        attempts = 0
        max_attempts = len(self.ledger.manifest.checks) * 3

        def statuses_now() -> dict[str, str]:
            self.ledger.observe_workspace("final_gate_status_sync")
            return {c.check_id: self.ledger.current_status(c.check_id)
                    for c in self.ledger.manifest.checks}

        while True:
            statuses = statuses_now()
            # Do not declare exhaustion before reviewing the final allowed result.
            if all(s == "CURRENT_PASS" for s in statuses.values()):
                self.ledger.observe_workspace("final_gate_patch_sync")
                # The final sync may itself reveal a mutation. Recompute *after* it.
                statuses = statuses_now()
                workspace = self.ledger.snapshot.sha256
                revision = self.ledger.revision
                if self._patch_digest() != initial_patch:
                    return GateDecision(False, "PATCH_CHANGED_DURING_GATE", statuses, attempts)
                # One additional consistency check after the patch capture.
                self.ledger.observe_workspace("final_gate_approval_sync")
                statuses = statuses_now()
                if (all(s == "CURRENT_PASS" for s in statuses.values()) and
                    self.ledger.snapshot.sha256 == workspace and
                    self.ledger.revision == revision and
                    self._patch_digest() == initial_patch and
                    not self.budget.over_limit and self.budget.remaining_wall > 0):
                    return GateDecision(True, "VERIFIED_VISIBLE", statuses, attempts,
                                        accepted_patch_sha256=initial_patch,
                                        accepted_workspace_sha256=workspace,
                                        accepted_revision=revision)
            if self.budget.over_limit or self.budget.remaining_wall <= 0:
                return GateDecision(False, "UNVERIFIED_BUDGET", statuses, attempts)
            if any(s in {"CURRENT_FAIL", "ERROR"} for s in statuses.values()):
                return GateDecision(False, "VISIBLE_CHECK_FAILED", statuses, attempts)
            pending = [cid for cid, s in statuses.items()
                       if s in {"STALE", "NOT_EXECUTED"}]
            if not pending:
                return GateDecision(False, "UNVERIFIED", statuses, attempts)
            if attempts >= max_attempts:
                return GateDecision(False, "UNVERIFIED_RECHECK_LIMIT", statuses, attempts)
            receipt = self._run_check(pending[0], "final_gate")
            if receipt is None or receipt.receipt_id in self.ledger.unqualified_receipts:
                return GateDecision(False, "UNVERIFIED_BUDGET", statuses_now(), attempts)
            attempts += 1

# Reused from the pinned GP02C-04B source gp02c_live_adapter.py.
class LiveVerificationBridge(VerificationController):
    """B/C controller with actual time caps and a single-charge test executor.

    Only trusted subprocesses are cancellable here; ADK model requests and other
    agent tools must be routed via the equivalent hard-deadline worker in GP02C-04.
    """
    def __init__(self, ledger: EvidenceLedger, budget: SharedBudget,
                 executor: "GP03PublicPytestAdapter",
                 production_patch_sha256: Callable[[], str], **kwargs):
        super().__init__(ledger, budget, executor, production_patch_sha256, **kwargs)
        self.deadline = executor.deadline
        self.repair_deadline_breached = False

    def _qualify_receipt(self, receipt, *, check_id: str,
                         provenance: str, allowed_seconds: float) -> None:
        super()._qualify_receipt(receipt, check_id=check_id,
                                 provenance=provenance, allowed_seconds=allowed_seconds)
        phase = "proactive" if provenance == "controller_recheck" else (
            "final" if provenance == "final_gate" else "ordinary")
        if receipt.timed_out or self.deadline.remaining(phase) <= 0:
            if receipt.receipt_id not in self.ledger.unqualified_receipts:
                self.ledger.disqualify_receipt(receipt, "ABSOLUTE_DEADLINE")
                self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                           "check_id": check_id, "provenance": provenance,
                                           "reason": "absolute_deadline"})

    def _run_check(self, check_id: str, provenance: str):
        phase = "proactive" if provenance == "controller_recheck" else (
            "final" if provenance == "final_gate" else "ordinary")
        if self.deadline.remaining(phase) <= 0:
            self.ledger.events.append({"type": "UNVERIFIED_BUDGET", "check_id": check_id,
                                       "provenance": provenance, "reason": "absolute_deadline"})
            return None
        old = self.executor.phase
        self.executor.phase = phase
        try:
            receipt = super()._run_check(check_id, provenance)
        finally:
            self.executor.phase = old
        return receipt

    def observe_agent_action(self, kind: str, action: Callable[[], object], **kwargs) -> dict:
        if self.deadline.remaining("ordinary") <= 0:
            raise RuntimeError("Absolute repair deadline exhausted")
        start = time.monotonic()
        budget_before = self.budget.used_wall_seconds
        try:
            return super().observe_agent_action(kind, action, elapsed_seconds=0.0)
        finally:
            # super has already charged all automatic checks; charge only action
            # and wrapper overhead (not another tool call, not double testing).
            wall = time.monotonic() - start
            already_charged = self.budget.used_wall_seconds - budget_before
            self.budget.used_wall_seconds += max(0.0, wall - already_charged)
            if self.deadline.remaining("ordinary") <= 0:
                self.repair_deadline_breached = True
                self.ledger.events.append({"type": "UNVERIFIED_BUDGET",
                                           "reason": "repair_deadline_reached_during_action"})

    def final_gate(self) -> GateDecision:
        if self.repair_deadline_breached:
            return GateDecision(False, "UNVERIFIED_BUDGET", {}, 0)
        if self.deadline.remaining("final") <= 0:
            return GateDecision(False, "UNVERIFIED_BUDGET", {}, 0)
        decision = super().final_gate()
        if (not decision.accepted and any(e.get("type") == "UNVERIFIED_BUDGET"
                                         and e.get("reason") == "absolute_deadline"
                                         for e in self.ledger.events)):
            return GateDecision(False, "UNVERIFIED_BUDGET", decision.checks, decision.attempts)
        if decision.accepted and self.deadline.remaining("final") <= 0:
            return GateDecision(False, "UNVERIFIED_BUDGET", decision.checks, decision.attempts)
        return decision

    def verify_delivery_identity(self, approved: GateDecision) -> bool:
        """Recheck both full workspace and trusted production-patch identities."""
        if not approved.accepted:
            return False
        self.ledger.observe_workspace("delivery_identity_sync")
        return (self.ledger.snapshot.sha256 == approved.accepted_workspace_sha256
                and self.ledger.revision == approved.accepted_revision
                and self._patch_digest() == approved.accepted_patch_sha256
                and self.deadline.remaining("final") > 0)

# Reused from the pinned GP02C-04B source gp02c04b_checkpoint.py.
class CaptureHold(RuntimeError):
    pass

PRODUCTION_ROOTS = ('rich/',)

SCRATCH_COMPONENTS = frozenset({'test', 'tests', 'example', 'examples', 'scratch',
    'scripts', '__pycache__', '.pytest_cache', '.venv', 'venv'})

SCRATCH_PREFIXES = ('repro', 'reproduce', 'scratch', 'debug', 'temp', 'tmp', 'probe', 'test_')

def sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def run_git(repo: Path, args: list[str], *, allowed=(0,), timeout=20):
    p=subprocess.run(['git', '-C', str(repo), *args], capture_output=True,timeout=timeout)
    if p.returncode not in allowed:
        raise CaptureHold('Git command rejected: '+str(args[:3])+': '+p.stderr.decode(errors='replace')[:250])
    return p

def is_production(rel: str, roots=PRODUCTION_ROOTS) -> bool:
    p=Path(rel)
    if (not rel or '\\' in rel or p.is_absolute() or '..' in p.parts or
        not any(rel.startswith(root) for root in roots)):
        return False
    if p.suffix not in ('.py', '.pyi'):
        return False
    if any(part.lower() in SCRATCH_COMPONENTS for part in p.parts):
        return False
    name=p.name.lower()
    if name.startswith(SCRATCH_PREFIXES) or name.endswith('_test.py'):
        return False
    return True

def candidate_bytes(workspace: Path, *, roots=PRODUCTION_ROOTS, base_ref="HEAD") -> tuple[bytes,dict]:
    workspace=Path(workspace).resolve(strict=True)
    if not (workspace/'.git').is_dir():
        raise CaptureHold('Live source lacks expected Git repository')
    tracked=run_git(workspace,['diff','--no-renames','--name-only','-z',base_ref,'--']).stdout
    unknown=run_git(workspace,['ls-files','--others','--exclude-standard','-z']).stdout
    edited=[v.decode('utf-8','surrogateescape') for v in tracked.split(b'\0') if v]
    untracked=[v.decode('utf-8','surrogateescape') for v in unknown.split(b'\0') if v]
    tracked_eligible=sorted(p for p in edited if is_production(p, roots))
    new_eligible=sorted(p for p in untracked if is_production(p, roots))
    for name in tracked_eligible+new_eligible:
        f=workspace/name
        if f.is_symlink() or (f.exists() and not f.is_file()):
            raise CaptureHold('Unsupported production-source file kind: '+name)
    patch=b''
    if tracked_eligible:
        patch+=run_git(workspace,['diff','--no-renames','--binary',base_ref,'--',*tracked_eligible]).stdout
    for rel in new_eligible:
        result=run_git(workspace,['diff','--no-index','--binary','--','/dev/null',rel],allowed=(1,))
        if b'new file mode' not in result.stdout:
            raise CaptureHold('Invalid untracked source diff: '+rel)
        patch+=result.stdout
    manifest={'schema':'gp02c04b.production-candidate.v1',
              'policy':'FROZEN_PACKAGE_SOURCE_ONLY',
              'roots':list(roots),
              'tracked':tracked_eligible,'untracked':new_eligible,
              'excluded':sorted(set(edited+untracked)-set(tracked_eligible+new_eligible)),
              'sha256':sha256(patch),'bytes':len(patch)}
    return patch,manifest

# Reused from the pinned GP02C-04B source gp02c04b_assurance.py.
class LiveSourceHold(RuntimeError):
    pass

def selected_visible_command(command:str, check_id:str='rich_style_visible', *,
                             selector='tests/test_style.py', allow_src_pythonpath=False)->str|None:
    """Recognise *whole* predeclared visible test selection without shell execution.

    This reports whether the agent actually requested the frozen test group. The
    GP02C trusted subprocess is still independently executed with pinned argv.
    Nontrivial shell pipelines, test node selectors, exclusions and arbitrary
    flags do NOT qualify as the full selected check.
    """
    if not isinstance(command,str) or '\n' in command or '\r' in command:
        return None
    try: tokens=shlex.split(command.strip(),posix=True)
    except ValueError: return None
    if tokens[:3]==['cd','/workspace','&&']:
        tokens=tokens[3:]
    if allow_src_pythonpath and tokens[:1] == ['PYTHONPATH=src']:
        tokens = tokens[1:]
    if not tokens: return None
    if tokens[0] in {'pytest','pytest3'}:
        tokens=tokens[1:]
    elif (len(tokens)>=3 and tokens[0] in {'python','python3','/usr/bin/python3'}
          and tokens[1:3]==['-m','pytest']):
        tokens=tokens[3:]
    else:return None
    # Formatting flags only; never trust -k/-m/-x/--maxfail/::node or shell operators.
    formatting = {'-q', '-v', '--disable-warnings', '--no-header', '--no-summary', '--tb=short'}
    selection = [token for token in tokens if token not in formatting]
    return check_id if selection == [selector] else None

def _proc_pid_state(pid: int) -> tuple[str, int | None]:
    """Actual process state and start ticks; absence is not an error."""
    try:
        rest=(Path('/proc') / str(pid) / 'stat').read_text().rsplit(')',1)[1].strip().split()
        return rest[0], int(rest[19])
    except FileNotFoundError:
        return 'ABSENT', None
    except (OSError, ValueError, IndexError):
        return 'UNKNOWN', None

def _start_ticks(pid:int):
    return _proc_pid_state(pid)[1]

def register_trusted_child(proc, env:dict, argv:tuple[str,...]):
    """Fsync allowed pytest child group identity for external parent supervisor."""
    reg=env.get('GP02C_TRUSTED_PYTEST_REGISTRY')
    if not reg:return
    if ('-m','pytest') != tuple(argv[1:3]):
        raise LiveSourceHold('Refusing to register untrusted child argv')
    row={'event':'TRUSTED_PYTEST_START','pid':proc.pid,'pgid':os.getpgid(proc.pid),
         'start_ticks':_start_ticks(proc.pid),
         'argv_sha256':canonical_hash(tuple(argv))}
    if (row['pid']!=row['pgid'] or os.getsid(proc.pid)!=proc.pid
            or row['start_ticks'] is None):
        raise LiveSourceHold('Missing auditable pytest process-group identity')
    path=Path(reg);path.parent.mkdir(parents=True,exist_ok=True)
    with path.open('a',encoding='utf-8') as f:
        f.write(json.dumps(row,sort_keys=True)+'\n');f.flush();os.fsync(f.fileno())

def _group_members(pgid: int) -> list[int]:
    # Use the kernel's group and session IDs. The /proc stat IDs can come from a
    # different namespace. Probe the known leader even if /proc omits its entry.
    candidates = {pgid}
    candidates.update(int(p.name) for p in Path('/proc').iterdir() if p.name.isdigit())
    members = []
    for pid in sorted(candidates):
        try:
            if os.getpgid(pid) == pgid and os.getsid(pid) == pgid:
                members.append(pid)
        except ProcessLookupError:
            continue
    return members

def _group_exists(pgid: int) -> bool:
    try:
        os.killpg(pgid, 0)
    except ProcessLookupError:
        return False
    return True

def _process_active(pid: int) -> bool:
    # A pidfd distinguishes an exited, unreaped child from a running one, even
    # when the sandbox's process table still shows a sleeping process.
    if hasattr(os, 'pidfd_open'):
        try:
            fd = os.pidfd_open(pid)
        except ProcessLookupError:
            return False
        except OSError:
            pass
        else:
            try:
                poller = select.poll()
                poller.register(fd, select.POLLIN)
                return not poller.poll(0)
            finally:
                os.close(fd)
    return _proc_pid_state(pid)[0] not in {'Z', 'X', 'ABSENT'}

def _active_registered_group(pgid: int, start_ticks: int) -> list[int]:
    if not _group_exists(pgid):
        return []
    state, ticks = _proc_pid_state(pgid)
    if state != 'ABSENT' and ticks != start_ticks:
        raise LiveSourceHold('REGISTERED_LEADER_IDENTITY_UNVERIFIED')
    members = _group_members(pgid)
    if pgid in members and ticks != start_ticks:
        raise LiveSourceHold('REGISTERED_LEADER_IDENTITY_UNVERIFIED')
    if not members and _group_exists(pgid):
        raise LiveSourceHold('REGISTERED_GROUP_MEMBERSHIP_UNVERIFIED')
    return [pid for pid in members if _process_active(pid)]

def reap_trusted_groups(path:Path, *, grace:float=0.35)->dict:
    """Reap registered pytest sessions; hold on missing/malformed evidence.

    Reports only evidence-supported *registered-group* cleanup, never an
    unbounded claim about deliberately detached descendants or the vLLM server.
    """
    path=Path(path)
    report={'registry':str(path),'registry_present':path.is_file(),
            'registered':0,'groups_reaped':0,'errors':[], 'survivors':[],
            'verification_scope':'REGISTERED_TRUSTED_PROCESS_GROUPS',
            'verified':False}
    if not path.is_file():
        report['errors'].append('REGISTRY_MISSING')
        return report
    rows=[]
    try: lines=path.read_text(encoding='utf-8').splitlines()
    except (OSError, UnicodeError):
        report['errors'].append('REGISTRY_UNREADABLE')
        return report
    for idx,line in enumerate(lines,1):
        if not line.strip():
            report['errors'].append('EMPTY_REGISTRY_LINE_'+str(idx));continue
        try:
            r=json.loads(line)
            if not (isinstance(r,dict) and r.get('event')=='TRUSTED_PYTEST_START' and
                type(r.get('pid')) is int and r['pid']>1 and r.get('pgid')==r['pid'] and
                type(r.get('start_ticks')) is int and r['start_ticks']>=0):
                raise ValueError('bad record')
            rows.append(r)
        except (ValueError,TypeError,KeyError):
            report['errors'].append('MALFORMED_REGISTRATION_'+str(idx))
    report['registered']=len(rows)
    if len({r['pid'] for r in rows})!=len(rows):
        report['errors'].append('DUPLICATE_REGISTERED_PROCESS_GROUP')
    for row in rows:
        pgid, ticks = row['pgid'], row['start_ticks']
        try:
            active = _active_registered_group(pgid, ticks)
            if not active:
                continue
            for sig, wait_seconds in ((signal.SIGTERM, grace), (signal.SIGKILL, 0.25)):
                try:
                    os.killpg(pgid, sig)
                except ProcessLookupError:
                    break
                deadline = time.monotonic() + max(wait_seconds, 0.0)
                while time.monotonic() < deadline:
                    if not _active_registered_group(pgid, ticks):
                        break
                    time.sleep(0.02)
                if not _active_registered_group(pgid, ticks):
                    break
            survivors = _active_registered_group(pgid, ticks)
            if survivors:
                report['survivors'].append({'pgid': pgid, 'pids': survivors})
            else:
                report['groups_reaped'] += 1
        except (OSError, LiveSourceHold) as exc:
            report['errors'].append(f'GROUP_{pgid}_{type(exc).__name__}: {exc}')
    report['verified']=report['registry_present'] and not report['errors'] and not report['survivors']
    return report

# Reused from the pinned GP02C-04B source gp02c_adk_bridge.py.
class PilotIntegrationHold(RuntimeError):
    """Stop rather than claiming ADK integration on an incompatible runtime."""

def _feedback_result(result: object, feedback: dict) -> object:
    """Preserve dict vs JSON string. Never fabricate an unknown tool schema."""
    if isinstance(result, dict):
        return {**result, "gp02c_evidence": feedback}
    if isinstance(result, str):
        try:
            parsed = json.loads(result)
        except (ValueError, TypeError):
            raise PilotIntegrationHold("Agent tool returned non-JSON text; feedback mapping not certified")
        if not isinstance(parsed, dict):
            raise PilotIntegrationHold("Agent tool JSON result must be an object")
        return json.dumps({**parsed, "gp02c_evidence": feedback}, ensure_ascii=False)
    raise PilotIntegrationHold("Unsupported agent tool response type")

@dataclass(frozen=True)
class ToolBoundaryEvent:
    tool: str
    elapsed_seconds: float
    workspace_sha256: str
    revision: int
    check_id: str | None
    status: str

class ADKVisibleEvidenceBoundary:
    """Wraps existing context.create_tools tools; no extra investigation gate.

    Selected test commands are opt-in EXACT aliases, with the same mapping in
    A/B/C. After the original agent run_command, an independent trusted
    visible pytest check is executed and fully budgeted. Other commands are
    unrestricted except existing GP02B protected-path policy.

    For real Kaggle: source_root MUST resolve to the live agent sandbox's
    actual /workspace host-mounted directory. If that path cannot be verified,
    raise PilotIntegrationHold before any inference.
    """
    REQUIRED = frozenset({"submit_patch", "run_command", "edit_file", "write_file"})

    def __init__(self, controller: LiveVerificationBridge, *,
                 selected_command_aliases: Mapping[str, str],
                 required_tools: frozenset[str] | None = None,
                 event_sink: Callable[[str,dict],object] | None = None):
        self.controller = controller
        self.aliases = dict(selected_command_aliases)
        self.required_tools = required_tools if required_tools is not None else self.REQUIRED
        self.events: list[ToolBoundaryEvent] = []
        self.event_sink = event_sink
        self.submission_status = "NOT_ATTEMPTED"
        self.last_gate = None
        for alias, check in self.aliases.items():
            if not isinstance(alias, str) or not alias.strip() or '\n' in alias:
                raise ValueError("Malformed selected visible command alias")
            self.controller.ledger.manifest.get(check)
        # Never let an evaluator-only command map into the agent's visible lane.
        forbidden = re.compile(r"(?:evaluator_only|test_patch|reference_patch|protected[_-]test)", re.I)
        if any(forbidden.search(alias) for alias in self.aliases):
            raise ValueError("Protected test command not permitted")

    def _result_event(self, name: str, started: float, check_id: str | None, status: str):
        led = self.controller.ledger
        led.observe_workspace("adk_tool_event_sync")
        self.events.append(ToolBoundaryEvent(name, max(0, time.monotonic()-started),
                                             led.snapshot.sha256, led.revision,
                                             check_id, status))
        if self.event_sink is not None:
            self.event_sink('ADK_TOOL_FINISHED',{
                'tool':name,'check_id':check_id,'status':status,
                'revision':led.revision,'workspace_sha256':led.snapshot.sha256,
                'elapsed_seconds':round(time.monotonic()-started,5),
                'budget':self.controller.budget.summary()})

    def _submit(self, name: str, fn: Callable, args: tuple, kwargs: dict):
        # The final gate bills its own trusted checks. Do not bill gate elapsed
        # AGAIN as if it were the submit_patch call itself.
        started = time.monotonic()
        ledger = self.controller.ledger
        if ledger.arm == "A":
            # Baseline has no final gate. Tool call still counts once.
            row = self.controller.observe_agent_action(name, lambda: fn(*args, **kwargs))
            self.submission_status = "BASELINE_SUBMISSION_ATTEMPTED"
            self._result_event(name, started, None, self.submission_status)
            return row["result"]
        gate = self.controller.final_gate()
        self.last_gate = gate
        if not gate.accepted:
            if self.controller.budget.used_tool_calls < self.controller.budget.max_tool_calls:
                self.controller.budget.charge_tool(0.0)
            self.submission_status = "BLOCKED_" + gate.status
            self._result_event(name, started, None, self.submission_status)
            return {"status": "error", "error_type": "FreshVisibleVerificationBlocked",
                    "error_message": "Final selected visible validation was not verified",
                    "gp02c_gate_status": gate.status}
        if self.controller.budget.used_tool_calls >= self.controller.budget.max_tool_calls:
            self.submission_status = "BLOCKED_TOOL_BUDGET"
            self._result_event(name, started, None, self.submission_status)
            return {"status": "error", "error_type": "FreshVisibleVerificationBlocked",
                    "error_message": "No tool budget remains for submission",
                    "gp02c_gate_status": "UNVERIFIED_BUDGET"}
        if not self.controller.verify_delivery_identity(gate):
            self.submission_status = "BLOCKED_IDENTITY_CHANGED"
            self._result_event(name, started, None, self.submission_status)
            return {"status": "error", "error_type": "FreshVisibleVerificationBlocked",
                    "error_message": "Workspace or patch changed after final validation",
                    "gp02c_gate_status": "PATCH_OR_WORKSPACE_CHANGED"}
        # The 840s repair cutoff does NOT block a final 840--900s submission.
        # Charge the submission tool separately; no model-selected investigation
        # action is required. The external worker supervisor owns hard cancellation.
        if self.controller.deadline.remaining("final") <= 0:
            self.submission_status = "BLOCKED_ABSOLUTE_DEADLINE"
            self._result_event(name, started, None, self.submission_status)
            return {"status": "error", "error_type": "FreshVisibleVerificationBlocked",
                    "gp02c_gate_status": "UNVERIFIED_BUDGET"}
        self.controller.ledger.observe_workspace("before_submit_patch")
        submit_started = time.monotonic()
        try:
            result = fn(*args, **kwargs)
        finally:
            elapsed_submit_only = time.monotonic() - submit_started
            self.controller.ledger.observe_workspace("after_submit_patch")
            self.controller.budget.charge_tool(elapsed_submit_only)
        self.submission_status = ("SUBMITTED_GATE_VALID" if
                                  self.controller.verify_delivery_identity(gate)
                                  and not self.controller.budget.over_limit
                                  else "SUBMITTED_BUT_IDENTITY_OR_BUDGET_INVALID")
        self._result_event(name, started, None, self.submission_status)
        return result

    def _ordinary(self, name: str, fn: Callable, args: tuple, kwargs: dict):
        started = time.monotonic()
        data = self.controller.observe_agent_action(name, lambda: fn(*args, **kwargs))
        result = data["result"]
        feedback = self.controller.agent_visible_feedback(data)
        chosen_id = None
        if name == "run_command" and isinstance(kwargs.get("command"), str):
            alias = kwargs["command"].strip()
            chosen_id = self.aliases.get(alias)
            if chosen_id is None and 'rich_style_visible' in self.aliases.values():
                chosen_id = selected_visible_command(alias)
            raw = result
            if isinstance(raw, str):
                try:
                    raw = json.loads(raw)
                except ValueError:
                    raw = {}
            raw_exit = raw.get('exit_code') if isinstance(raw, dict) else None
            if chosen_id is None:
                for check in self.controller.ledger.manifest.checks:
                    if not check.check_id.startswith(('dev_', 'gp03_')) or check.check_id not in self.aliases.values():
                        continue
                    chosen_id = selected_visible_command(
                        alias, check.check_id, selector=check.test_selection,
                        allow_src_pythonpath=self.controller.ledger.task_id == 'httpx_3672')
                    if chosen_id is None and raw_exit == 0:
                        # A successful explicit whole-suite request covers the
                        # selected file. Only the independently rerun file is
                        # eligible for a receipt. Never match bare pytest.
                        for suite in ('tests', 'tests/'):
                            chosen_id = selected_visible_command(
                                alias, check.check_id, selector=suite,
                                allow_src_pythonpath=self.controller.ledger.task_id == 'httpx_3672')
                            if chosen_id is not None:
                                break
                    if chosen_id is not None:
                        break
            if chosen_id is not None and chosen_id not in self.aliases.values():
                chosen_id = None
            if chosen_id is not None:
                # Common A/B/C trusted observation: one additional qualified
                # visible check, charged to the common shared tool budget.
                receipt = self.controller._run_check(chosen_id, "agent")
                self.controller.ledger.observe_workspace("after_trusted_selected_check")
                if receipt is not None:
                    status = self.controller.ledger.status(receipt)
                else:
                    status = "UNVERIFIED_BUDGET"
                self.controller.ledger.events.append({"type": "ADK_SELECTED_VISIBLE_CHECK",
                                                       "check_id": chosen_id,
                                                       "status": status,
                                                       "receipt_id": receipt.receipt_id if receipt else None,
                                                       "original_exit_code": raw_exit,
                                                       "original_command": alias})
        self._result_event(name, started, chosen_id, "COMPLETED")
        # In B (and A), the controller is observational before submission.
        if self.controller.ledger.arm != "C" or not feedback.get("changed"):
            return result
        if (not isinstance(result, (dict, str))):
            raise PilotIntegrationHold("Unexpected ADK tool return type during C feedback")
        return _feedback_result(result, feedback)

    def wrap(self, tools: Mapping[str, Callable]) -> dict[str, Callable]:
        if not isinstance(tools, Mapping) or not self.required_tools.issubset(tools):
            raise PilotIntegrationHold("Required v2.7 agent tools unavailable in ADK registry")
        wrapped = {}
        for name, fn in tools.items():
            if not callable(fn):
                raise PilotIntegrationHold("Noncallable ADK tool")
            @functools.wraps(fn)
            def invoke(*args, _name=name, _fn=fn, **kwargs):
                if self.event_sink is not None:
                    self.event_sink('ADK_TOOL_STARTED',{'tool':_name,
                        'revision':self.controller.ledger.revision,
                        'workspace_sha256':self.controller.ledger.snapshot.sha256})
                if _name == "submit_patch":
                    return self._submit(_name, _fn, args, kwargs)
                return self._ordinary(_name, _fn, args, kwargs)
            # Python signature introspection follows __wrapped__ automatically.
            if inspect.signature(invoke) != inspect.signature(fn):
                raise PilotIntegrationHold("Callable signature not preserved")
            wrapped[name] = invoke
        return wrapped

    def attach_to_context(self, context: object) -> Callable[[], None]:
        """Reversible pilot-only insertion AFTER v2.7 existing tool policy."""
        create_tools = getattr(context, "create_tools", None)
        if not callable(create_tools):
            raise PilotIntegrationHold("Expected ADK SwegemmaContext.create_tools is unavailable")
        original = create_tools
        @functools.wraps(original)
        def create_tools_with_evidence():
            return self.wrap(original())
        context.create_tools = create_tools_with_evidence
        def restore():
            context.create_tools = original
        return restore

    def sync_model_calls(self, observed_count: int) -> None:
        """Live worker must call from model-use observer; fail on excess calls."""
        budget = self.controller.budget
        if observed_count < budget.used_model_calls:
            raise PilotIntegrationHold("Model counter went backward")
        while budget.used_model_calls < observed_count:
            budget.charge_model(0.0)
        if self.event_sink is not None:
            self.event_sink('MODEL_COUNTER_SYNC',{'model_calls':budget.used_model_calls,
                                                   'budget':budget.summary()})

    def export_audit(self) -> dict:
        return {"schema": "gp02c.adk-tool-boundary.v1", "arm": self.controller.ledger.arm,
                "events": [asdict(e) for e in self.events],
                "selected_command_aliases": dict(self.aliases),
                "submission_status": self.submission_status,
                "last_gate": asdict(self.last_gate) if self.last_gate else None,
                "budget": self.controller.budget.summary()}

PAIR_BUDGET = {"ordinary_seconds": 840, "worker_seconds": 900, "final_reserve_seconds": 60,
               "max_tool_calls": 60, "max_model_calls": 60}


class GP03ProductionFingerprinter:
    """Track implementation changes separately from the immutable test definition."""

    def __init__(self, root, production_root):
        self.root = Path(root).resolve(strict=True)
        self.production_root = production_root

    def snapshot(self):
        package = self.root / self.production_root
        if not package.is_dir() or package.resolve() != package:
            raise CPUHold("PRODUCTION_ROOT_CHANGED", "The implementation root is missing or linked")
        entries = []
        for name, kind, digest in WorkspaceFingerprinter(package).snapshot().entries:
            relative = self.production_root + name
            if kind == "L":
                raise CPUHold("PRODUCTION_LINK_HOLD", f"Unexpected implementation link {relative}")
            if is_production(relative, (self.production_root,)):
                entries.append((relative, kind, digest))
        return Snapshot(canonical_hash(entries), len(entries), tuple(entries))


def gp03_test_directory_links(root):
    links = {}
    for parent, directories, _ in os.walk(root / "tests", followlinks=False):
        for name in directories:
            path = Path(parent) / name
            if path.is_symlink():
                links[path.relative_to(root).as_posix()] = os.readlink(path)
    return links


class GP03PairedLedger(EvidenceLedger):
    def __init__(self, root, visible, *, check_manifest, adapter, **identity):
        super().__init__(root, visible, **identity)
        self.fingerprinter = GP03ProductionFingerprinter(root, check_manifest["production_root"])
        self.snapshot = self.fingerprinter.snapshot()
        self.adapter = adapter

    def _definition_matches(self, check):
        try:
            self.adapter._validate_definition(check, self.fingerprinter.root)
        except (CPUHold, OSError, RuntimeError, ValueError):
            return False
        return True


class GP03PairedAdapter(GP03PublicPytestAdapter):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.directory_links = gp03_test_directory_links(self.source_root)

    def _validate_definition(self, check, root):
        super()._validate_definition(check, root)
        if gp03_test_directory_links(root) != self.directory_links:
            raise CPUHold("VISIBLE_LINK_CHANGED", "A public test directory link differs from the baseline")


def attach_gp03_pair(*, context, root, results_root, manifest, arm, seed, run_id, environment_sha,
                     started_monotonic, baseline_ref, event_sink=None):
    """Bind B or C to the same public manifest and one fresh per-arm deadline."""
    root = Path(root).resolve(strict=True)
    if arm not in {"B", "C"} or seed != 42:
        raise CPUHold("PAIR_IDENTITY", "Expected B/C and the registered seed label")
    if manifest["environment_sha256"] != environment_sha:
        raise CPUHold("PAIR_ENVIRONMENT", "The public-check runtime differs from qualification")
    if WorkspaceFingerprinter(root).snapshot().sha256 != manifest["baseline_source_sha256"]:
        raise CPUHold("PAIR_BASELINE_SOURCE", "The initial working tree differs from the accepted baseline")
    deadline = RealDeadline(PAIR_BUDGET["ordinary_seconds"], PAIR_BUDGET["worker_seconds"],
                            started_monotonic=started_monotonic)
    adapter = GP03PairedAdapter(
        source_root=root, manifest=manifest, environment_sha256=environment_sha, deadline=deadline,
        results_root=results_root,
    )
    check = adapter.check
    adapter._validate_definition(check, root)
    visible = VisibleManifest((check,), strict=True)
    ledger = GP03PairedLedger(root, visible, check_manifest=manifest, adapter=adapter,
                             run_id=run_id, task_id=manifest["task_id"], arm=arm, seed=seed,
                             environment_sha256=environment_sha)
    budget = SharedBudget(max_wall_seconds=PAIR_BUDGET["worker_seconds"],
                          final_reserve_seconds=PAIR_BUDGET["final_reserve_seconds"],
                          max_tool_calls=PAIR_BUDGET["max_tool_calls"], max_model_calls=PAIR_BUDGET["max_model_calls"])

    def production_patch_sha():
        patch, _ = candidate_bytes(root, roots=(manifest["production_root"],), base_ref=baseline_ref)
        return hashlib.sha256(patch).hexdigest()

    controller = LiveVerificationBridge(ledger, budget, adapter, production_patch_sha)
    selector = check.test_selection
    boundary = ADKVisibleEvidenceBoundary(
        controller, selected_command_aliases={f"pytest {selector}": check.check_id,
                                              f"python -m pytest {selector}": check.check_id},
        event_sink=event_sink,
    )
    restore = boundary.attach_to_context(context)
    return boundary, restore

def load_acceptance_pins(working, explicit_path, evidence, plan):
    if explicit_path is not None:
        path = Path(explicit_path)
    else:
        path = working / evidence["filename"]
        if not path.is_file():
            matches = sorted(Path("/kaggle/input").rglob(evidence["filename"]))
            path = next((match for match in matches if sha256_file(match) == evidence["sha256"]), path)
    if not path.is_file():
        raise CPUHold("CPU_EVIDENCE_MISSING", f"Place {evidence['filename']} in /kaggle/working or mount it as an input")
    if sha256_file(path) != evidence["sha256"]:
        raise CPUHold("CPU_EVIDENCE_MISMATCH", "The qualification evidence ZIP differs from its SHA-256 pin")
    index = verify_zip(path)
    pins = {}
    with zipfile.ZipFile(path) as archive:
        report = json.loads(archive.read("GP03_CPU_Report.json"))
        if (report["run_sha256"] != evidence["qualification_run_sha256"]
                or report["notebook_recipe_sha256"] != evidence["notebook_recipe_sha256"]
                or report["tasks_sha256"] != evidence["tasks_sha256"]
                or report["gp00_split_sha256"] != evidence["gp00_split_sha256"]
                or environment_fingerprint(report["environment"]) != evidence["cpu_environment_sha256"]
                or report["environment_sha256"] != evidence["cpu_environment_sha256"]
                or report["evaluation_frozen"] or report["model_calls"] or report["protected_verifier_calls"]):
            raise CPUHold("CPU_REPORT_MISMATCH", "The qualification report disagrees with the verified public evidence")
        rows = {row["task_id"]: row for row in report["rows"]}
        for entry in plan:
            task = entry["task_id"]
            prefix = f"tasks/{task}/"
            manifest_name = prefix + "CPU_Check_Manifest.json"
            receipt_name = prefix + "Task_Receipt.json"
            result_name = prefix + "Task_Result.json"
            junit_name = prefix + "baseline/junit.xml"
            manifest = json.loads(archive.read(manifest_name))
            receipt = json.loads(archive.read(receipt_name))
            prior = json.loads(archive.read(result_name))
            expected = junit_counts(archive.read(junit_name))
            run_sha = (report["prior_evidence"]["run_sha256"] if entry["cpu_receipt_origin"] == "v1.0_retained"
                       else report["run_sha256"])
            check = gp03_check_from_manifest(manifest)
            same_identity = all(manifest[k] == entry[k] for k in (
                "task_id", "repo", "base_commit", "native_head", "native_tree", "snapshot_sha256", "baseline_outcome",
            ))
            if (not same_identity or manifest_name != entry["cpu_check_member"]
                    or index[manifest_name]["sha256"] != entry["cpu_check_sha256"]
                    or index[receipt_name]["sha256"] != entry["cpu_receipt_sha256"]
                    or receipt["task_id"] != task or receipt["run_sha256"] != run_sha
                    or receipt["files"] != {k.removeprefix(prefix): v for k, v in index.items()
                                             if k.startswith(prefix) and k != receipt_name}
                    or prior != rows[task] or prior["status"] != "CPU_READY"
                    or manifest["schema"] != "gp03.public-cpu-check.v1"
                    or manifest["environment_sha256"] != evidence["cpu_environment_sha256"]
                    or manifest["source_policy"] != "gp02c_workspace_all_files_excluding_fixed_generated_artifacts"
                    or manifest["baseline_junit_sha256"] != index[junit_name]["sha256"]
                    or manifest["expected_nodeids_sha256"] != canonical_hash(manifest["expected_nodeids"])
                    or list(check.command) != entry["public_check_argv"]
                    or check.expected_collected != entry["expected_collected"]
                    or check.expected_collected != len(manifest["expected_nodeids"])
                    or check.expected_collected != expected[0]):
                raise CPUHold("TASK_RECEIPT_MISMATCH", f"Qualification receipt or check differs from its pins for {task}")
            link_file = prefix + "Snapshot_Links.json"
            links = json.loads(archive.read(link_file)) if link_file in index else []
            pins[task] = (manifest, expected, links)
    return pins

def verify_public_acceptance(path, working, pins, plan):
    if path is None:
        path = working / GP03_ADAPTER_EVIDENCE_NAME
        if not path.is_file():
            matches = sorted(Path("/kaggle/input").rglob(GP03_ADAPTER_EVIDENCE_NAME))
            path = next((p for p in matches if sha256_file(p) == GP03_ADAPTER_EVIDENCE_SHA256), path)
    path = Path(path)
    if not path.is_file():
        raise CPUHold("ADAPTER_EVIDENCE_MISSING", f"Make {GP03_ADAPTER_EVIDENCE_NAME} available to the notebook")
    if sha256_file(path) != GP03_ADAPTER_EVIDENCE_SHA256:
        raise CPUHold("ADAPTER_EVIDENCE_MISMATCH", "Public adapter ZIP differs from the reviewed evidence")
    index = verify_zip(path)
    with zipfile.ZipFile(path) as archive:
        def read(name):
            return json.loads(archive.read(name))

        report = read("GP03_Public_Adapter_Report.json")
        profile = read("CPU_Environment.json")
        if (read("GP03_Proposed_Pair_Order.json") != {"evaluation_frozen": False, "pairs": plan}
                or report["status"] != "GP03_PUBLIC_ADAPTER_ACCEPTED"
                or report["adapter_recipe_sha256"] != ADAPTER_RECIPE_SHA256
                or report["eligible_pool_sha256"] != ELIGIBLE_POOL_SHA256
                or report["qualification_evidence_sha256"] != GP03_CPU_EVIDENCE["sha256"]
                or report["environment_sha256"] != environment_fingerprint(profile)
                or report["environment_sha256"] != GP03_CPU_EVIDENCE["cpu_environment_sha256"]
                or report["model_calls"] or report["protected_verifier_calls"] or report["held_out_agent_attempts"]
                or report["evaluation_frozen"] or report["preflight_hold"]
                or (report["planned"], report["ready"], report["holds"], report["pending"]) != (len(plan), len(plan), 0, 0)
                or [row["task_id"] for row in report["rows"]] != [entry["task_id"] for entry in plan]):
            raise CPUHold("ADAPTER_REPORT_MISMATCH", "The accepted report does not match the pinned cohort and runtime")
        for row in report["rows"]:
            task = row["task_id"]
            manifest, counts, links = pins[task]
            task_dir = f"tasks/{task}"
            check_dir = task_dir + "/checks/check_0001"
            check = read(check_dir + "/Public_Check_Receipt.json")
            phase = read(check_dir + "/pytest/command.json")
            observed_links = [{"path": link["path"], "target": link["target"]}
                              for link in links if link["kind"] == "symlink"]
            if (read(task_dir + "/Adapter_Acceptance.json") != row
                    or row["status"] != "ADAPTER_READY" or row["reason"] is not None
                    or row["outcome"] != manifest["baseline_outcome"]
                    or row["check_seconds"] > PAIR_BUDGET["final_reserve_seconds"]
                    or row["isolated_links"] != len(observed_links) or check["isolated_links"] != observed_links
                    or check["check_id"] != manifest["check"]["check_id"] or check["phase"] != "final"
                    or check["reason"] is not None or check["outcome"] != row["outcome"]
                    or check["actual_nodeids_sha256"] != manifest["expected_nodeids_sha256"]
                    or row["actual_nodeids_sha256"] != manifest["expected_nodeids_sha256"]
                    or any(check[k] != manifest["baseline_source_sha256"] for k in
                           ("source_sha256_before", "source_sha256_after", "isolated_source_sha256_after"))
                    or phase["argv"][:6] != manifest["check"]["command"] or phase["timed_out"]
                    or check["junit_sha256"] != index[check_dir + "/pytest/junit.xml"]["sha256"]
                    or junit_counts(archive.read(check_dir + "/pytest/junit.xml")) != counts):
                raise CPUHold("ADAPTER_RECEIPT_MISMATCH", f"Accepted public observation changed for {task}")
    return report


def proposed_pair_lock(plan):
    return {
        "schema": "gp03.paired-runner-lock-draft.v1", "evaluation_frozen": False,
        "public_adapter_evidence_sha256": GP03_ADAPTER_EVIDENCE_SHA256,
        "cpu_qualification_evidence_sha256": GP03_CPU_EVIDENCE["sha256"],
        "eligible_pool_sha256": ELIGIBLE_POOL_SHA256,
        "paired_wiring_recipe_sha256": GP03_PAIR_RECIPE_SHA256,
        "pair_order_sha256": canonical_hash(plan), "pairs": plan, "budget": PAIR_BUDGET,
        "planned_batches": [5, 4], "worker_wall_cap_minutes": 270, "planning_reserve_minutes": 90,
        "seed_label": 42, "effective_live_request_seed_verified": False,
        "pending": ["official agent-worker integration", "L4 runtime and serving pins", "live request seed",
                    "protected grading and paired scoring pins", "launch quota confirmation"],
    }


## 4. gp03_organizer_runtime.py

In [ ]:
%%gp03_source gp03_organizer_runtime.py
import dataclasses
from contextlib import contextmanager
from datetime import datetime, timezone
import importlib
import importlib.metadata
import inspect
import json
import os
from pathlib import Path
import shutil
import socket
import subprocess
import sys
import threading
import time
import zipfile

from gp03_public_runtime import sha256_file, write_json, _cancel_process_group


# Reused from the pinned GP02C-04B source gp02c_dev_registry.py.
BASELINE_ZIP_SHA256 = '9d0021324097f386a13f88bbce1e5344afa3e2a3244a75c28b7791d9efcd0b3e'

TASKS_SHA256 = 'e4b3fd60f69dbc2b9213e54eeb9636db78aefe92c1d06269d73d9f5f8f3c8ad6'

RUNNER_SHA256 = '4d7eccabd203c26f7c39e882684d2fd851f383f6a5c76c4a75b0e559cd5d6a18'

VERIFIER_SHA256 = '99e2a5fdab2be2b5e0388de29080da9d6e7003cf8b53299cda790464f4ecd2aa'

SELECTED_WHEELS = [{'name': 'adk_eval_core-0.1.0-py3-none-any.whl',
  'sha256': 'e614e1b2fe8d886a634871e4c57f9887e1cc66e8c27300f8503a43c6e3ca82f3'},
 {'name': 'adk_submission-0.2.13-py3-none-any.whl',
  'sha256': 'f156f0bd84580ccb9334726d114d7a5481f16d3a6e2887cc4d276ced0ef8d99d'},
 {'name': 'swegemma-0.2.11-py3-none-any.whl',
  'sha256': '9ad876581f3acde8ec3c369858f0834e2b9f41a3cffce3ca42f09cb090aea9d3'},
 {'name': 'google_adk-1.36.1-py3-none-any.whl',
  'sha256': '1a2f6868c509e3151fb0de3575a7d18b45c338be86f420924dad74e7193631a0'},
 {'name': 'vllm-0.19.1-cp38-abi3-manylinux_2_31_x86_64.whl',
  'sha256': '73c69bafca24fbdad44690c44d6c6446514351e6f3cc9a6057cba05e66051310'},
 {'name': 'transformers-5.13.1-py3-none-any.whl',
  'sha256': '53f0ea8aa397e29244c2377ba981bcaf0c87adcf44fbdd447ef6306522afcacd'},
 {'name': 'tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl',
  'sha256': '369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67'},
 {'name': 'compressed_tensors-0.15.0.1-py3-none-any.whl',
  'sha256': 'e1b1f322e82e475715e242bad46925a304ea8e5c98b5055a15b8eb22fb6bfea9'},
 {'name': 'safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl',
  'sha256': 'f216afd4ba743e744c86cbd08e7185efa8749e4c905f14e4e977d65576466cb0'}]

MODEL_NAME = 'gemma-4-31b-it-qat-w4a16-ct'

MODEL_CONTEXT = 32768

OUTPUT_TOKENS = 2048

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

# Reused from the pinned GP02C-04B source gp02c_dev_pilot.py.
def find_baseline(override, working):
    if override:
        candidates = [Path(override)]
    else:
        candidates = []
        for parent in (Path('/kaggle/input'), Path(working)):
            candidates += sorted(parent.rglob('GP02B_v2_7_rich_2943*.zip'))
        baseline_tree = Path(working) / 'Gemma4_Paper/GP02B_Cross_Task'
        candidates += sorted(baseline_tree.rglob('GP02A_results.zip'))
    for path in candidates:
        if path.is_file() and sha256_file(path) == BASELINE_ZIP_SHA256:
            return path
    raise FileNotFoundError(
        'Attach the original GP02B v2.7 rich_2943 evidence ZIP, or set B_BASELINE_ZIP. '
        'Use the GP02B archive rather than the later GP02C B/C output ZIP.'
    )

def restore_frozen_submission(archive, destination):
    destination.mkdir()
    with zipfile.ZipFile(archive) as z:
        prefix = 'run_sources/submission/'
        for name in z.namelist():
            if not name.startswith(prefix) or name.endswith('/'):
                continue
            relative = Path(name.removeprefix(prefix))
            if relative.is_absolute() or '..' in relative.parts:
                raise RuntimeError('Unsafe frozen submission path')
            path = destination / relative
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(z.read(name))
        template = json.loads(z.read('trial_config.json'))
        config = json.loads(z.read('server_config.json'))['config']
        policy = json.loads(z.read('generation_policy.json'))
    if (template['model_name'] != MODEL_NAME or config['tensor_parallel_size'] != 4
        or config['max_model_len'] != MODEL_CONTEXT or config['model'] != template['model_path']
        or config['host'] != '127.0.0.1' or config['tool_call_parser'] != 'gemma4'
        or config['reasoning_parser'] != 'gemma4'
        or policy['max_output_tokens'] != OUTPUT_TOKENS
        or policy['sampling_yaml_sha256'] != sha256_file(destination / 'configs/sampling.yaml')
        or template['expected_composed_system_sha256'] != sha256_file(destination / 'prompts/system.md')
        or template['tasks_sha256'] != TASKS_SHA256
        or template['agent_runner_source_sha256'] != RUNNER_SHA256
        or template['verification_source_sha256'] != VERIFIER_SHA256):
        raise RuntimeError('Original v2.7 model, prompt, sampler or harness pin changed')
    return template, config

def _require_l4():
    try:
        names = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            timeout=15, text=True, stderr=subprocess.STDOUT,
        ).splitlines()
    except (OSError, subprocess.SubprocessError) as exc:
        raise RuntimeError('Select the Kaggle L4 x4 accelerator before enabling B') from exc
    if len(names) != 4 or any('L4' not in name for name in names):
        raise RuntimeError(f'The frozen serving profile requires L4 x4. Detected {names}')
    return names

def _offline_environment():
    os.environ.update({
        'TRANSFORMERS_NO_TF': '1', 'TRANSFORMERS_NO_FLAX': '1',
        'TRANSFORMERS_OFFLINE': '1', 'HF_HUB_OFFLINE': '1',
        'VLLM_NO_USAGE_STATS': '1', 'VLLM_WORKER_MULTIPROC_METHOD': 'spawn',
        'VLLM_ENGINE_READY_TIMEOUT_S': '1200',
        'VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS': '1',
        'NCCL_DEBUG': 'WARN', 'LITELLM_LOCAL_MODEL_COST_MAP': 'True',
        'LITELLM_TELEMETRY': 'False', 'LITELLM_LOG': 'WARNING',
        'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
        'OTEL_SDK_DISABLED': 'true', 'OTEL_PYTHON_DISABLED': 'true',
        'PIP_NO_INDEX': '1', 'PIP_NO_INPUT': '1', 'DO_NOT_TRACK': '1',
    })
    os.environ.setdefault('LOCAL_API_KEY', 'EMPTY')
    os.environ.setdefault('OPENAI_API_KEY', 'EMPTY')

def prepare_gpu_runtime(wheelhouse, output, allow_install):
    """Use the mounted organizer wheels and check the archived runtime pins."""
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename

    compatible = {}
    tags = set(sys_tags())
    wheel_dir = output / 'offline_wheel_aliases'
    wheel_dir.mkdir()
    for path in sorted(Path(wheelhouse).rglob('*.whl')):
        if 'cutlass' in path.name.lower():
            continue
        name = path.name.replace('cu128', '+cu128') if 'cu128' in path.name and '+' not in path.name else path.name
        try:
            distribution, version, _, wheel_tags = parse_wheel_filename(name)
        except ValueError:
            continue
        if tags.isdisjoint(wheel_tags):
            continue
        if name in compatible:
            raise RuntimeError(f'Duplicate compatible organizer wheel {name}')
        compatible[name] = (path, distribution, version)

    expected = {'torch': '2.10.0+cu128'}
    for pin in SELECTED_WHEELS:
        name = pin['name']
        if name not in compatible or sha256_file(compatible[name][0]) != pin['sha256']:
            raise RuntimeError(f'Missing or changed organizer wheel {name}. Mount {wheelhouse}.')
        _, distribution, version = compatible[name]
        expected[distribution] = str(version)

    def actual_versions():
        versions = {}
        for name in expected:
            try:
                versions[name] = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                versions[name] = None
        return versions

    actual = actual_versions()
    if actual != expected:
        if not allow_install:
            raise RuntimeError(f'Organizer runtime mismatch. Expected {expected}. Detected {actual}.')
        choices = {}
        for name, (path, distribution, version) in compatible.items():
            if distribution in expected and str(version) != expected[distribution]:
                continue
            current = choices.get(distribution)
            if current is None or version > current[0]:
                choices[distribution] = (version, name, path)
        wheels = []
        for distribution, (_, name, path) in sorted(choices.items()):
            # Match the original offline wheelhouse setup. Avoid replacing a
            # matching torch build when it is already installed in the image.
            if distribution in expected and actual.get(distribution) == expected[distribution]:
                continue
            alias = wheel_dir / name
            alias.symlink_to(path.resolve())
            wheels.append(str(alias))
        if not wheels:
            raise RuntimeError('No compatible organizer wheels for this runtime')
        print(f'Installing {len(wheels)} mounted offline wheels | model calls 0', flush=True)
        command = [sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps',
                   '--force-reinstall', *wheels]
        log = output / 'offline_runtime_install.log'
        with log.open('w') as stream:
            subprocess.run(command, stdout=stream, stderr=subprocess.STDOUT,
                           check=True, timeout=600)
        importlib.invalidate_caches()
        actual = actual_versions()
    if actual != expected:
        raise RuntimeError(f'GPU package versions do not match the archive: {actual}')
    write_json(output / 'GPU_Runtime.json', {'status': 'RUNTIME_MATCHED', 'versions': actual,
                                           'wheel_pins': SELECTED_WHEELS, 'model_calls': 0})
    print('Organizer runtime pins matched | model calls 0', flush=True)

def _stage_pre_server(config, competition, rows, ready, output):
    """Verify the installed harness, quantized checkpoint and public graphs."""
    import torch
    from swegemma import graph as sg
    from swegemma.harness import verification
    from swegemma.harness.agent_runner import run_agent_sandbox

    if (sha256_file(Path(inspect.getsourcefile(run_agent_sandbox))) != RUNNER_SHA256
        or sha256_file(Path(inspect.getsourcefile(verification.verify_task))) != VERIFIER_SHA256):
        raise RuntimeError('Installed organizer harness source differs from the frozen GP02B worker')
    if not torch.cuda.is_available() or torch.cuda.device_count() != 4:
        raise RuntimeError('PyTorch cannot access all four L4 GPUs')
    model = Path(config['model'])
    model_json = read_json(model / 'config.json')
    if not (model_json.get('quantization_config') or
            (model_json.get('text_config') or {}).get('quantization_config')):
        raise RuntimeError('The archived model path must resolve to the QAT checkpoint')
    weight_index = model / 'model.safetensors.index.json'
    weights = sorted(set(read_json(weight_index)['weight_map'].values())) if weight_index.exists() else ['model.safetensors']
    if not weights or any(Path(name).name != name or not (model / name).is_file() for name in weights):
        raise RuntimeError('Missing Gemma checkpoint weight shards')
    reports = {}
    for task in ready:
        row = rows[task]
        graph = sg.get_graph(repo_name=row['repo'], graph_dir=str(Path(competition) / 'graphs'),
                             embeddings_dir=str(Path(competition) / 'embeddings'),
                             base_commit=row['base_commit'])
        reports[task] = {'nodes': graph.number_of_nodes(), 'edges': graph.number_of_edges()}
        if reports[task]['nodes'] < 2 or reports[task]['edges'] < 1:
            raise RuntimeError(f'{task} public graph is missing or empty')
        print(f'[{task}] graph {reports[task]["nodes"]} nodes / {reports[task]["edges"]} edges', flush=True)
    write_json(output / 'Graph_Preflight.json', reports)

def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def hhmmss(seconds):
    minutes, seconds = divmod(int(round(seconds)), 60)
    hours, minutes = divmod(minutes, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

class RunClock:
    def __init__(self):
        self.started = time.monotonic()
        self.started_utc = utc_now()
        self.phases = []

    def elapsed(self):
        return round(time.monotonic() - self.started, 3)

    def log(self, message):
        print(f'[{hhmmss(self.elapsed())} total] {message}', flush=True)

    @contextmanager
    def phase(self, name, *, task=None):
        started = time.monotonic()
        row = {'phase': name, 'task_id': task, 'started_utc': utc_now()}
        stop = threading.Event()
        label = f'{task} | {name}' if task else name
        cap = ''

        def heartbeat():
            while not stop.wait(15):
                self.log(f'{label} | elapsed {hhmmss(time.monotonic() - started)}{cap}')

        self.log(f'{label} started{cap}')
        monitor = threading.Thread(target=heartbeat, daemon=True)
        monitor.start()
        try:
            yield row
        except BaseException:
            row['status'] = 'INTERRUPTED_OR_HELD'
            raise
        finally:
            stop.set()
            monitor.join(timeout=2)
            seconds = round(time.monotonic() - started, 3)
            row.update(elapsed_seconds=seconds, elapsed=hhmmss(seconds), finished_utc=utc_now())
            row.setdefault('status', 'COMPLETE')
            self.phases.append(row)
            self.log(f'{label} | {row["status"]} | elapsed {row["elapsed"]}')

def require_ready_model(base_url, checkpoint):
    import requests

    response = requests.get(
        base_url.rstrip('/') + '/models',
        headers={'Authorization': 'Bearer ' + os.environ['LOCAL_API_KEY']}, timeout=15,
    )
    response.raise_for_status()
    if checkpoint not in {model['id'] for model in response.json()['data']}:
        raise RuntimeError('Local server did not advertise the pinned checkpoint')

def require_free_port(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
        if probe.connect_ex((host, port)) == 0:
            raise RuntimeError(f'Local port {port} is occupied by a server outside this warm-up.')

@contextmanager
def local_server(serving, output, clock, report):
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
    import litellm

    litellm.drop_params = True
    adapters = discover_adapters(str(output / 'frozen_submission'),
                                 adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    if len(getattr(adapters, 'adapters', {}) or {}) != 2:
        raise RuntimeError('Original two LoRA adapters could not be restored')
    cfg = VllmConfig(**serving)
    require_free_port(cfg.host, cfg.port)
    server = VllmServer(cfg, adapter_manifest=adapters)
    try:
        with clock.phase('vLLM load and readiness'):
            boot = time.monotonic()
            server.start()
            report['gpu_server_started'] = True
            require_ready_model(server.base_url, cfg.model)
            report['server_start_seconds'] = round(time.monotonic() - boot, 3)
            config = {'config': dataclasses.asdict(cfg), 'base_url': server.base_url,
                      'log_path': server.log_path, 'command': server.build_cmd(),
                      'lifecycle_owner': 'gp02c_dev_14A_warmup'}
            write_json(output / 'Server_Config.json', config)
        yield config
    finally:
        with clock.phase('vLLM shutdown and log capture') as stage:
            try:
                server.stop()
            except Exception as exc:
                stage['status'] = 'CLEANUP_HOLD'
                report['server_cleanup_hold'] = f'{type(exc).__name__}: {exc}'
                clock.log(report['server_cleanup_hold'])
            try:
                if server.log_path and Path(server.log_path).is_file():
                    shutil.copyfile(server.log_path, output / 'gemma_server.log')
            except OSError as exc:
                report['server_log_hold'] = str(exc)

# Reused from the pinned GP02C-04B source gp02c_worker_supervisor.py.
def supervise_worker(command: tuple[str,...], *, cwd: str | Path,
                     evidence_root: str | Path, overall_seconds: float = 900.0,
                     env: dict[str,str] | None = None) -> dict:
    if os.name != 'posix':
        raise RuntimeError('GP02C pilot supervisor requires POSIX process groups')
    if not command or overall_seconds <= 0:
        raise ValueError('Need worker command and positive timeout')
    evidence=Path(evidence_root).resolve();evidence.mkdir(parents=True,exist_ok=True)
    # Create the trusted child registry BEFORE worker spawn. The worker inherits
    # this exact file path; deletion or corruption afterward is a cleanup HOLD.
    runtime_env=dict(env or os.environ.copy())
    registry=Path(runtime_env.get('GP02C_TRUSTED_PYTEST_REGISTRY',
                 str(evidence/'trusted_pytest_child_groups.jsonl'))).resolve()
    if not registry.is_relative_to(evidence):
        raise ValueError('Trusted pytest registry must be within the per-worker evidence root')
    registry.parent.mkdir(parents=True,exist_ok=True)
    with registry.open('x',encoding='utf-8') as stream:
        stream.flush();os.fsync(stream.fileno())
    runtime_env['GP02C_TRUSTED_PYTEST_REGISTRY']=str(registry)
    started=time.monotonic()
    interrupted = None
    timed_out = False
    with (evidence/'worker_stdout.log').open('wb') as out, (evidence/'worker_stderr.log').open('wb') as err:
        proc=subprocess.Popen(list(command),cwd=str(cwd),env=runtime_env,
                              stdout=out,stderr=err,start_new_session=True)
        try:
            proc.wait(timeout=overall_seconds)
        except subprocess.TimeoutExpired:
            timed_out=True
            _cancel_process_group(proc)
        except BaseException as exc:
            interrupted = exc
            _cancel_process_group(proc)
    from gp03_public_runtime import reap_trusted_groups
    cleanup=reap_trusted_groups(registry)
    cleanup_ok=bool(cleanup['verified'])
    row={'schema':'gp02c.worker-supervisor.v1.10','elapsed_seconds':time.monotonic()-started,
         'trusted_child_cleanup':cleanup,
         'trusted_process_cleanup_verified':cleanup_ok,
         'timed_out':timed_out,'returncode':proc.returncode,
         'interrupted':type(interrupted).__name__ if interrupted is not None else None,
         'verdict':('TRUSTED_CHILD_CLEANUP_HOLD' if not cleanup_ok
                    else 'WORKER_INTERRUPTED' if interrupted is not None
                    else 'UNVERIFIED_BUDGET' if timed_out else 'WORKER_EXITED_NOT_SCORED'),
         'worker_pid':proc.pid,'worker_command':list(command)}
    (evidence/'supervisor.json').write_text(json.dumps(row,sort_keys=True,indent=2)+'\n')
    if interrupted is not None:
        raise interrupted
    return row


## 5. gp03_frozen_agent_parts.py

In [ ]:
%%gp03_source gp03_frozen_agent_parts.py
import hashlib
import json
import logging
import os
from pathlib import Path
import time

# Reused from the pinned GP02C-04B source gp02c_dev_frozen_worker.py.
def safe_task_fields(row):
    """Explicit allowlist: no reference fix, protected tests, hints or expected nodes."""
    keys = ('instance_id', 'repo', 'base_commit', 'problem_statement', 'created_at')
    return {key: row.get(key, '') for key in keys}

def connect_starter_server(config):
    """Build clients for the running starter server; never start/stop a server."""
    from adk_submission import VllmConfig, VllmServer, discover_adapters
    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
    import requests
    shared = config['starter_server']
    cfg = VllmConfig(**shared['config'])
    if cfg.host != '127.0.0.1' or cfg.model != config['model_path']:
        raise RuntimeError('The starter server must use the recorded local checkpoint and localhost.')
    adapters = discover_adapters(config['submission_dir'], adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
    client = VllmServer(cfg, adapter_manifest=adapters)
    if client.base_url != shared['base_url']:
        raise RuntimeError('Recorded starter endpoint differs from its configuration.')
    api_key = os.environ.get('LOCAL_API_KEY', 'EMPTY')
    response = requests.get(client.base_url + '/models',
                            headers={'Authorization': 'Bearer ' + api_key}, timeout=15)
    response.raise_for_status()
    model_ids = [row['id'] for row in response.json().get('data', [])]
    if cfg.model not in model_ids:
        raise RuntimeError('The running starter server does not advertise the selected checkpoint.')
    registry = client.create_model_registry(aliases=[config['model_name']], model_prefix='openai/', api_key=api_key)
    return client, adapters, registry

def cap_generation_constraints(constraints, output_tokens):
    # Preserve official bounds and thinking_budget; alter only default maximum
    # output length. Explicit ADK sampling YAML is capped separately.
    from adk_submission import GenerationConstraints
    defaults = dict(constraints.defaults or {})
    defaults['max_output_tokens'] = int(output_tokens)
    return GenerationConstraints(
        allowed_fields=constraints.allowed_fields,
        max_output_tokens=constraints.max_output_tokens,
        thinking_budget=constraints.thinking_budget,
        defaults=defaults,
    )

def make_eval_config(config, EvalConfig, registry, adapters, snapshot):
    from google.adk.agents.context_cache_config import ContextCacheConfig
    from google.adk.apps._configs import EventsCompactionConfig
    from swegemma.config import build_submission_limits
    from swegemma.budget import HarnessLimits
    limits, generation_constraints = build_submission_limits()
    generation_constraints = cap_generation_constraints(generation_constraints, config['max_output_tokens'])
    tool_harness = HarnessLimits(
        command_timeout_seconds=config['command_timeout_seconds'],
        verification_timeout_seconds=config['verification_timeout_seconds'],
        max_file_lines=config['max_file_lines'],
        max_file_chars=config['max_file_chars'],
        max_stdout_chars=config['max_stdout_chars'],
    )
    return EvalConfig(tasks_path=Path(config['public_task_path']), snapshots_dir=snapshot.parent,
            results_dir=Path(config['artifact_dir']) / 'agent', submission_dir=Path(config['submission_dir']),
            models=registry, sandbox='subprocess', wheels_dir=Path(config['competition_root']) / 'wheels',
            graph_dir=str(Path(config['competition_root']) / 'graphs'),
            embeddings_dir=str(Path(config['competition_root']) / 'embeddings'),
            timeout_seconds=config['command_timeout_seconds'],
            verification_timeout_seconds=config['verification_timeout_seconds'],
            max_tool_calls=config['max_tool_calls'], max_turns=config['max_llm_calls'],
            max_time_minutes=config['agent_time_minutes'], harness=tool_harness, limits=limits,
            generation_constraints=generation_constraints, adapter_manifest=adapters,
            context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
            events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2,
                    token_threshold=14336, event_retention_size=5),
            display_mode='quiet', enable_sandbox_testing=True)

def install_model_observer(emit, artifacts, config):
    """Passive instrumentation of the ADK LiteLlm async-generator boundary.

    The original request is forwarded unchanged; each original response is
    yielded unchanged. Record exact client-method wall time; do not claim that
    it equals device inference time or that provider retries are independent.
    """
    from google.adk.models.lite_llm import LiteLlm
    original = LiteLlm.generate_content_async
    folder = Path(artifacts) / 'agent/model_transport'
    folder.mkdir(parents=True, exist_ok=True)
    summary_path = folder / 'calls.jsonl'
    sequence = 0

    def dump_model(value):
        # Multiple serialization paths: one unsupported SDK field should not
        # discard the entire model input. Record explicit capture errors.
        errors = []
        if hasattr(value, 'model_dump_json'):
            try:
                return value.model_dump_json(exclude_none=True)
            except Exception as exc:
                errors.append('model_dump_json:' + type(exc).__name__)
        if hasattr(value, 'model_dump'):
            try:
                return json.dumps(value.model_dump(exclude_none=True),
                                  ensure_ascii=False, default=str)
            except Exception as exc:
                errors.append('model_dump:' + type(exc).__name__)
        try:
            return json.dumps(value, ensure_ascii=False, default=str)
        except Exception as exc:
            errors.append('json:' + type(exc).__name__)
        return json.dumps({'capture_error':'; '.join(errors),
                           'class':type(value).__name__,
                           'fallback_repr':repr(value)[:200000]},ensure_ascii=False)

    def log_summary(row):
        with summary_path.open('a', encoding='utf-8') as handle:
            handle.write(json.dumps(row, ensure_ascii=False, default=str) + '\n')

    async def timed_generate(self, llm_request, *args, **kwargs):
        nonlocal sequence
        sequence += 1
        seq = sequence
        start = time.monotonic()
        utc = __import__('datetime').datetime.now(__import__('datetime').timezone.utc).isoformat()
        path = folder / f'request_{seq:04d}.json'
        payload = dump_model(llm_request)
        # Protect the download archive from unexpected multi-megabyte contexts.
        max_bytes = 3_000_000
        request_utf8 = payload.encode('utf-8')
        truncated = len(request_utf8) > max_bytes
        if truncated:
            path.write_text(json.dumps({'truncated':True, 'source_bytes':len(request_utf8),
                'sha256':hashlib.sha256(request_utf8).hexdigest(),
                'first_chars':payload[:200000]}, ensure_ascii=False),encoding='utf-8')
        else:
            path.write_text(payload,encoding='utf-8')
        # Fail closed if explicit request configuration contradicts cap.
        request_config = getattr(llm_request, 'config', None)
        actual_cap = getattr(request_config, 'max_output_tokens', None)
        if actual_cap is not None and int(actual_cap) != int(config['max_output_tokens']):
            message = f'ADK request max_output_tokens={actual_cap}, expected {config["max_output_tokens"]}.'
            emit('agent_error', '', {'message':message, 'cause':'generation_contract_violation'})
            raise RuntimeError(message)
        emit('model_request', '', {'request_number':seq, 'at_utc':utc,
              'effective_max_output_tokens':actual_cap,
              'expected_max_output_tokens':config['max_output_tokens'],
              'input_capture_bytes':len(request_utf8),
              'input_record':str(path.relative_to(Path(artifacts))),
              'input_record_truncated':truncated})
        count = 0
        response_path = folder / f'responses_{seq:04d}.jsonl'
        failure = None
        try:
            async for response in original(self, llm_request, *args, **kwargs):
                count += 1
                row = dump_model(response)
                with response_path.open('a', encoding='utf-8') as out:
                    out.write(row + '\n')
                yield response
        except Exception as exc:
            failure = type(exc).__name__ + ': ' + str(exc)
            emit('model_failure', '', {'request_number':seq, 'error':failure[:1500]})
            raise
        finally:
            duration = round(time.monotonic() - start,3)
            result = {'request_number':seq, 'utc_start':utc,
                      'duration_seconds':duration, 'response_chunks':count,
                      'request_path':path.relative_to(Path(artifacts)).as_posix(),
                      'response_path':response_path.relative_to(Path(artifacts)).as_posix(),
                      'input_bytes':len(request_utf8), 'input_capture_truncated':truncated,
                      'effective_max_output_tokens':actual_cap,
                      'error':failure, 'timing_semantics':'ADK client model call duration including transport/retries; not GPU decode-only time'}
            try:
                log_summary(result)
                emit('model_complete', '', result)
            except Exception:
                logging.warning('Model-transport observer could not write summary; preserving model behavior.')

    LiteLlm.generate_content_async = timed_generate
    return lambda: setattr(LiteLlm, 'generate_content_async', original)


## 6. gp03_frozen_helpers.py

In [ ]:
%%gp03_source gp03_frozen_helpers.py
import hashlib
import importlib.metadata
import inspect
import json
import os
from pathlib import Path
import shlex

# Reused from the pinned GP02C-04B source gp02c_dev_frozen_helpers.py.
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n')
    temporary.replace(path)

def prepare_snapshot_workspace(original_exec, manager, sandbox_id, task, artifact_dir):
    """Use the supplied archive unchanged and put its source ahead of installed packages."""
    paths = manager.sandboxes[sandbox_id]
    workspace = Path(paths['workspace']).resolve()
    package = task.repo.rsplit('/', 1)[-1].replace('-', '_')
    diagnostic = {'declared_base_commit': task.base_commit, 'workspace': str(workspace),
                  'package': package, 'status': 'PREPARING'}
    diagnostic_path = Path(artifact_dir) / 'workspace_setup.json'

    def execute(command):
        return original_exec(sandbox_id, command, timeout=60)

    def fail(message):
        diagnostic['status'] = 'WORKSPACE_HOLD'
        diagnostic['error'] = message
        write_json(diagnostic_path, diagnostic)
        raise RuntimeError(message)

    before = execute('cd /workspace && git rev-parse HEAD')
    diagnostic['snapshot_head_before'] = (before.stdout or '').strip()
    diagnostic['root_entries_before'] = sorted(p.name for p in workspace.iterdir())[:100]
    if before.exit_code != 0 or not diagnostic['snapshot_head_before']:
        fail('The published snapshot has no readable Git HEAD; inspect its setup diagnostics.')
    commit = shlex.quote(task.base_commit + '^{commit}')
    exists = execute('cd /workspace && git cat-file -e ' + commit)
    diagnostic['declared_commit_exists'] = exists.exit_code == 0
    diagnostic['declared_commit_id_matches_head'] = diagnostic['snapshot_head_before'] == task.base_commit
    tree = execute('cd /workspace && git rev-parse HEAD^{tree}')
    diagnostic['snapshot_git_tree'] = (tree.stdout or '').strip()
    if tree.exit_code != 0 or not diagnostic['snapshot_git_tree']:
        fail('The published snapshot has no readable Git tree.')
    diagnostic['identity_policy'] = 'task_named_published_archive_sha256_and_recorded_native_git_identity'
    diagnostic['original_commit_id_required'] = False
    # The official verifier consumes the packaged snapshot directly. Its rebuilt
    # Git history need not retain the original repository's commit identifiers.
    # Do not fetch another repository or rewrite the supplied working tree.

    candidates = []
    for root in [workspace, workspace / 'src']:
        if (root / package / '__init__.py').is_file() or (root / (package + '.py')).is_file():
            candidates.append(root)
    diagnostic['source_roots_found'] = [str(p) for p in candidates]
    if len(candidates) != 1:
        fail('Expected one explicit root/src source location for the repository package; inspect workspace_setup.json.')
    source_root = candidates[0]
    environment = Path(paths['venv']).resolve()
    site_dirs = {p.resolve() for p in environment.glob('lib/python*/site-packages') if p.is_dir()}
    if len(site_dirs) != 1:
        fail('Could not identify the sandbox virtual environment site-packages directory.')
    site_dir = next(iter(site_dirs))
    if not site_dir.is_relative_to(environment):
        fail('Sandbox source-path setup would escape its own virtual environment.')
    overlay = site_dir / 'zz_gp01b_snapshot_source.pth'
    overlay.write_text('import sys; sys.path.insert(0, ' + repr(str(source_root)) + ')\n', encoding='utf-8')
    diagnostic['source_root'] = str(source_root)
    diagnostic['source_path_file'] = str(overlay)
    diagnostic['status'] = 'WORKSPACE_READY'
    write_json(diagnostic_path, diagnostic)
    return diagnostic

def verify_snapshot_archive(path, expected_sha256):
    actual = digest(path)
    if not expected_sha256 or actual != expected_sha256:
        raise RuntimeError('Published snapshot bytes differ from the pinned task archive; review the dataset release.')
    return actual

def check_import_origin(original_exec, sandbox_id, package, workspace):
    script = '''import importlib, importlib.util, json, sys
from pathlib import Path
name = sys.argv[1]
spec = importlib.util.find_spec(name)
out = {"package": name, "spec_origin": str(spec.origin) if spec else None}
try:
    module = importlib.import_module(name)
    out["imported_file"] = str(getattr(module, "__file__", None))
    out["version"] = str(getattr(module, "__version__", "unknown"))
except Exception as exc:
    out["import_error"] = type(exc).__name__ + ": " + str(exc)
print(json.dumps(out))
'''
    command = 'python3 -s -c ' + shlex.quote(script) + ' ' + shlex.quote(package)
    result = original_exec(sandbox_id, command, timeout=60)
    try:
        data = json.loads((result.stdout or '').splitlines()[-1])
    except (ValueError, IndexError):
        data = {'import_error': 'Origin probe did not return JSON.', 'stderr': result.stderr or ''}
    actual = data.get('imported_file')
    data['inside_snapshot'] = bool(actual and Path(actual).resolve().is_relative_to(Path(workspace).resolve()))
    data['probe_exit_code'] = result.exit_code
    return data

def load_official(config):
    # These switches prevent optional model metadata downloads during imports.
    os.environ.setdefault('LITELLM_LOCAL_MODEL_COST_MAP', 'True')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
    from adk_submission import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.models import load_tasks
    from swegemma.sandbox import SubprocessManager
    from swegemma.harness import verification
    source_path = Path(inspect.getsourcefile(verification.verify_task))
    actual_hash = digest(source_path)
    if actual_hash != config['verification_source_sha256']:
        raise RuntimeError('Installed verification.py differs from the GP01A-inspected source.')
    parameters = list(inspect.signature(verification.verify_task).parameters)
    if parameters != config['verify_task_parameters']:
        raise RuntimeError('Verifier signature differs from the inspected contract.')
    from packaging.version import Version
    adk_version = importlib.metadata.version('google-adk')
    if not Version('1.34.0') <= Version(adk_version) < Version('2'):
        raise RuntimeError('The isolated runtime must use google-adk>=1.34,<2.')
    return ModelRegistry, EvalConfig, load_tasks, SubprocessManager, verification


## 7. tool_gate.py

In [ ]:
%%gp03_source tool_gate.py
"""GP02B v2.7: observational tool audit + protected test paths, no forced investigation tools."""
from __future__ import annotations
from collections import Counter
from pathlib import Path
import functools
import json
import re
import shlex
import time

PROTECTED_PATH = re.compile(r'(^|/)(tests?/|test_[^/]*\.py$|[^/]*_test\.py$)')
READ_ONLY_COMMANDS = {'grep', 'rg', 'find', 'ls', 'pwd', 'cat', 'head', 'tail', 'wc'}


def safe_discovery_command(command):
    """Narrow pre-graph shell allowlist; no shell expansion/redirect/pipe."""
    if not isinstance(command, str) or not command.strip():
        return False
    if re.search(r'[;&|><`$\\\n]', command):
        return False
    try:
        tokens = shlex.split(command)
    except ValueError:
        return False
    if not tokens:
        return False
    executable = tokens[0].split('/')[-1]
    return executable in READ_ONLY_COMMANDS or (tokens[:2] == ['git', 'grep'])


class InvestigationGate:
    """Log tool execution and protect tests without requiring any specific tool calls."""
    def __init__(self, audit_path):
        self.path = Path(audit_path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.sources = set()
        self.graph_nodes = set()
        self.calls = Counter()
        self.blocked = Counter()
        self.seq = 0
        self.started = time.monotonic()

    @property
    def ready(self):
        return bool(self.sources) and bool(self.graph_nodes)

    def snapshot(self):
        return {'gate_passed': None, 'gate_enforced': False, 'investigation_evidence_observed': self.ready, 'read_source_files': sorted(self.sources),
                'successful_graph_nodes': sorted(self.graph_nodes),
                'tool_call_counts': dict(self.calls), 'blocked_counts': dict(self.blocked)}

    def log(self, name, args, response, duration, blocked=False):
        self.seq += 1
        payload = {'seq': self.seq, 'elapsed_seconds': round(time.monotonic() - self.started, 4),
                   'tool': name, 'arguments': args,
                   'result_status': response.get('status') if isinstance(response, dict) else str(type(response)),
                   'count': response.get('count') if isinstance(response, dict) else None,
                   'error_type': response.get('error_type') if isinstance(response, dict) else None,
                   'duration_seconds': round(duration, 6), 'blocked': blocked,
                   'gate': self.snapshot()}
        with self.path.open('a', encoding='utf-8') as file:
            file.write(json.dumps(payload, ensure_ascii=False, default=str) + '\n')

    @staticmethod
    def _blocked(message, missing, error_type='InvestigationGateBlocked'):
        return {'status': 'error', 'error_type': error_type,
                'error_message': message, 'missing_requirements': missing,
                'instruction': 'Select relevant source files and code-graph symbols yourself. '
                               'Choose a non-protected path; repository tests must not be modified.'}

    def wrap(self, tools):
        required = {'read_file', 'get_code_neighbors', 'edit_file', 'write_file', 'run_command', 'submit_patch'}
        if not required.issubset(set(tools)):
            raise RuntimeError('Registered agent tools missing: ' + str(sorted(required - set(tools))))
        wrapped = {}
        for name, fn in tools.items():
            @functools.wraps(fn)
            def guarded(*args, _name=name, _fn=fn, **kwargs):
                # Keep the original callable signature visible to the ADK schema builder.
                self.calls[_name] += 1
                inputs = dict(kwargs)
                if args:
                    inputs['_positional'] = repr(args)[:500]
                before = time.monotonic()
                denied = None
                if _name in ('edit_file', 'write_file'):
                    path = str(kwargs.get('filepath', '')).replace('\\', '/')
                    if PROTECTED_PATH.search(path):
                        denied = self._blocked('Test files cannot be changed by the repair agent.', [],
                                               'ProtectedPathDenied')
                elif _name == 'run_command':
                    command = str(kwargs.get('command', ''))
                    # Detect obvious command-based test-file mutation attempts; not a complete OS sandbox.
                    if re.search(r'(?:>|>>|\bsed\s+-i\b|\btee\b|\btouch\b)', command) and re.search(r'(?:tests?/|test_[^\s/]*\.py)', command):
                        denied = self._blocked('Command appears to modify test files.', [], 'ProtectedPathDenied')
                if denied is not None:
                    self.blocked[_name] += 1
                    self.log(_name, inputs, denied, time.monotonic() - before, blocked=True)
                    return denied
                try:
                    result = _fn(*args, **kwargs)
                    if isinstance(result, dict):
                        payload = result
                    elif isinstance(result, str):
                        try:
                            payload = json.loads(result)
                        except json.JSONDecodeError:
                            payload = {'status': 'unparsed'}
                    else:
                        payload = {'status': 'unparsed'}
                    if payload.get('status') == 'ok':
                        if _name == 'read_file' and isinstance(payload.get('content'), str) and payload['content'].strip():
                            self.sources.add(str(kwargs.get('filepath') or payload.get('filepath') or 'unknown'))
                        if _name == 'get_code_neighbors' and isinstance(payload.get('neighbors'), list) and len(payload['neighbors']) > 0:
                            self.graph_nodes.add(str(kwargs.get('node') or payload.get('node') or 'unknown'))
                    self.log(_name, inputs, payload, time.monotonic() - before)
                    return result
                except Exception as exc:
                    self.log(_name, inputs,
                             {'status': 'error', 'error_type': type(exc).__name__},
                             time.monotonic() - before)
                    raise
            # functools.wraps preserves the original argument schema for ADK.
            wrapped[name] = guarded
        return wrapped


## 8. gp03_seed_proxy.py

In [ ]:
%%gp03_source gp03_seed_proxy.py
"""Loopback-only model endpoint for official worker acceptance."""

from contextlib import contextmanager
import hashlib
import http.client
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
import json
import os
from pathlib import Path
import threading
import time
from urllib.parse import urlsplit


class SeedProxy(ThreadingHTTPServer):
    daemon_threads = True

    def __init__(self, *, model_ids, event_path, upstream_url=None, timeout=180, max_requests=1):
        self.model_ids = frozenset(model_ids)
        self.event_path = Path(event_path)
        self.deadline = time.monotonic() + timeout
        self.lock = threading.Lock()
        if type(max_requests) is not int or not 1 <= max_requests <= 60:
            raise ValueError("The per-arm transport cap must be between 1 and 60")
        self.max_requests = max_requests
        self.forwarded = 0
        self.denied = 0
        self.completions = []
        self.upstream = urlsplit(upstream_url) if upstream_url else None
        if self.upstream and (self.upstream.scheme != "http" or self.upstream.hostname != "127.0.0.1"
                              or not self.upstream.port or self.upstream.path.rstrip("/") != "/v1"
                              or self.upstream.username or self.upstream.query or self.upstream.fragment):
            raise ValueError("The seed probe requires the pinned loopback vLLM endpoint")
        super().__init__(("127.0.0.1", 0), ProbeHandler)

    def event(self, row):
        with self.lock:
            with self.event_path.open("a", encoding="utf-8") as stream:
                stream.write(json.dumps(row, sort_keys=True) + "\n")

    @property
    def base_url(self):
        return f"http://127.0.0.1:{self.server_port}/v1"


class ProbeHandler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        return

    def reply_json(self, status, value):
        body = json.dumps(value).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path != "/v1/models":
            self.reply_json(404, {"error": "Only the local model registry is available"})
            return
        self.reply_json(200, {"object": "list", "data": [
            {"id": model, "object": "model", "owned_by": "local"} for model in sorted(self.server.model_ids)
        ]})

    def deny(self, status, reason):
        with self.server.lock:
            self.server.denied += 1
        self.server.event({"event": "POST_DENIED", "reason": reason})
        self.reply_json(status, {"error": reason})

    def do_POST(self):
        server = self.server
        if self.path != "/v1/chat/completions" or server.upstream is None:
            self.deny(403, "Inference is disabled for this official sandbox probe")
            return
        try:
            length = int(self.headers.get("Content-Length", "0"))
            if not 0 < length <= 12 * 1024 * 1024:
                raise ValueError("Missing or invalid body length")
            self.connection.settimeout(max(1.0, min(15.0, server.deadline - time.monotonic())))
            data = json.loads(self.rfile.read(length))
            if not isinstance(data, dict) or data.get("model") not in server.model_ids:
                raise ValueError("Request model is outside the pinned vLLM registry")
            if not data.get("messages") or not isinstance(data["messages"], list):
                raise ValueError("Expected an ADK chat request")
            token_caps = [data[name] for name in ("max_completion_tokens", "max_tokens") if data.get(name) is not None]
            if any(type(value) is not int or value != 2048 for value in token_caps):
                raise ValueError(f"Expected the effective 2048-token output cap, found {token_caps!r}")
            output_cap_source = "CLIENT"
            if not token_caps:
                # ADK history compaction can omit both OpenAI token-limit fields.
                data.pop("max_completion_tokens", None)
                data["max_tokens"] = 2048
                output_cap_source = "PROXY_DEFAULT"
        except (OSError, ValueError) as exc:
            self.deny(400, str(exc))
            return
        with server.lock:
            available = server.forwarded < server.max_requests and server.deadline > time.monotonic()
            if available:
                server.forwarded += 1
                request_number = server.forwarded
        if not available:
            self.deny(429, "The seeded transport call budget or ordinary deadline is exhausted")
            return

        client_seed = data.get("seed")
        data["seed"] = 42
        outgoing = json.dumps(data, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
        server.event({"event": "VLLM_REQUEST", "request_number": request_number, "model": data["model"], "client_seed": client_seed,
                      "forwarded_seed": json.loads(outgoing)["seed"], "max_output_tokens": 2048, "output_cap_source": output_cap_source,
                      "stream": bool(data.get("stream")), "request_bytes": len(outgoing),
                      "request_sha256": hashlib.sha256(outgoing).hexdigest()})
        connection = http.client.HTTPConnection("127.0.0.1", server.upstream.port,
                                                timeout=max(0.1, server.deadline - time.monotonic()))
        started = time.monotonic()
        status = None
        count = 0
        complete = False
        try:
            connection.request("POST", "/v1/chat/completions", body=outgoing, headers={
                "Content-Type": "application/json", "Authorization": "Bearer " + os.environ.get("LOCAL_API_KEY", "EMPTY"),
            })
            response = connection.getresponse()
            status = response.status
            self.send_response(status)
            self.send_header("Content-Type", response.getheader("Content-Type", "application/json"))
            self.send_header("Connection", "close")
            self.end_headers()
            while time.monotonic() < server.deadline:
                if connection.sock is not None:
                    connection.sock.settimeout(max(0.1, server.deadline - time.monotonic()))
                block = response.read1(64 * 1024)
                if not block:
                    complete = True
                    break
                self.wfile.write(block)
                self.wfile.flush()
                count += len(block)
        except (OSError, http.client.HTTPException) as exc:
            server.event({"event": "UPSTREAM_ERROR", "type": type(exc).__name__, "detail": str(exc)[:300]})
            if status is None:
                try:
                    self.reply_json(502, {"error": "Local vLLM request failed"})
                except OSError:
                    pass
        finally:
            connection.close()
            result = {"event": "VLLM_RESPONSE", "request_number": request_number, "status": status, "response_bytes": count,
                      "completed": complete, "seconds": round(time.monotonic() - started, 3)}
            with server.lock:
                server.completions.append(result)
            server.event(result)


@contextmanager
def seed_proxy(**kwargs):
    with SeedProxy(**kwargs) as server:
        thread = threading.Thread(target=server.serve_forever, kwargs={"poll_interval": 0.1}, daemon=True)
        thread.start()
        try:
            yield server
        finally:
            server.shutdown()
            thread.join(timeout=3)


## 9. gp03_official_hook.py

In [ ]:
%%gp03_source gp03_official_hook.py
"""Attach the accepted B/C wiring inside the official agent worker."""

import functools
import hashlib
import inspect
import json
import os
from pathlib import Path
import time

from gp03_public_runtime import (
    WorkspaceFingerprinter, attach_gp03_pair, canonical_hash, environment_fingerprint,
    environment_profile, prepare_pytest_config, run_git, sha256_file, write_json,
)

_ATTACHMENT = None
SANDBOX_STOP = "GP03_OFFICIAL_SANDBOX_PROBE_COMPLETE"
SEED_STOP = "GP03_DEVELOPMENT_SEED_PROBE_COMPLETE"


class ProbeStop(RuntimeError):
    pass


def prepare_official_workspace(manager, sandbox_id):
    """Apply the same public pytest configuration before the official baseline commit."""
    root = Path(manager.sandboxes[sandbox_id]["workspace"]).resolve(strict=True)
    prepare_pytest_config(root)


def checked_manifest(config):
    path = Path(config["gp03_manifest"])
    if sha256_file(path) != config["gp03_manifest_sha256"]:
        raise RuntimeError("Official worker manifest checksum mismatch")
    manifest = json.loads(path.read_text())
    if (manifest["task_id"] != config["task_id"] or manifest["repo"] != config["task_expected_repo"]
            or manifest["base_commit"] != config["task_expected_commit"]
            or manifest["snapshot_sha256"] != config["snapshot_pins"][config["task_id"]]):
        raise RuntimeError("Official worker public task and manifest disagree")
    if config["gp03_mode"] == "seed_probe" and manifest["task_id"] != "rich_2943":
        raise RuntimeError("The single live request is restricted to the development case rich_2943")
    return manifest


def _git_identity(root, *argv):
    return run_git(root, list(argv)).stdout.decode().strip()


def _agent_view_head(tools, head):
    response = tools["run_command"](command="cd /workspace && git rev-parse HEAD")
    if isinstance(response, str):
        response = json.loads(response)
    if not isinstance(response, dict):
        raise RuntimeError("Official run_command returned an unknown response schema")
    result = response.get("result")
    values = [response.get("stdout"), response.get("output")]
    if isinstance(result, dict):
        values.extend((result.get("stdout"), result.get("output")))
    if head not in [value.strip() for value in values if isinstance(value, str)]:
        raise RuntimeError("Host and agent views of the official sandbox HEAD differ")
    return head


def install_official_hook(*, context, manager, config, artifacts):
    global _ATTACHMENT
    if _ATTACHMENT is not None or config["gp03_mode"] not in {"sandbox_probe", "seed_probe", "evaluation"}:
        raise RuntimeError("Expected one official worker acceptance mode")
    if (config["max_tool_calls"], config["max_llm_calls"], config["agent_time_minutes"],
            config["max_output_tokens"], config["model_context_tokens"]) != (60, 60, 15, 2048, 32768):
        raise RuntimeError("Frozen agent budgets or model settings differ")
    manifest = checked_manifest(config)
    artifacts = Path(artifacts)
    original = context.create_tools
    _ATTACHMENT = {"artifacts": artifacts, "mode": config["gp03_mode"], "ready": False, "model_started": False}

    @functools.wraps(original)
    def bind_after_sandbox_setup():
        if _ATTACHMENT["ready"]:
            return _ATTACHMENT["create_tools"]()
        sandbox_id = context.container_id
        if not sandbox_id:
            raise RuntimeError("Official context has no sandbox identity")
        root = Path(manager.sandboxes[sandbox_id]["workspace"]).resolve(strict=True)
        head = _git_identity(root, "rev-parse", "HEAD^{commit}")
        baseline = _git_identity(root, "rev-parse", "_swegemma_baseline^{commit}")
        if head != baseline:
            raise RuntimeError("Official agent workspace HEAD differs from its recorded baseline tag")
        native = manifest["native_head"]
        if (_git_identity(root, "rev-parse", native + "^{commit}") != native
                or _git_identity(root, "rev-parse", native + "^{tree}") != manifest["native_tree"]):
            raise RuntimeError("Official sandbox native snapshot identity differs from the CPU receipt")
        run_git(root, ["merge-base", "--is-ancestor", native, head])
        tools = original()
        agent_head = _agent_view_head(tools, head)
        source_sha = WorkspaceFingerprinter(root).snapshot().sha256
        actual_env = environment_fingerprint(environment_profile())
        write_json(artifacts / "Official_Sandbox_Identity.json", {
            "task_id": manifest["task_id"], "sandbox_id": sandbox_id, "host_head": head, "agent_head": agent_head,
            "native_head": native, "native_tree": manifest["native_tree"], "baseline_tag": baseline,
            "source_sha256": source_sha, "expected_source_sha256": manifest["baseline_source_sha256"],
            "environment_sha256": actual_env, "expected_environment_sha256": manifest["environment_sha256"],
            "tool_names": sorted(tools),
        })
        context.create_tools = original
        boundary, restore = attach_gp03_pair(
            context=context, root=root, results_root=artifacts / "trusted_checks", manifest=manifest,
            arm=config["gp03_arm"], seed=42, run_id=config["run_id"], environment_sha=actual_env,
            started_monotonic=float(os.environ["GP03_WORKER_STARTED_MONOTONIC"]), baseline_ref=baseline,
        )
        wrapped = context.create_tools()
        if (set(wrapped) != set(tools)
                or any(inspect.signature(wrapped[name]) != inspect.signature(tools[name]) for name in tools)):
            raise RuntimeError("The B/C wrapper changed the official tool registry signatures")
        if config["gp03_mode"] == "evaluation":
            from gp03_eval_capture import attach_evaluation_capture
            _ATTACHMENT.update(boundary=boundary, restore=restore)
            return attach_evaluation_capture(_ATTACHMENT, context, manager, sandbox_id, config, root, baseline, manifest, tools)
        controller = boundary.controller
        gate = controller.final_gate()
        ledger = controller.ledger
        receipt = ledger.latest(manifest["check"]["check_id"])
        write_json(artifacts / "Official_Gate_Ledger.json", ledger.export())
        write_json(artifacts / "Official_Tool_Boundary.json", boundary.export_audit())
        counts = [getattr(receipt, "tests_" + field) for field in ("collected", "passed", "failed", "errored", "skipped")] if receipt else []
        if (not gate.accepted or receipt is None or receipt.provenance != "final_gate"
                or counts != config["gp03_expected_counts"] or controller.budget.used_test_runs != 1
                or controller.budget.used_tool_calls != 1 or controller.budget.used_model_calls != 0
                or gate.accepted_patch_sha256 != hashlib.sha256(b"").hexdigest()):
            raise RuntimeError("Official baseline did not reproduce the public counts through the shared final gate")
        _ATTACHMENT.update(ready=True, create_tools=context.create_tools, restore=restore, boundary=boundary)
        write_json(artifacts / "Official_Worker_Attestation.json", {
            "status": "OFFICIAL_PUBLIC_GATE_ACCEPTED", "task_id": manifest["task_id"], "arm": config["gp03_arm"],
            "mode": config["gp03_mode"], "check_manifest_sha256": config["gp03_manifest_sha256"],
            "visible_manifest_sha256": ledger.manifest.sha256, "public_counts": counts,
            "tool_signatures_sha256": canonical_hash({name: str(inspect.signature(fn)) for name, fn in sorted(tools.items())}),
            "environment_sha256": actual_env, "gate": gate.status, "protected_verifier_calls": 0,
        })
        if config["gp03_mode"] == "sandbox_probe":
            raise ProbeStop(SANDBOX_STOP)
        return wrapped

    context.create_tools = bind_after_sandbox_setup


def install_probe_model_guard():
    """Permit one first-turn response on the development case, then stop before tool use."""
    from google.adk.models.lite_llm import LiteLlm

    original = LiteLlm.generate_content_async
    if not inspect.isasyncgenfunction(original):
        raise RuntimeError("Pinned model observer is expected to be an async generator")

    @functools.wraps(original)
    async def guarded(*args, **kwargs):
        state = _ATTACHMENT
        if state is None or not state["ready"] or state["mode"] != "seed_probe" or state["model_started"]:
            raise ProbeStop("GP03_MODEL_REQUEST_DENIED")
        state["model_started"] = True
        boundary = state["boundary"]
        if boundary.controller.deadline.remaining("ordinary") <= 0:
            raise ProbeStop("GP03_ORDINARY_DEADLINE")
        boundary.sync_model_calls(1)
        responses = 0
        async for _ in original(*args, **kwargs):
            responses += 1
        if not responses:
            raise RuntimeError("Official development request produced no response events")
        write_json(state["artifacts"] / "Official_Model_Probe.json", {
            "status": "OFFICIAL_FIRST_RESPONSE_RECEIVED", "response_events": responses,
            "model_calls": 1, "response_delivered_to_agent": False, "task_id": "rich_2943",
            "elapsed_monotonic": time.monotonic(),
        })
        raise ProbeStop(SEED_STOP)
        yield  # Preserve the async-generator protocol without passing a model event to ADK.

    LiteLlm.generate_content_async = guarded
    return lambda: setattr(LiteLlm, "generate_content_async", original)


## 10. gp03_official_worker.py

In [ ]:
%%gp03_source gp03_official_worker.py
"""Bounded GP03 acceptance entry to the pinned organizer agent runner."""

import argparse
import asyncio
import hashlib
import inspect
import json
import os
from pathlib import Path
import time

from gp03_eval_capture import command_deadline, install_evaluation_model_guard, record_runner_result
from gp03_frozen_agent_parts import connect_starter_server, install_model_observer, make_eval_config, safe_task_fields
from gp03_frozen_helpers import check_import_origin, load_official, prepare_snapshot_workspace, verify_snapshot_archive
from gp03_official_hook import ProbeStop, install_official_hook, install_probe_model_guard, prepare_official_workspace
from gp03_public_runtime import sha256_file, write_json
from tool_gate import InvestigationGate


async def probe(config):
    _, EvalConfig, _, SubprocessManager, _ = load_official(config)
    from swegemma.context import SwegemmaContext
    from swegemma.harness import agent_runner
    from swegemma.models import Task
    import yaml

    actual_runner = Path(inspect.getsourcefile(agent_runner.run_agent_sandbox))
    if sha256_file(actual_runner) != config["agent_runner_source_sha256"]:
        raise RuntimeError("Official agent runner source differs from its pin")
    public = json.loads(Path(config["public_task_path"]).read_text())
    if public != safe_task_fields(public) or public["instance_id"] != config["task_id"]:
        raise RuntimeError("The official acceptance worker requires only the allowlisted public fields")
    task = Task(**public)
    if task.patch or task.test_patch or task.hints_text or task.FAIL_TO_PASS or task.PASS_TO_PASS:
        raise RuntimeError("Protected fields reached the public agent worker")
    snapshot = Path(config["competition_root"]) / "snapshots" / (config["task_id"] + ".tgz")
    verify_snapshot_archive(snapshot, config["snapshot_pins"][config["task_id"]])
    artifacts = Path(config["artifact_dir"])
    result_path = artifacts / "agent/result.json"
    state = {"stage": "OFFICIAL_PREFLIGHT_STARTED", "worker_complete": False, "reference_fields_provided_to_agent": False}
    write_json(result_path, state)
    manager = None
    try:
        _, adapters, registry = connect_starter_server(config)
        manager = SubprocessManager(timeout_seconds=config["command_timeout_seconds"],
                                   base_dir=Path(config["agent_sandbox_dir"]), system_site_packages=True)
        manager.base_dir.mkdir(parents=True, exist_ok=True)
        raw_exec = manager.exec
        configured = False
        started_monotonic = float(os.environ["GP03_WORKER_STARTED_MONOTONIC"])

        def bounded_exec(sandbox_id, command, *, timeout=None):
            seconds = command_deadline(started_monotonic, config["gp03_worker_cap"]) - time.monotonic()
            if seconds <= 0:
                raise TimeoutError("Official acceptance command budget expired")
            limit = config["command_timeout_seconds"] if timeout is None else timeout
            return raw_exec(sandbox_id, command, timeout=min(seconds, limit))

        def prepare_exec(sandbox_id, command, *, timeout=None):
            nonlocal configured
            if not configured and ".git/info/exclude" in command:
                prepare_snapshot_workspace(bounded_exec, manager, sandbox_id, task, artifacts / "agent")
                prepare_official_workspace(manager, sandbox_id)
                setup = bounded_exec(sandbox_id, 'cd /workspace && git config user.name GP03 && git config user.email gp03@example.invalid', timeout=15)
                if setup.exit_code:
                    raise RuntimeError("Official workspace Git identity setup failed")
                workspace = manager.sandboxes[sandbox_id]["workspace"]
                origin = check_import_origin(bounded_exec, sandbox_id, task.repo.rsplit("/", 1)[-1].replace("-", "_"), workspace)
                write_json(artifacts / "agent/import_origin.json", origin)
                if not origin.get("inside_snapshot") or origin.get("import_error"):
                    raise RuntimeError("The agent's source import did not resolve inside its official sandbox")
                configured = True
            return bounded_exec(sandbox_id, command, timeout=timeout)

        manager.exec = prepare_exec
        eval_config = make_eval_config(config, EvalConfig, registry, adapters, snapshot)
        sampling = yaml.safe_load((Path(config["submission_dir"]) / "configs/sampling.yaml").read_text())
        if (int(sampling["max_output_tokens"]) != 2048
                or int(eval_config.generation_constraints.defaults.get("max_output_tokens", -1)) != 2048):
            raise RuntimeError("Official sampling and generation constraints must both retain the 2048-token cap")
        context = SwegemmaContext(docker_manager=manager, task=task, repo=task.repo, graph_dir=eval_config.graph_dir,
                                 embeddings_dir=eval_config.embeddings_dir, budget=eval_config.budget, harness=eval_config.harness)
        original_tools = context.create_tools
        policy = InvestigationGate(artifacts / "agent/investigation_gate_events.jsonl")

        def agent_tools():
            return policy.wrap(original_tools())

        context.create_tools = agent_tools
        install_official_hook(context=context, manager=manager, config=config, artifacts=artifacts)
        original_prompt = agent_runner.build_agent_prompt

        def record_prompt(*args, **kwargs):
            prompt = original_prompt(*args, **kwargs)
            state["initial_public_prompt_sha256"] = hashlib.sha256(prompt.encode()).hexdigest()
            return prompt

        def emit(kind, tool, detail, **kwargs):
            with (artifacts / "agent/model_probe_events.jsonl").open("a", encoding="utf-8") as stream:
                stream.write(json.dumps({"kind": kind, "tool": tool, "detail": detail}, default=str) + "\n")

        restore_observer = install_model_observer(emit, artifacts, config)
        try:
            restore_guard = (install_evaluation_model_guard() if config["gp03_mode"] == "evaluation"
                             else install_probe_model_guard())
            patch = trace = None
            agent_runner.build_agent_prompt = record_prompt
            try:
                patch, stop, trace = await agent_runner.run_agent_sandbox(manager, eval_config, task, snapshot, context=context)
            except ProbeStop as exc:
                stop = str(exc)
            finally:
                restore_guard()
                agent_runner.build_agent_prompt = original_prompt
        finally:
            restore_observer()
        if config["gp03_mode"] == "evaluation" and patch is not None:
            state.update(record_runner_result(artifacts, patch, trace))
        state.update(stage="OFFICIAL_WORKER_RETURNED", worker_complete=True, agent_error=str(stop or ""),
                     sandbox_source_setup_ran=configured, official_agent_call_counter=context.llm_calls_used)
    except Exception as exc:
        state.update(stage="OFFICIAL_PREFLIGHT_HOLD", worker_error=f"{type(exc).__name__}: {exc}")
        raise
    finally:
        try:
            if manager is not None:
                manager.close()
        except Exception as exc:
            state.update(stage="OFFICIAL_PREFLIGHT_HOLD", worker_complete=False, worker_error=f"Sandbox cleanup failed: {exc}")
            raise
        finally:
            write_json(result_path, state)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    args = parser.parse_args()
    asyncio.run(probe(json.loads(Path(args.config).read_text())))


## 11. gp03_official_driver.py

In [ ]:
%%gp03_source gp03_official_driver.py
"""Check the real organizer worker before enabling any held-out inference."""

import argparse
from datetime import datetime, timezone
import hashlib
import inspect
import json
import os
from pathlib import Path
import shutil
import sys
import tempfile
import time
import traceback
import zipfile

from gp03_official_hook import SANDBOX_STOP, SEED_STOP
from gp03_organizer_runtime import (
    BASELINE_ZIP_SHA256, RUNNER_SHA256, VERIFIER_SHA256, RunClock, _offline_environment,
    _require_l4, _stage_pre_server, find_baseline, local_server, prepare_gpu_runtime,
    require_ready_model, restore_frozen_submission, supervise_worker,
)
from gp03_public_runtime import (
    ADAPTER_RECIPE_SHA256, CAPTURE_PLUGIN, ELIGIBLE_POOL_SHA256, GP03_CPU_EVIDENCE, GP03_DEVELOPMENT,
    GP03_PROPOSED_PAIRS, PAIR_BUDGET, POLICY, WorkspaceFingerprinter, artifact_paths, canonical_hash,
    command_env, environment_fingerprint, environment_profile, git_bytes, load_acceptance_pins,
    load_gp03_inputs, prepare_pytest_config, proposed_pair_lock, read_node_capture, run_visible_phase,
    seal_zip, sha256_file, stage_public_snapshot, verify_zip, visible_definition_binding,
    visible_definition_paths, write_json,
)
from gp03_seed_proxy import seed_proxy

SOURCE = Path(__file__).resolve().parent
PAIRED_ARCHIVE = "GP03_Paired_Runner_CPU_20261109T081452Z_b22e3ff3.zip"
PAIRED_SHA256 = "e702d7c67556a324614a979dab96e0143762a1c54b95b8ffe810cf0eb7f7b7c9"
DEV_ID = "rich_2943"
DEV_PIN = {"task_id": DEV_ID, "repo": "Textualize/rich", "base_commit": "610fd75c433c3dfcd9c296e8b91363979127b9fa",
           "snapshot_sha256": "9b7afcf417a93866ce39ec0feec48e8b2448423f098ec28172171f31dd1e05ad",
           "native_head": "55b05dc6f76df3e88c98499b40b7c161287a68f7", "native_tree": "b421343dd8bf42c5d65c0221cf93acc517aea77a"}
PROBE_SECONDS = 180
SEED_PROBE_SECONDS = 240
MODEL_IDS = ("main_lora", "tool_lora")
PUBLIC_FIELDS = ("instance_id", "repo", "base_commit", "problem_statement", "created_at")


def read_json(path):
    return json.loads(Path(path).read_text())


def locate_paired_archive(working, override=None):
    if override:
        paths = [Path(override)]
    else:
        paths = [working / PAIRED_ARCHIVE, *sorted(Path("/kaggle/input").rglob(PAIRED_ARCHIVE))]
    for path in paths:
        if path.is_file() and sha256_file(path) == PAIRED_SHA256:
            return path
    raise RuntimeError(f"Make the verified {PAIRED_ARCHIVE} available in Kaggle working or as an input")


def verify_paired_archive(path):
    if sha256_file(path) != PAIRED_SHA256:
        raise RuntimeError("Paired CPU evidence checksum changed")
    verify_zip(path)
    with zipfile.ZipFile(path) as archive:
        report = json.loads(archive.read("GP03_Paired_CPU_Report.json"))
        lock = json.loads(archive.read("GP03_Proposed_Execution_Lock.json"))
        cpu_env = json.loads(archive.read("CPU_Environment.json"))
        plan = GP03_PROPOSED_PAIRS
        if (report["status"] != "GP03_PAIRED_CPU_READY" or report["completed_pairs"] != len(plan)
                or report["hold"] or report["evaluation_frozen"] or report["model_calls"]
                or report["protected_verifier_calls"] or report["held_out_agent_attempts"]
                or not report["scripted_cpu_boundary_only"] or lock != proposed_pair_lock(plan)
                or environment_fingerprint(cpu_env) != GP03_CPU_EVIDENCE["cpu_environment_sha256"]):
            raise RuntimeError("Paired CPU lock or acceptance report does not match the frozen inputs")
        cleanup = report["trusted_cleanup"]
        if not cleanup["verified"] or cleanup["registered"] != 2 * len(plan) + 6 or cleanup["survivors"] or cleanup["errors"]:
            raise RuntimeError("Paired CPU pytest cleanup did not reconcile")
        rows = report["rows"]
        if [(row["task_id"], row["arm"]) for row in rows] != [(p["task_id"], arm) for p in plan for arm in p["arm_order"]]:
            raise RuntimeError("Paired CPU rows differ from the proposed task and arm order")
        if len({row["sandbox_root"] for row in rows}) != len(rows):
            raise RuntimeError("Paired CPU arms reused a workspace")
        for number, entry in enumerate(plan):
            pair = rows[2 * number:2 * number + 2]
            if len({row["visible_manifest_sha256"] for row in pair}) != 1 or len({row["baseline_production_sha256"] for row in pair}) != 1:
                raise RuntimeError(f"Paired baselines differ for {entry['task_id']}")
            for row in pair:
                member = f"tasks/{row['task_id']}/{row['arm']}"
                evidence = json.loads(archive.read(member + "/CPU_Arm_Result.json"))
                ledger = json.loads(archive.read(member + "/Ledger.json"))
                if (row != evidence or row["status"] != "PAIR_ARM_READY" or row["gate"] != "SUBMITTED_GATE_VALID"
                        or row["actual_collected"] != entry["expected_collected"] or row["trusted_runs"] != 1
                        or row["native_head"] != entry["native_head"] or row["native_tree"] != entry["native_tree"]
                        or len(ledger["receipts"]) != 1 or ledger["receipts"][0]["provenance"] != "final_gate"
                        or ledger["receipts"][0]["tests_passed"] != entry["expected_collected"]
                        or ledger["manifest_sha256"] != row["visible_manifest_sha256"]):
                    raise RuntimeError(f"Unverified CPU receipt for {row['task_id']} arm {row['arm']}")
        fixture = report["fixture"]
        if ([(row["arm"], row["after_production_edit"]) for row in fixture] != [("B", "STALE"), ("C", "CURRENT_FAIL")]
                or any(row["broken_gate"] != "BLOCKED_VISIBLE_CHECK_FAILED"
                       or row["restored_gate"] != "SUBMITTED_GATE_VALID" for row in fixture)):
            raise RuntimeError("Paired CPU controlled replay differs from the protocol")
    return report, lock, cpu_env


def public_rows(competition, planned):
    expected = {row["task_id"]: row for row in planned}
    public = {}
    with (competition / "tasks.jsonl").open(encoding="utf-8") as stream:
        for line in stream:
            if not line.strip():
                continue
            row = json.loads(line)
            task = row.get("instance_id")
            if task in expected:
                pin = expected[task]
                if task in public or (row["repo"], row["base_commit"]) != (pin["repo"], pin["base_commit"]):
                    raise RuntimeError(f"Wrong public task identity for {task}")
                public[task] = {name: row.get(name, "") for name in PUBLIC_FIELDS}
    if set(public) != set(expected):
        raise RuntimeError("Public task metadata is incomplete")
    return public


def live_manifest(cpu_manifest, environment_sha):
    # Both arms use this new pin only after the official preflight reproduces the original public nodes and outcomes.
    manifest = json.loads(json.dumps(cpu_manifest))
    check = manifest["check"]
    check["command"][0] = sys.executable
    check["pinned_environment_sha256"] = environment_sha
    check["definition_sha256"] = canonical_hash({"task": manifest["task_id"], "argv": check["command"],
                                                  "definition_files_sha256": check["definition_files_sha256"]})
    manifest["environment_sha256"] = environment_sha
    return manifest


def development_manifest(competition, output, environment_sha):
    path = competition / "snapshots" / (DEV_ID + ".tgz")
    if not path.is_file() or sha256_file(path) != DEV_PIN["snapshot_sha256"]:
        raise RuntimeError("The pinned development Rich snapshot is missing or changed")
    output.mkdir()
    with tempfile.TemporaryDirectory(prefix="gp03_development_stage_", dir=output.parent) as scratch:
        temporary = Path(scratch).resolve()
        deadline = time.monotonic() + POLICY["task_wall_seconds"]
        root, package, package_dir, _ = stage_public_snapshot(path, temporary / "checkout", DEV_PIN["repo"], deadline)
        env = command_env(root, temporary)
        for key, ref in (("native_head", "HEAD^{commit}"), ("native_tree", "HEAD^{tree}")):
            if git_bytes(root, env, deadline, "rev-parse", "--verify", ref).decode().strip() != DEV_PIN[key]:
                raise RuntimeError("Development archive native Git identity changed")
        prepare_pytest_config(root)
        source_sha = WorkspaceFingerprinter(root).snapshot().sha256
        files = visible_definition_paths(root)
        definition_sha, links = visible_definition_binding(root, files)
        (temporary / "_gp03_capture.py").write_text(CAPTURE_PLUGIN)
        selector = "tests/test_style.py"
        run, capture = run_visible_phase(selector=selector, root=root, package=package, package_dir=package_dir,
                                        temporary=temporary, output=output / "collection", deadline=deadline,
                                        baseline_sha=source_sha, collect_only=True)
        nodes, unavailable = read_node_capture(capture, selector, root, package_dir)
        if run["exit_code"] or unavailable or len(nodes) != 26:
            raise RuntimeError("The development Rich check must collect the historical 26 nodes")
    return live_manifest({**DEV_PIN, "production_root": package_dir + "/", "baseline_source_sha256": source_sha,
                          "expected_nodeids": nodes, "expected_nodeids_sha256": canonical_hash(nodes),
                          "definition_link_targets": links, "baseline_outcome": "PASS",
                          "check": {"check_id": "gp03_rich_2943_seed_probe", "command": [sys.executable, "-m", "pytest", selector, "-q", "--tb=short"],
                                    "test_selection": selector, "expected_collected": len(nodes), "definition_files": list(files),
                                    "definition_files_sha256": definition_sha, "allow_missing_definition_files": True,
                                    "reserve_seconds": 15.0}}, environment_sha)


def runtime_source_pins():
    from swegemma.context import SwegemmaContext
    from swegemma.harness import verification
    from swegemma.harness.agent_runner import run_agent_sandbox
    from swegemma.sandbox import SubprocessManager

    paths = {"agent_runner": inspect.getsourcefile(run_agent_sandbox), "verifier": inspect.getsourcefile(verification.verify_task),
             "context": inspect.getsourcefile(SwegemmaContext), "subprocess_sandbox": inspect.getsourcefile(SubprocessManager)}
    pins = {key: sha256_file(Path(path)) for key, path in paths.items()}
    if pins["agent_runner"] != RUNNER_SHA256 or pins["verifier"] != VERIFIER_SHA256:
        raise RuntimeError("Installed official runner or verifier differs from the recorded organizer source")
    return pins


def checkpoint_inventory(model_root):
    root = Path(model_root).resolve(strict=True)
    model = read_json(root / "config.json")
    if not (model.get("quantization_config") or (model.get("text_config") or {}).get("quantization_config")):
        raise RuntimeError("Mounted Gemma checkpoint lacks the archived quantization configuration")
    index = root / "model.safetensors.index.json"
    shards = sorted(set(read_json(index)["weight_map"].values())) if index.is_file() else ["model.safetensors"]
    if not shards or any(Path(name).name != name or not (root / name).is_file() for name in shards):
        raise RuntimeError("Mounted Gemma checkpoint weight shards are incomplete")
    names = set(shards) | {path.name for path in root.iterdir() if path.is_file() and (
        path.suffix in {".json", ".model", ".jinja"} or path.name.startswith("tokenizer"))}
    return {name: {"bytes": (root / name).stat().st_size, "sha256": sha256_file(root / name)} for name in sorted(names)}


def run_official_probe(*, mode, public, manifest, counts, arm, settings, template, serving, output, upstream=None):
    task = manifest["task_id"]
    if (mode, task == DEV_ID) not in {("sandbox_probe", False), ("seed_probe", True)}:
        raise RuntimeError("Only the development case may send a live model request")
    output.mkdir(parents=True, exist_ok=False)
    private = settings["root"] / "private" / f"{mode}_{task}"
    private.mkdir()
    artifacts = output / "artifacts"
    cap = SEED_PROBE_SECONDS if mode == "seed_probe" else PROBE_SECONDS
    public_path = private / "public_task.json"
    manifest_path = output / "Public_Check_Manifest.json"
    write_json(public_path, public)
    write_json(manifest_path, manifest)
    event_path = output / "Transport.jsonl"
    event_path.touch(exist_ok=False)
    started = time.monotonic()
    with seed_proxy(model_ids=(*MODEL_IDS, serving["model"]), event_path=event_path, upstream_url=upstream, timeout=cap) as proxy:
        worker = {key: template[key] for key in (
            "version", "model_name", "model_path", "agent_time_minutes", "max_llm_calls", "max_tool_calls",
            "max_output_tokens", "model_context_tokens", "max_file_lines", "max_file_chars", "max_stdout_chars",
            "command_timeout_seconds", "verification_timeout_seconds", "verification_source_sha256",
            "agent_runner_source_sha256", "verify_task_parameters",
        )}
        worker.update(task_id=task, task_expected_repo=public["repo"], task_expected_commit=public["base_commit"],
                      snapshot_pins={task: manifest["snapshot_sha256"]}, run_id=f"gp03_preflight_{task}_{arm}",
                      tasks_sha256=GP03_CPU_EVIDENCE["tasks_sha256"], competition_root=str(settings["competition"]),
                      artifact_dir=str(artifacts), agent_sandbox_dir=str(private / "sandboxes"),
                      evaluator_worker=str(SOURCE / "gp03_frozen_helpers.py"), public_task_path=str(public_path),
                      submission_dir=str(settings["root"] / "private/frozen_submission"), gp03_manifest=str(manifest_path),
                      gp03_manifest_sha256=sha256_file(manifest_path), gp03_expected_counts=list(counts),
                      gp03_mode=mode, gp03_arm=arm, gp03_worker_cap=cap,
                      starter_server={"base_url": proxy.base_url, "config": {**serving, "port": proxy.server_port}})
        config_path = private / "worker_config.json"
        write_json(config_path, worker)
        env = dict(os.environ, GP03_WORKER_STARTED_MONOTONIC=str(started), PYTHONPATH=str(SOURCE),
                   NO_PROXY="127.0.0.1,localhost", no_proxy="127.0.0.1,localhost")
        env.pop("GP02C_TRUSTED_PYTEST_REGISTRY", None)
        supervisor = supervise_worker((sys.executable, "-u", str(SOURCE / "gp03_official_worker.py"), "--config", str(config_path)),
                                      cwd=private, evidence_root=output, overall_seconds=cap, env=env)
    state_path = artifacts / "agent/result.json"
    state = read_json(state_path) if state_path.is_file() else {}
    attestation_path = artifacts / "Official_Worker_Attestation.json"
    attestation = read_json(attestation_path) if attestation_path.is_file() else {}
    seed_path = artifacts / "Official_Model_Probe.json"
    seed = read_json(seed_path) if seed_path.is_file() else {}
    expected_stop = SEED_STOP if mode == "seed_probe" else SANDBOX_STOP
    stop = str(state.get("agent_error") or state.get("worker_error") or "")
    trusted_ok = (supervisor["trusted_process_cleanup_verified"] and not supervisor["timed_out"]
                  and supervisor["returncode"] == 0 and state.get("worker_complete")
                  and supervisor["trusted_child_cleanup"]["registered"] == 1)
    attested = (attestation.get("status") == "OFFICIAL_PUBLIC_GATE_ACCEPTED"
                and attestation.get("public_counts") == list(counts)
                and attestation.get("check_manifest_sha256") == worker["gp03_manifest_sha256"]
                and state.get("reference_fields_provided_to_agent") is False and state.get("sandbox_source_setup_ran")
                and expected_stop in stop)
    transport_ok = (proxy.forwarded == 0 and proxy.denied == 0 and not seed and state.get("official_agent_call_counter") == 0
                    if mode == "sandbox_probe" else
                    proxy.forwarded == 1 and proxy.denied == 0 and len(proxy.completions) == 1
                    and proxy.completions[0]["status"] == 200 and proxy.completions[0]["completed"]
                    and seed.get("status") == "OFFICIAL_FIRST_RESPONSE_RECEIVED"
                    and seed.get("response_delivered_to_agent") is False)
    row = {"task_id": task, "mode": mode, "arm": arm, "public_counts": attestation.get("public_counts"),
           "check_manifest_sha256": worker["gp03_manifest_sha256"], "official_public_gate_accepted": bool(attested),
           "trusted_process_cleanup_verified": trusted_ok, "transport_verified": bool(transport_ok),
           "model_requests_forwarded": proxy.forwarded, "denied_model_requests": proxy.denied,
           "status": "READY" if trusted_ok and attested and transport_ok else "HOLD",
           "worker_stop": stop[:700], "elapsed_seconds": round(time.monotonic() - started, 3)}
    write_json(output / "Probe_Result.json", row)
    return row


def prepare(settings):
    root = settings["root"]
    private, evidence = root / "private", root / "evidence"
    competition, working = settings["competition"], settings["working"]
    report = {"schema": "gp03.official-preflight.v1", "status": "GP03_OFFICIAL_PREPARE_HOLD",
              "model_calls": 0, "held_out_agent_attempts": 0, "evaluation_frozen": False, "rows": []}
    evidence.mkdir(parents=True, exist_ok=False)
    private.mkdir()
    clock = RunClock()
    try:
        with clock.phase("Verify CPU handoffs and public task release"):
            paired, lock, cpu_env = verify_paired_archive(locate_paired_archive(working, settings.get("paired_zip")))
            load_gp03_inputs(competition, GP03_PROPOSED_PAIRS, GP03_DEVELOPMENT, GP03_CPU_EVIDENCE["tasks_sha256"])
            pins = load_acceptance_pins(working, settings.get("cpu_zip"), GP03_CPU_EVIDENCE, GP03_PROPOSED_PAIRS)
            public = public_rows(competition, [*GP03_PROPOSED_PAIRS, DEV_PIN])
            baseline = find_baseline(settings.get("baseline_zip"), working)
            template, serving = restore_frozen_submission(baseline, private / "frozen_submission")
            write_json(private / "Trial_Template.json", template)
            write_json(private / "Serving_Config.json", serving)
            write_json(evidence / "CPU_Execution_Lock.json", lock)
            write_json(evidence / "Source_Pins.json", {
                "paired_cpu_evidence_sha256": PAIRED_SHA256, "paired_cpu_run": paired["run_id"],
                "frozen_submission_archive_sha256": BASELINE_ZIP_SHA256, "eligible_pool_sha256": ELIGIBLE_POOL_SHA256,
                "adapter_recipe_sha256": ADAPTER_RECIPE_SHA256,
                "source_sha256": {p.name: sha256_file(p) for p in sorted(SOURCE.glob("*.py"))},
            })
        with clock.phase("Prepare pinned offline organizer runtime"):
            report["hardware"] = _require_l4()
            _offline_environment()
            prepare_gpu_runtime(settings["wheelhouse"], private, settings["allow_offline_install"])
            for name in ("GPU_Runtime.json", "offline_runtime_install.log"):
                if (private / name).is_file():
                    shutil.copyfile(private / name, evidence / name)
            report["official_sources"] = runtime_source_pins()
            profile = environment_profile()
            gpu_env = environment_fingerprint(profile)
            write_json(evidence / "GPU_Public_Test_Environment.json", profile)
            report.update(cpu_environment_sha256=environment_fingerprint(cpu_env), gpu_environment_sha256=gpu_env,
                          public_runtime_matches_cpu=profile == cpu_env)
            clock.log(f"Public test environment {'matches CPU' if profile == cpu_env else 'has a separate GPU fingerprint'}")
        with clock.phase("Verify organizer graph inputs and CUDA devices"):
            _stage_pre_server(serving, competition, public, tuple(public), evidence)
        for entry in GP03_PROPOSED_PAIRS:
            task = entry["task_id"]
            cpu_manifest, counts, _ = pins[task]
            arm = entry["arm_order"][0]
            with clock.phase("Official worker baseline gate", task=task):
                row = run_official_probe(mode="sandbox_probe", public=public[task], manifest=live_manifest(cpu_manifest, gpu_env),
                                         counts=counts, arm=arm, settings=settings, template=template, serving=serving,
                                         output=evidence / "official_sandboxes" / task)
                report["rows"].append(row)
                clock.log(f"{task} arm {arm} | {row['status']} | public counts {row['public_counts']} | model calls {row['model_requests_forwarded']}")
                if row["status"] != "READY":
                    raise RuntimeError(f"Official sandbox probe held for {task}. See its probe result and worker log.")
        with clock.phase("Stage public development seed probe and pin checkpoint"):
            manifest = development_manifest(competition, evidence / "development_collection", gpu_env)
            write_json(private / "Development_Manifest.json", manifest)
            write_json(private / "Development_Public_Task.json", public[DEV_ID])
            write_json(evidence / "Gemma_Checkpoint.json", {"mount": serving["model"], "files": checkpoint_inventory(serving["model"])})
        report["status"] = "GP03_OFFICIAL_SANDBOXES_READY"
    except Exception as exc:
        report["hold"] = f"{type(exc).__name__}: {exc}"
        (evidence / "Prepare_Error.txt").write_text(traceback.format_exc())
        clock.log(report["hold"])
    finally:
        report.update(elapsed_seconds=clock.elapsed(), phases=clock.phases,
                      model_calls=sum(row["model_requests_forwarded"] for row in report["rows"]))
        write_json(evidence / "Official_Prepare_Report.json", report)
    print(f"{report['status']} | {sum(row['status']=='READY' for row in report['rows'])}/9 official sandboxes | model calls {report['model_calls']}", flush=True)


def require_prepared(root):
    report = read_json(root / "evidence/Official_Prepare_Report.json")
    if (report["status"] != "GP03_OFFICIAL_SANDBOXES_READY" or len(report["rows"]) != len(GP03_PROPOSED_PAIRS)
            or any(row["status"] != "READY" for row in report["rows"]) or report["model_calls"]):
        raise RuntimeError("Complete all official sandbox checks before loading vLLM")
    if runtime_source_pins() != report["official_sources"]:
        raise RuntimeError("The official runtime source changed after sandbox acceptance")
    if environment_fingerprint(environment_profile()) != report["gpu_environment_sha256"]:
        raise RuntimeError("The public test environment changed after sandbox acceptance")
    return report


def warm(settings):
    root = settings["root"]
    private, evidence = root / "private", root / "evidence"
    report = require_prepared(root)
    clock = RunClock()
    _offline_environment()
    serving = read_json(private / "Serving_Config.json")
    # The existing warm-up helper uses this private frozen submission and writes its own server configuration.
    ready_path = private / "Warm_Server.json"
    try:
        with local_server(serving, private, clock, report) as server:
            receipt = {"owner_pid": os.getpid(), "server": server, "started_utc": datetime.now(timezone.utc).isoformat(),
                       "startup_seconds": report["server_start_seconds"], "hardware": report["hardware"]}
            write_json(evidence / "Server_Config.json", receipt)
            write_json(ready_path, receipt)
            print("GP03_VLLM_READY | local pinned Gemma 31B", flush=True)
            sys.stdin.read()
    finally:
        ready_path.unlink(missing_ok=True)
        if (private / "gemma_server.log").is_file():
            shutil.copyfile(private / "gemma_server.log", evidence / "gemma_server.log")


def seed(settings):
    root = settings["root"]
    private, evidence = root / "private", root / "evidence"
    report = {"status": "GP03_OFFICIAL_WORKER_HOLD", "model_calls": 0,
              "held_out_agent_attempts": 0, "evaluation_frozen": False, "effective_seed_verified": False}
    _offline_environment()
    try:
        require_prepared(root)
        warm = read_json(private / "Warm_Server.json")
        os.kill(warm["owner_pid"], 0)
        template = read_json(private / "Trial_Template.json")
        serving = read_json(private / "Serving_Config.json")
        if warm["server"]["config"] != serving:
            raise RuntimeError("Warmed server configuration differs from the pinned serving profile")
        base_url = warm["server"]["base_url"]
        for model_id in (serving["model"], *MODEL_IDS):
            require_ready_model(base_url, model_id)
        print(f"{DEV_ID} | starting one seeded development request | worker cap {SEED_PROBE_SECONDS}s", flush=True)
        row = run_official_probe(mode="seed_probe", public=read_json(private / "Development_Public_Task.json"),
                                 manifest=read_json(private / "Development_Manifest.json"), counts=(26, 26, 0, 0, 0), arm="B",
                                 settings=settings, template=template, serving=serving,
                                 output=evidence / "development_seed", upstream=base_url)
        report.update(probe=row, model_calls=row["model_requests_forwarded"])
        if row["status"] != "READY":
            raise RuntimeError("The development seed or official response probe held. Review its transport and worker logs.")
        transport = [json.loads(line) for line in (evidence / "development_seed/Transport.jsonl").read_text().splitlines()]
        requests = [event for event in transport if event["event"] == "VLLM_REQUEST"]
        if len(requests) != 1 or requests[0]["forwarded_seed"] != 42 or requests[0]["max_output_tokens"] != 2048:
            raise RuntimeError("The actual forwarded seed and output cap did not reconcile")
        report.update(status="GP03_OFFICIAL_WORKER_READY", effective_seed_verified=True,
                      observed_seed=42, max_output_tokens=2048, official_sandboxes_ready=len(GP03_PROPOSED_PAIRS))
    except Exception as exc:
        report["hold"] = f"{type(exc).__name__}: {exc}"
        (evidence / "Seed_Error.txt").write_text(traceback.format_exc())
        print(report["hold"], flush=True)
    finally:
        transport_path = evidence / "development_seed/Transport.jsonl"
        if transport_path.is_file():
            report["model_calls"] = sum(json.loads(line).get("event") == "VLLM_REQUEST"
                                        for line in transport_path.read_text().splitlines() if line)
        write_json(evidence / "Official_Seed_Report.json", report)
    print(f"{report['status']} | forwarded model calls {report['model_calls']} | held-out model calls 0", flush=True)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("action", choices=("prepare", "warm", "seed"))
    parser.add_argument("settings")
    args = parser.parse_args()
    settings = read_json(args.settings)
    for key in ("root", "working", "competition", "wheelhouse"):
        settings[key] = Path(settings[key]).resolve()
    {"prepare": prepare, "warm": warm, "seed": seed}[args.action](settings)


if __name__ == "__main__":
    main()


## 12. gp03_private_grader.py

In [ ]:
%%gp03_source gp03_private_grader.py
"""GP02A evaluator worker; reference controls and agent patch validation only."""
from __future__ import annotations

import argparse
import asyncio
import dataclasses
import hashlib
import importlib.metadata
import inspect
import json
import os
import re
import shlex
import sys
import time
import xml.etree.ElementTree as ET
from pathlib import Path


def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + '\n')
    temporary.replace(path)


def prepare_snapshot_workspace(original_exec, manager, sandbox_id, task, artifact_dir):
    """Use the supplied archive unchanged and put its source ahead of installed packages."""
    paths = manager.sandboxes[sandbox_id]
    workspace = Path(paths['workspace']).resolve()
    package = task.repo.rsplit('/', 1)[-1].replace('-', '_')
    diagnostic = {'declared_base_commit': task.base_commit, 'workspace': str(workspace),
                  'package': package, 'status': 'PREPARING'}
    diagnostic_path = Path(artifact_dir) / 'workspace_setup.json'

    def execute(command):
        return original_exec(sandbox_id, command, timeout=60)

    def fail(message):
        diagnostic['status'] = 'WORKSPACE_HOLD'
        diagnostic['error'] = message
        write_json(diagnostic_path, diagnostic)
        raise RuntimeError(message)

    before = execute('cd /workspace && git rev-parse HEAD')
    diagnostic['snapshot_head_before'] = (before.stdout or '').strip()
    diagnostic['root_entries_before'] = sorted(p.name for p in workspace.iterdir())[:100]
    if before.exit_code != 0 or not diagnostic['snapshot_head_before']:
        fail('The published snapshot has no readable Git HEAD; inspect its setup diagnostics.')
    commit = shlex.quote(task.base_commit + '^{commit}')
    exists = execute('cd /workspace && git cat-file -e ' + commit)
    diagnostic['declared_commit_exists'] = exists.exit_code == 0
    diagnostic['declared_commit_id_matches_head'] = diagnostic['snapshot_head_before'] == task.base_commit
    tree = execute('cd /workspace && git rev-parse HEAD^{tree}')
    diagnostic['snapshot_git_tree'] = (tree.stdout or '').strip()
    if tree.exit_code != 0 or not diagnostic['snapshot_git_tree']:
        fail('The published snapshot has no readable Git tree.')
    diagnostic['identity_policy'] = 'task_named_published_archive_sha256_and_recorded_native_git_identity'
    diagnostic['original_commit_id_required'] = False
    # The official verifier consumes the packaged snapshot directly. Its rebuilt
    # Git history need not retain the original repository's commit identifiers.
    # Do not fetch another repository or rewrite the supplied working tree.

    candidates = []
    for root in [workspace, workspace / 'src']:
        if (root / package / '__init__.py').is_file() or (root / (package + '.py')).is_file():
            candidates.append(root)
    diagnostic['source_roots_found'] = [str(p) for p in candidates]
    if len(candidates) != 1:
        fail('Expected one explicit root/src source location for the repository package; inspect workspace_setup.json.')
    source_root = candidates[0]
    environment = Path(paths['venv']).resolve()
    site_dirs = {p.resolve() for p in environment.glob('lib/python*/site-packages') if p.is_dir()}
    if len(site_dirs) != 1:
        fail('Could not identify the sandbox virtual environment site-packages directory.')
    site_dir = next(iter(site_dirs))
    if not site_dir.is_relative_to(environment):
        fail('Sandbox source-path setup would escape its own virtual environment.')
    overlay = site_dir / 'zz_gp01b_snapshot_source.pth'
    overlay.write_text('import sys; sys.path.insert(0, ' + repr(str(source_root)) + ')\n', encoding='utf-8')
    diagnostic['source_root'] = str(source_root)
    diagnostic['source_path_file'] = str(overlay)
    diagnostic['status'] = 'WORKSPACE_READY'
    write_json(diagnostic_path, diagnostic)
    return diagnostic


def verify_snapshot_archive(path, expected_sha256):
    actual = digest(path)
    if not expected_sha256 or actual != expected_sha256:
        raise RuntimeError('Published snapshot bytes differ from the pinned task archive; review the dataset release.')
    return actual


def check_import_origin(original_exec, sandbox_id, package, workspace):
    script = '''import importlib, importlib.util, json, sys
from pathlib import Path
name = sys.argv[1]
spec = importlib.util.find_spec(name)
out = {"package": name, "spec_origin": str(spec.origin) if spec else None}
try:
    module = importlib.import_module(name)
    out["imported_file"] = str(getattr(module, "__file__", None))
    out["version"] = str(getattr(module, "__version__", "unknown"))
except Exception as exc:
    out["import_error"] = type(exc).__name__ + ": " + str(exc)
print(json.dumps(out))
'''
    command = 'python3 -s -c ' + shlex.quote(script) + ' ' + shlex.quote(package)
    result = original_exec(sandbox_id, command, timeout=60)
    try:
        data = json.loads((result.stdout or '').splitlines()[-1])
    except (ValueError, IndexError):
        data = {'import_error': 'Origin probe did not return JSON.', 'stderr': result.stderr or ''}
    actual = data.get('imported_file')
    data['inside_snapshot'] = bool(actual and Path(actual).resolve().is_relative_to(Path(workspace).resolve()))
    data['probe_exit_code'] = result.exit_code
    return data


def junit_summary(text):
    """Require actual testcase records, not only a successful process exit."""
    out = {'valid_xml': False, 'tests': 0, 'passed': 0, 'failures': 0,
           'errors': 0, 'skipped': 0, 'failed_nodes': []}
    try:
        root = ET.fromstring(text)
    except (ET.ParseError, TypeError):
        return out
    suites = [root] if root.tag == 'testsuite' else list(root.findall('.//testsuite'))
    if not suites:
        return out
    cases = list(root.findall('.//testcase'))
    out['valid_xml'] = True
    out['tests'] = len(cases)
    for case in cases:
        name = case.attrib.get('name', '')
        classname = case.attrib.get('classname', '')
        node = classname + '::' + name if classname else name
        if case.find('error') is not None:
            out['errors'] += 1
        elif case.find('failure') is not None:
            out['failures'] += 1
            out['failed_nodes'].append(node)
        elif case.find('skipped') is not None:
            out['skipped'] += 1
        else:
            out['passed'] += 1
    # Collection errors can also appear only in suite counters.
    try:
        out['errors'] = max(out['errors'], sum(int(s.attrib.get('errors', 0)) for s in suites))
    except ValueError:
        out['valid_xml'] = False
    return out


def classify_pair(baseline, reference):
    if not baseline or not reference:
        return {'qualified': False, 'status': 'PENDING', 'reason': 'Both phases are required.'}
    for phase in (baseline, reference):
        if phase.get('worker_error') or phase.get('pytest_calls') != 1:
            return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'A phase did not reach exactly one pytest execution.'}
        if not phase.get('junit', {}).get('valid_xml') or phase['junit'].get('errors', 0) > 0:
            return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'Missing/malformed JUnit or collection/setup errors.'}
        if phase.get('baseline_commit_exit_code') != 0 or not phase.get('snapshot_identity_verified'):
            return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'Published archive identity or baseline commit was not verified.'}
        if not all(phase.get('workspace_setup', {}).get(k) for k in ('snapshot_head_before', 'snapshot_git_tree')):
            return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'Native Git snapshot identity was not recorded.'}
        if not phase.get('import_origin', {}).get('inside_snapshot'):
            return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'The repository package was not imported from the active snapshot.'}
    if baseline.get('test_exit_code') == 0:
        return {'qualified': False, 'status': 'BASELINE_ALREADY_PASSES', 'reason': 'The unmodified snapshot did not reproduce a failing test.'}
    if baseline.get('test_exit_code') != 1 or baseline['junit'].get('failures', 0) < 1:
        return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'The baseline did not produce ordinary test failures with pytest exit 1.'}
    if not baseline.get('required_failure_matched'):
        return {'qualified': False, 'status': 'REPRODUCTION_REVIEW', 'reason': 'No baseline failure was matched to a required verification test.'}
    rj = reference['junit']
    if (not reference.get('resolved') or reference.get('test_exit_code') != 0
            or rj.get('passed', 0) < 1 or rj.get('failures', 0) or rj.get('errors', 0)
            or not reference.get('official_junit_ok')):
        return {'qualified': False, 'status': 'REFERENCE_NOT_VERIFIED', 'reason': 'The reference fix did not satisfy both the official verifier and independent JUnit checks.'}
    if baseline.get('snapshot_sha256') != reference.get('snapshot_sha256'):
        return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'The phases used different snapshot bytes.'}
    for key in ('snapshot_head_before', 'snapshot_git_tree'):
        if baseline.get('workspace_setup', {}).get(key) != reference.get('workspace_setup', {}).get(key):
            return {'qualified': False, 'status': 'INFRA_HOLD', 'reason': 'The phases used different native Git snapshot identities.'}
    return {'qualified': True, 'status': 'QUALIFIED', 'reason': 'Baseline required-test failure and reference-fix pass were observed in separate fresh sandboxes.'}


def load_official(config):
    # These switches prevent optional model metadata downloads during imports.
    os.environ.setdefault('LITELLM_LOCAL_MODEL_COST_MAP', 'True')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
    from adk_submission import ModelRegistry
    from swegemma.config import EvalConfig
    from swegemma.models import load_tasks
    from swegemma.sandbox import SubprocessManager
    from swegemma.harness import verification
    source_path = Path(inspect.getsourcefile(verification.verify_task))
    actual_hash = digest(source_path)
    if actual_hash != config['verification_source_sha256']:
        raise RuntimeError('Installed verification.py differs from the GP01A-inspected source.')
    parameters = list(inspect.signature(verification.verify_task).parameters)
    if parameters != config['verify_task_parameters']:
        raise RuntimeError('Verifier signature differs from the inspected contract.')
    from packaging.version import Version
    adk_version = importlib.metadata.version('google-adk')
    if not Version('1.34.0') <= Version(adk_version) < Version('2'):
        raise RuntimeError('The isolated runtime must use google-adk>=1.34,<2.')
    return ModelRegistry, EvalConfig, load_tasks, SubprocessManager, verification


def preflight(config):
    api = load_official(config)
    names = {d.metadata['Name'] for d in importlib.metadata.distributions() if d.metadata.get('Name')}
    package_versions = {name: importlib.metadata.version(name) for name in names}
    return {'status': 'RUNTIME_READY', 'python': sys.version.split()[0],
            'verification_source_sha256': config['verification_source_sha256'],
            'verify_task_signature': str(inspect.signature(api[-1].verify_task)),
            'packages': dict(sorted(package_versions.items())), 'model_calls': 0}


async def run_phase(config, task_id, phase):
    ModelRegistry, EvalConfig, load_tasks, SubprocessManager, verification = load_official(config)
    data_root = Path(config['competition_root'])
    if digest(data_root / 'tasks.jsonl') != config['tasks_sha256']:
        raise RuntimeError('Task release changed since GP00/GP01A.')
    tasks = {t.instance_id: t for t in load_tasks(data_root / 'tasks.jsonl')}
    if task_id not in config['development_task_ids']:
        raise RuntimeError('This evaluator is restricted to the frozen task assignment.')
    raw_task = tasks[task_id]
    expected = config['development_metadata'][task_id]
    if (raw_task.repo, raw_task.base_commit) != (expected['repo'], expected['base_commit']):
        raise RuntimeError('Task identity differs from the GP00 candidate manifest.')
    reference_patch = raw_task.patch
    task = dataclasses.replace(raw_task, patch='', hints_text='')
    del raw_task, tasks
    snapshot = data_root / 'snapshots' / (task_id + '.tgz')
    snapshot_hash = verify_snapshot_archive(snapshot, config['snapshot_pins'].get(task_id))
    artifact_dir = Path(config['artifact_dir']) / 'evaluator_only' / task_id / phase
    artifact_dir.mkdir(parents=True, exist_ok=True)
    manager = SubprocessManager(timeout_seconds=config['command_timeout_seconds'],
                                base_dir=Path(config['sandbox_dir']), system_site_packages=True)
    manager.base_dir.mkdir(parents=True, exist_ok=True)
    original_exec = manager.exec
    test_events = []
    git_setup = {}
    workspace_setup = None
    import_origin = {}

    def recorded_exec(sandbox_id, command, *, timeout=None):
        nonlocal workspace_setup, import_origin
        # This is the first official setup command after snapshot extraction,
        # before any test configuration or reference patch has been applied.
        if workspace_setup is None and '.git/info/exclude' in command:
            workspace_setup = prepare_snapshot_workspace(original_exec, manager, sandbox_id, task, artifact_dir)
        if 'git commit -m "eval_baseline"' in command:
            # The sanitized subprocess HOME has no global Git identity. Set only
            # sandbox-local author metadata, identically in both phases.
            identity = original_exec(sandbox_id,
                'cd /workspace && git config user.name GP01B && git config user.email gp01b@example.invalid && git rev-parse HEAD', timeout=30)
            git_setup['snapshot_head_matches'] = identity.exit_code == 0 and (identity.stdout or '').strip() == task.base_commit
            git_setup['native_snapshot_head_unchanged'] = bool(workspace_setup and identity.exit_code == 0
                and (identity.stdout or '').strip() == workspace_setup['snapshot_head_before'])
        is_pytest = bool(re.search(r'python3\s+-s\s+-m\s+pytest\b', command) and '--junitxml=' in command)
        if is_pytest:
            if workspace_setup is None:
                raise RuntimeError('Snapshot setup did not run before pytest; inspect the harness setup sequence.')
            import_origin = check_import_origin(original_exec, sandbox_id, task.repo_name.replace('-', '_'), manager.sandboxes[sandbox_id]['workspace'])
            write_json(artifact_dir / 'import_origin.json', import_origin)
            if import_origin.get('import_error') or not import_origin.get('inside_snapshot'):
                raise RuntimeError('Repository import did not resolve inside the checked-out snapshot. See import_origin.json.')
        result = original_exec(sandbox_id, command, timeout=timeout)
        if 'git commit -m "eval_baseline"' in command:
            git_setup['baseline_commit_exit_code'] = result.exit_code
        if is_pytest:
            xml_match = re.search(r'--junitxml=([^\s]+)', command)
            xml_text = ''
            if xml_match:
                xml_path = xml_match.group(1)
                xml_result = original_exec(sandbox_id, 'cat ' + shlex.quote(xml_path) + ' 2>/dev/null || true', timeout=30)
                xml_text = xml_result.stdout or ''
            serial = len(test_events) + 1
            (artifact_dir / f'pytest_{serial}.xml').write_text(xml_text, encoding='utf-8')
            (artifact_dir / f'pytest_{serial}.log').write_text(
                'COMMAND: ' + command + '\nSTDOUT:\n' + (result.stdout or '') + '\nSTDERR:\n' + (result.stderr or ''), encoding='utf-8')
            test_events.append({'exit_code': result.exit_code, 'xml': xml_text})
        return result

    # Instance instrumentation preserves the official SubprocessManager class identity
    # and passes each command and return value through unchanged.
    manager.exec = recorded_exec
    eval_config = EvalConfig(tasks_path=data_root / 'tasks.jsonl', snapshots_dir=data_root / 'snapshots',
                             results_dir=artifact_dir, submission_dir=artifact_dir / 'unused_submission',
                             models=ModelRegistry(), sandbox='subprocess', wheels_dir=data_root / 'wheels',
                             timeout_seconds=config['command_timeout_seconds'],
                             verification_timeout_seconds=config['verification_timeout_seconds'], verbose=False)
    started = time.perf_counter()
    try:
        result = await verification.verify_task(manager, eval_config, task, snapshot,
                                               agent_patch=(Path(config['agent_patch_path']).read_text(encoding='utf-8') if phase == 'candidate' else (reference_patch if phase == 'reference' else '')),
                                               start_time=started)
        xml_text = test_events[-1]['xml'] if test_events else ''
        parsed = junit_summary(xml_text)
        official_ok, official_reason = verification._validate_junit_xml(
            xml_text, fail_to_pass=task.FAIL_TO_PASS, pass_to_pass=task.PASS_TO_PASS, test_patch=task.test_patch)
        required = list(task.FAIL_TO_PASS) or verification._extract_test_functions_from_patch(task.test_patch)
        matched = any(verification._node_matches_requirement(node, req, req.split('::')[-1], req.split('::')[-1].split('[', 1)[0])
                      for node in parsed['failed_nodes'] for req in required)
        payload = {'instance_id': task_id, 'repo': task.repo, 'phase': phase,
                   'snapshot_sha256': snapshot_hash, 'reference_patch_sha256': hashlib.sha256(reference_patch.encode()).hexdigest(),
                   'resolved': bool(result.resolved), 'test_exit_code': result.test_exit_code,
                   'error': getattr(result, 'error_message', None), 'pytest_calls': len(test_events),
                   'junit': parsed, 'official_junit_ok': official_ok, 'official_junit_reason': official_reason,
                   'required_tests': required, 'required_failure_matched': matched,
                   'baseline_commit_exit_code': git_setup.get('baseline_commit_exit_code'),
                   'snapshot_head_matches': git_setup.get('snapshot_head_matches', False),
                   'snapshot_identity_verified': bool(workspace_setup and workspace_setup['status'] == 'WORKSPACE_READY'
                       and git_setup.get('native_snapshot_head_unchanged')
                       and snapshot_hash == config['snapshot_pins'].get(task_id)),
                   'workspace_setup': workspace_setup,
                   'import_origin': import_origin,
                   'duration_seconds': round(time.perf_counter() - started, 3), 'model_calls': 0,
                   'environment_profile': 'official_subprocess_pinned_published_snapshot_and_source_path',
                   'raw_reference_patch_exported': False, 'raw_test_patch_exported': False}
        # Never serialize TaskResult wholesale: its agent_patch contains the reference fix.
        result_output = result.test_output or ''
        if reference_patch.strip() and reference_patch.strip() in result_output:
            result_output = result_output.replace(reference_patch.strip(), '[REFERENCE PATCH REMOVED]')
        if task.test_patch.strip() and task.test_patch.strip() in result_output:
            result_output = result_output.replace(task.test_patch.strip(), '[VERIFICATION PATCH REMOVED]')
        (artifact_dir / 'verifier_output.log').write_text(result_output, encoding='utf-8')
        write_json(artifact_dir / 'result.json', payload)
        print(json.dumps({'instance_id': task_id, 'phase': phase, 'resolved': payload['resolved'],
                          'exit_code': payload['test_exit_code'], 'junit': {k:v for k,v in parsed.items() if k != 'failed_nodes'}}), flush=True)
        return payload
    finally:
        manager.close()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', required=True)
    parser.add_argument('--preflight', action='store_true')
    parser.add_argument('--task-id')
    parser.add_argument('--phase', choices=['baseline', 'reference', 'candidate'])
    args = parser.parse_args()
    config = json.loads(Path(args.config).read_text())
    if args.preflight:
        result = preflight(config)
        write_json(Path(config['artifact_dir']) / 'runtime_preflight.json', result)
        print('Runtime import and verifier-source checks passed.', flush=True)
    else:
        try:
            asyncio.run(run_phase(config, args.task_id, args.phase))
        except Exception as exc:
            payload = {'instance_id': args.task_id, 'phase': args.phase,
                       'worker_error': type(exc).__name__ + ': ' + str(exc), 'model_calls': 0}
            write_json(Path(config['artifact_dir']) / 'evaluator_only' / args.task_id / args.phase / 'result.json', payload)
            print(json.dumps(payload), flush=True)
            raise


if __name__ == '__main__':
    main()


## 13. gp03_grade_runtime.py

In [ ]:
%%gp03_source gp03_grade_runtime.py
from importlib import metadata
import os
from pathlib import Path

from gp03_public_runtime import sha256_file

_G15D_CORE = {"adk-eval-core", "adk-submission", "swegemma", "google-adk"}

_G15D_SUPPORT = {
    "accelerate", "cachetools", "docker", "google-genai", "litellm", "mcp",
    "networkx", "pandas", "pydantic", "python-dotenv", "pyyaml", "rich",
    "safetensors", "tokenizers", "transformers",
}

_G15D_WORKER_ENTRY = """
from importlib import import_module
from pathlib import Path
import runpy
import sys
import time

worker, seconds = sys.argv[1:3]
sys.path.insert(0, str(Path(worker).parent))
sys.argv = [worker, *sys.argv[3:]]
deadline = time.monotonic() + float(seconds)
sandbox = import_module("swegemma.sandbox.subprocess")
execute = sandbox.execute_subprocess_command


def bounded_command(*args, timeout=None, **kwargs):
    remaining = deadline - time.monotonic()
    if remaining <= 0:
        raise TimeoutError("Independent evaluator command deadline expired")
    # The organizer executor terminates its own detached command groups on
    # timeout. Let it do that before the external worker cap is reached.
    limit = remaining if timeout is None else min(timeout, remaining)
    return execute(*args, timeout=limit, **kwargs)


sandbox.execute_subprocess_command = bounded_command
runpy.run_path(worker, run_name="__main__")
"""

def _g15d_offline_env(private):
    env = os.environ.copy()
    for name in ("PYTHONPATH", "PYTHONHOME", "GP02C_TRUSTED_PYTEST_REGISTRY"):
        env.pop(name, None)
    env.update(
        CUDA_VISIBLE_DEVICES="", HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
        TRANSFORMERS_NO_TF="1", LITELLM_LOCAL_MODEL_COST_MAP="True",
        LITELLM_TELEMETRY="False", OTEL_SDK_DISABLED="true",
        PIP_NO_INDEX="1", PIP_NO_INPUT="1", PIP_CONFIG_FILE=os.devnull,
        PYTHONNOUSERSITE="1", PYTHONDONTWRITEBYTECODE="1", TMPDIR=str(private),
    )
    return env

def _g15d_mounted_wheels(competition, wheelhouse, core_pins):
    from packaging.tags import sys_tags
    from packaging.utils import parse_wheel_filename

    missing = set()
    for name in _G15D_SUPPORT:
        try:
            metadata.version(name)
        except metadata.PackageNotFoundError:
            missing.add(name)
    tags = set(sys_tags())
    pinned, support = {}, {}
    for folder in (wheelhouse, competition / "wheels"):
        for path in sorted(folder.rglob("*.whl")):
            try:
                name, version, _, wheel_tags = parse_wheel_filename(path.name)
            except ValueError:
                continue
            if tags.isdisjoint(wheel_tags):
                continue
            if path.name in core_pins and sha256_file(path) == core_pins[path.name]:
                pinned[path.name] = path
            if name in missing and (name not in support or version > support[name][0]):
                support[name] = (version, path)
    if set(pinned) != set(core_pins):
        absent = ", ".join(sorted(set(core_pins) - set(pinned)))
        raise FileNotFoundError(f"Mount the organizer wheelhouse with the pinned wheels: {absent}")
    return [*pinned.values(), *(item[1] for item in support.values())]

# Reused from the pinned GP02C-04B source gp02c_dev_frozen_worker.py.
def classify_candidate(result):
    if not result or result.get('worker_error'):
        return 'VERIFICATION_HOLD'
    j = result.get('junit', {})
    if (result.get('pytest_calls') != 1 or not j.get('valid_xml') or j.get('errors', 0)
            or not result.get('snapshot_identity_verified')
            or not result.get('import_origin', {}).get('inside_snapshot')
            or result.get('baseline_commit_exit_code') != 0):
        return 'VERIFICATION_HOLD'
    if (result.get('resolved') and result.get('test_exit_code') == 0
            and result.get('official_junit_ok') and j.get('passed', 0) > 0
            and not j.get('failures', 0)):
        return 'PATCH_VERIFIED'
    if result.get('test_exit_code') == 1 and j.get('failures', 0) > 0:
        return 'PATCH_NOT_VERIFIED'
    return 'VERIFICATION_HOLD'

# Reused from the pinned GP02C-04B source gp02c04b_reporting.py.
def delivery_gate_matches(audit: dict, ledger: dict, patch_sha256: str) -> bool:
    """Bind the independently graded patch to the accepted gate and revision."""
    gate = audit.get('last_gate') or {}
    return bool(
        audit.get('submission_status') == 'SUBMITTED_GATE_VALID'
        and gate.get('accepted') is True
        and gate.get('accepted_patch_sha256') == patch_sha256
        and ledger.get('workspace_sha256')
        and gate.get('accepted_workspace_sha256') == ledger['workspace_sha256']
        and type(ledger.get('workspace_revision')) is int
        and gate.get('accepted_revision') == ledger['workspace_revision'])


## 14. gp03_eval_freeze.py

In [ ]:
%%gp03_source gp03_eval_freeze.py
"""Verify the accepted organizer evidence and describe the immutable GP03 inputs."""

import json
from pathlib import Path
from zipfile import ZipFile

from gp03_public_runtime import GP03_PROPOSED_PAIRS, canonical_hash, environment_fingerprint, junit_counts, proposed_pair_lock, sha256_file, verify_zip


OFFICIAL_ZIP = "GP03_Official_Worker_Acceptance_20261109T093704Z_4c374696.zip"
OFFICIAL_SHA = "798d3005be6d519fb2c2449ecf204562c6621ab63e7d66586881d3ef06e8a943"


def normalize_serving_command(command):
    """Preserve effective serving arguments while abstracting the output workspace path."""
    normalized = []
    for value in command:
        alias, eq, target = value.partition("=")
        if eq and alias in {"main_lora", "tool_lora"}:
            if not target.endswith(f"/frozen_submission/adapters/{alias}"):
                raise ValueError(f"Unexpected LoRA path for {alias}")
            value = f"{alias}=@frozen_submission/adapters/{alias}"
        normalized.append(value)
    return normalized


def audit_official(path):
    if sha256_file(path) != OFFICIAL_SHA:
        raise ValueError("The official worker ZIP differs from the accepted run")
    verify_zip(path)
    with ZipFile(path) as archive:
        def read(member):
            return json.loads(archive.read(member))

        summary, prepared, seed = map(read, ("GP03_Official_Worker_Report.json", "Official_Prepare_Report.json", "Official_Seed_Report.json"))
        lock = read("CPU_Execution_Lock.json")
        if (summary != {"status": "GP03_OFFICIAL_WORKER_READY", "prepared": "GP03_OFFICIAL_SANDBOXES_READY",
                        "seed_probe": "GP03_OFFICIAL_WORKER_READY", "official_sandboxes_ready": 9,
                        "held_out_model_calls": 0, "development_model_calls": 1,
                        "effective_seed_verified": True, "evaluation_frozen": False}
                or lock != proposed_pair_lock(GP03_PROPOSED_PAIRS)
                or prepared["status"] != summary["prepared"] or prepared["model_calls"] or prepared["held_out_agent_attempts"]
                or seed["status"] != summary["seed_probe"] or seed["model_calls"] != 1 or seed["held_out_agent_attempts"]
                or seed["observed_seed"] != 42 or seed["max_output_tokens"] != 2048):
            raise ValueError("The accepted run does not close the official sandbox and seed gates")
        profile = read("GPU_Public_Test_Environment.json")
        env = environment_fingerprint(profile)
        if env != prepared["gpu_environment_sha256"] or prepared["hardware"] != ["NVIDIA L4"] * 4:
            raise ValueError("Official GPU environment or accelerator evidence changed")
        pairs, manifests = [], {}
        if [r["task_id"] for r in prepared["rows"]] != [p["task_id"] for p in GP03_PROPOSED_PAIRS]:
            raise ValueError("Official probes are out of frozen task order")
        for plan, row in zip(GP03_PROPOSED_PAIRS, prepared["rows"], strict=True):
            task = plan["task_id"]
            parent = f"official_sandboxes/{task}"
            raw = archive.read(f"{parent}/Public_Check_Manifest.json")
            manifest = json.loads(raw)
            from hashlib import sha256
            digest = sha256(raw).hexdigest()
            attestation = read(f"{parent}/artifacts/Official_Worker_Attestation.json")
            identity = read(f"{parent}/artifacts/Official_Sandbox_Identity.json")
            state = read(f"{parent}/artifacts/agent/result.json")
            supervisor = read(f"{parent}/supervisor.json")
            counts = list(junit_counts(archive.read(f"{parent}/artifacts/trusted_checks/check_0001/pytest/junit.xml")))
            receipt = read(f"{parent}/artifacts/trusted_checks/check_0001/Public_Check_Receipt.json")
            if (row != read(f"{parent}/Probe_Result.json") or row["status"] != "READY"
                    or row["arm"] != plan["arm_order"][0] or row["model_requests_forwarded"] or row["denied_model_requests"]
                    or row["public_counts"] != counts or counts != [plan["expected_collected"], plan["expected_collected"], 0, 0, 0]
                    or digest != row["check_manifest_sha256"] or attestation["check_manifest_sha256"] != digest
                    or attestation["status"] != "OFFICIAL_PUBLIC_GATE_ACCEPTED"
                    or manifest["environment_sha256"] != env or identity["environment_sha256"] != env
                    or identity["host_head"] != identity["agent_head"] or identity["baseline_tag"] != identity["host_head"]
                    or identity["source_sha256"] != manifest["baseline_source_sha256"]
                    or receipt["actual_nodeids_sha256"] != manifest["expected_nodeids_sha256"]
                    or receipt["outcome"] != "PASS" or receipt["reason"]
                    or any(manifest[key] != plan[key] for key in ("task_id", "repo", "base_commit", "snapshot_sha256", "native_head", "native_tree"))
                    or not state["worker_complete"] or state["official_agent_call_counter"] or state["reference_fields_provided_to_agent"]
                    or not supervisor["trusted_process_cleanup_verified"] or supervisor["returncode"] or supervisor["timed_out"]
                    or archive.read(f"{parent}/Transport.jsonl")):
                raise ValueError(f"Official sandbox evidence disagrees for {task}")
            pairs.append({**plan, "official_manifest_sha256": digest, "visible_manifest_sha256": attestation["visible_manifest_sha256"],
                          "tool_signatures_sha256": attestation["tool_signatures_sha256"], "public_counts": counts})
            manifests[task] = raw
        transport = [json.loads(line) for line in archive.read("development_seed/Transport.jsonl").splitlines()]
        response = read("development_seed/artifacts/Official_Model_Probe.json")
        if (len(transport) != 2 or transport[0]["event"] != "VLLM_REQUEST" or transport[0]["forwarded_seed"] != 42
                or transport[0]["max_output_tokens"] != 2048 or transport[1]["event"] != "VLLM_RESPONSE"
                or transport[1]["status"] != 200 or not transport[1]["completed"]
                or response["task_id"] != "rich_2943" or response["response_delivered_to_agent"] or response["model_calls"] != 1
                or seed["probe"]["status"] != "READY" or seed["probe"]["task_id"] != "rich_2943"):
            raise ValueError("The actual development transport and response do not verify the seed")
        server = read("Server_Config.json")
        runtime = read("GPU_Runtime.json")
        if (server["server"]["config"]["max_model_len"] != 32768 or server["server"]["config"]["tensor_parallel_size"] != 4
                or runtime["status"] != "RUNTIME_MATCHED"):
            raise ValueError("The effective Gemma runtime differs from the planned configuration")
        evidence = {"pairs": pairs, "hardware": prepared["hardware"], "environment_sha256": env,
                    "official_sources": prepared["official_sources"], "organizer_runtime": runtime,
                    "server_config": server["server"]["config"],
                    "effective_server_command": normalize_serving_command(server["server"]["command"]),
                    "model": read("Gemma_Checkpoint.json"), "graph_inventory": read("Graph_Preflight.json"),
                    "accepted_source_pins": read("Source_Pins.json"), "seed_probe": transport[0],
                    "observed_preflight_seconds": prepared["elapsed_seconds"], "observed_startup_seconds": server["startup_seconds"]}
    return evidence, manifests


def find_official_zip(working, override=None):
    paths = [Path(override)] if override else [working / OFFICIAL_ZIP, *sorted(Path("/kaggle/input").rglob(OFFICIAL_ZIP))]
    for path in paths:
        if path.is_file() and sha256_file(path) == OFFICIAL_SHA:
            return path
    raise FileNotFoundError(f"Make the exact {OFFICIAL_ZIP} available in Kaggle working or as a mounted input")


## 15. gp03_eval_capture.py

In [ ]:
%%gp03_source gp03_eval_capture.py
"""Common B/C live patch capture and budget enforcement for GP03 evaluation."""

import asyncio
from contextvars import ContextVar
import functools
import hashlib
import inspect
import json
from pathlib import Path
import threading
import time

from gp03_public_runtime import PAIR_BUDGET, candidate_bytes, canonical_hash, write_json

_TOOL_PHASE = ContextVar("gp03_tool_phase", default=None)
ORDINARY_STOP = "GP03_ORDINARY_DEADLINE_REACHED"
MODEL_STOP = "GP03_SHARED_MODEL_BUDGET_REACHED"


def command_deadline(started, overall):
    if _TOOL_PHASE.get() == "ordinary":
        return started + PAIR_BUDGET["ordinary_seconds"]
    return started + overall - 3.0


def write_bytes(path, data):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_bytes(data)
    temporary.replace(path)


class EvaluationCapture:
    def __init__(self, root, baseline, production_root, boundary, evidence):
        self.root = root
        self.baseline = baseline
        self.production_root = production_root
        self.boundary = boundary
        self.evidence = evidence
        self.evidence.mkdir(parents=True, exist_ok=True)
        self.sequence = 0
        self.lock = threading.RLock()

    def checkpoint(self, source):
        with self.lock:
            controller = self.boundary.controller
            ledger = controller.ledger.export()
            patch, manifest = candidate_bytes(self.root, roots=(self.production_root,), base_ref=self.baseline)
            patch_path = self.evidence / f"production_{manifest['sha256']}.patch"
            if not patch_path.exists():
                write_bytes(patch_path, patch)
            self.sequence += 1
            row = {"sequence": self.sequence, "source": source, "elapsed_seconds": controller.deadline.elapsed(),
                   "workspace_sha256": ledger["workspace_sha256"], "workspace_revision": ledger["workspace_revision"],
                   "patch_file": patch_path.name, "candidate": manifest}
            write_json(self.evidence / "Latest_Checkpoint.json", {**row, "ledger": ledger, "audit": self.boundary.export_audit()})
            with (self.evidence / "Checkpoints.jsonl").open("a", encoding="utf-8") as stream:
                stream.write(json.dumps(row, sort_keys=True) + "\n")

    def capture_error(self, exc):
        write_json(self.evidence / "Capture_Error.json", {"type": type(exc).__name__, "detail": str(exc)[:1000]})

    def wrap(self, tools):
        wrapped = {}
        for name, function in tools.items():
            @functools.wraps(function)
            def invoke(*args, _name=name, _function=function, **kwargs):
                token = _TOOL_PHASE.set("final" if _name == "submit_patch" else "ordinary")
                try:
                    return _function(*args, **kwargs)
                finally:
                    _TOOL_PHASE.reset(token)
                    try:
                        self.checkpoint("AFTER_" + _name)
                    except Exception as exc:
                        self.capture_error(exc)
                        raise
            wrapped[name] = invoke
        return wrapped


def attach_evaluation_capture(state, context, manager, sandbox_id, config, root, baseline, manifest, tools):
    boundary = state["boundary"]
    signatures = canonical_hash({name: str(inspect.signature(fn)) for name, fn in sorted(tools.items())})
    if (signatures != config["gp03_tool_signatures_sha256"]
            or boundary.controller.ledger.manifest.sha256 != config["gp03_visible_manifest_sha256"]):
        raise RuntimeError("Held-out tool or visible-check identity differs from its official acceptance")
    capture = EvaluationCapture(root, baseline, manifest["production_root"], boundary, state["artifacts"] / "capture")
    # Keep the accepted controller empty until the agent produces evidence or submits.
    capture.checkpoint("ATTACH")
    create_tools = context.create_tools

    @functools.wraps(create_tools)
    def create_captured_tools():
        return capture.wrap(create_tools())

    stop = manager.stop

    def stop_with_capture(target_id):
        try:
            if target_id == sandbox_id and target_id in manager.sandboxes:
                capture.checkpoint("BEFORE_TEARDOWN")
        except Exception as exc:
            capture.capture_error(exc)
            raise
        finally:
            stop(target_id)

    manager.stop = stop_with_capture
    context.create_tools = create_captured_tools
    state.update(ready=True, capture=capture, create_tools=create_captured_tools)
    write_json(state["artifacts"] / "Official_Worker_Attestation.json", {
        "status": "GP03_EVALUATION_ATTACHED", "task_id": config["task_id"], "arm": config["gp03_arm"],
        "check_manifest_sha256": config["gp03_manifest_sha256"], "visible_manifest_sha256": config["gp03_visible_manifest_sha256"],
        "tool_signatures_sha256": signatures, "environment_sha256": manifest["environment_sha256"],
        "protected_verifier_calls": 0,
    })
    return create_captured_tools()


def install_evaluation_model_guard():
    from google.adk.models.lite_llm import LiteLlm
    import gp03_official_hook as hook

    original = LiteLlm.generate_content_async
    if not inspect.isasyncgenfunction(original):
        raise RuntimeError("Pinned official model observer is expected to be an async generator")

    @functools.wraps(original)
    async def guarded(*args, **kwargs):
        state = hook._ATTACHMENT
        if state is None or state["mode"] != "evaluation" or not state["ready"]:
            raise RuntimeError("Model request reached an unattached evaluation worker")
        boundary = state["boundary"]
        controller = boundary.controller
        remaining = controller.deadline.remaining("ordinary")
        if remaining <= 0:
            raise hook.ProbeStop(ORDINARY_STOP)
        if controller.budget.used_model_calls >= PAIR_BUDGET["max_model_calls"]:
            raise hook.ProbeStop(MODEL_STOP)
        boundary.sync_model_calls(controller.budget.used_model_calls + 1)
        try:
            async with asyncio.timeout(remaining):
                async for event in original(*args, **kwargs):
                    yield event
        except TimeoutError as exc:
            if controller.deadline.remaining("ordinary") <= 0:
                raise hook.ProbeStop(ORDINARY_STOP) from exc
            raise

    LiteLlm.generate_content_async = guarded
    return lambda: setattr(LiteLlm, "generate_content_async", original)


def record_runner_result(artifacts, patch, trace):
    if not isinstance(patch, str):
        raise RuntimeError("Unexpected patch type from the official runner")
    raw = patch.encode("utf-8")
    capture = artifacts / "capture"
    capture.mkdir(parents=True, exist_ok=True)
    write_bytes(capture / "Official_Returned.patch", raw)
    if trace is not None:
        trace.save(artifacts / "agent" / "trace.json", format="legacy")
    return {"raw_patch_bytes": len(raw), "raw_patch_sha256": hashlib.sha256(raw).hexdigest()}


## 16. gp03_eval_run.py

In [ ]:
%%gp03_source gp03_eval_run.py
"""Launch the frozen GP03 pairs in order and preserve each attempt for CPU grading."""

import argparse
import fcntl
import json
import math
import os
from pathlib import Path
import shutil
import sys
import tempfile
import time
import traceback
from zipfile import ZipFile

from gp03_eval_capture import MODEL_STOP, ORDINARY_STOP
from gp03_eval_freeze import audit_official, find_official_zip, normalize_serving_command
from gp03_official_driver import MODEL_IDS, checkpoint_inventory, public_rows, runtime_source_pins
from gp03_organizer_runtime import (_offline_environment, _require_l4, _stage_pre_server, find_baseline,
                                    local_server, prepare_gpu_runtime, require_ready_model, restore_frozen_submission,
                                    RunClock, supervise_worker)
from gp03_public_runtime import (artifact_paths, environment_fingerprint, environment_profile, load_gp03_inputs,
                                 GP03_DEVELOPMENT, seal_zip, sha256_file, verify_zip, write_json)
from gp03_seed_proxy import seed_proxy

SOURCE = Path(__file__).resolve().parent


def read_json(path):
    return json.loads(Path(path).read_text())


def load_settings(path):
    settings = read_json(path)
    for name in ("root", "working", "competition", "wheelhouse", "session", "freeze"):
        settings[name] = Path(settings[name]).resolve()
    if sha256_file(settings["freeze"]) != settings["freeze_sha256"]:
        raise ValueError("The evaluation freeze differs from its pinned bytes")
    lock = read_json(settings["freeze"])
    for name, digest in lock["source_sha256"].items():
        if sha256_file(SOURCE / name) != digest:
            raise ValueError(f"Frozen evaluation source changed for {name}")
    settings["lock"] = lock
    return settings


def wave_pairs(settings):
    lock = settings["lock"]
    wave = settings["wave"]
    if type(wave) is not int or wave not in (1, 2):
        raise ValueError("Select evaluation wave 1 or 2")
    positions = lock["waves"][str(wave)]["pair_positions"]
    return [row for row in lock["pairs"] if row["pair_index"] in positions]


def attempt_path(root, task, arm, number):
    return root / "attempts" / task / arm / f"attempt_{number:02d}"


def attempt_rows(root, pairs, freeze_sha):
    rows = []
    for pair in pairs:
        for arm in pair["arm_order"]:
            for number in (1, 2):
                folder = attempt_path(root, pair["task_id"], arm, number)
                if not folder.exists():
                    continue
                path = folder / "Attempt_Result.json"
                if not path.is_file():
                    raise RuntimeError(f"An interrupted attempt requires review before resume: {pair['task_id']} {arm} #{number}")
                row = read_json(path)
                if (row["task_id"], row["arm"], row["attempt_number"], row["freeze_sha256"]) != (pair["task_id"], arm, number, freeze_sha):
                    raise ValueError("Saved attempt identity differs from the freeze")
                if row["candidate_bytes"]:
                    candidate = folder / "Production_Candidate.patch"
                    if candidate.stat().st_size != row["candidate_bytes"] or sha256_file(candidate) != row["candidate_sha256"]:
                        raise ValueError("Saved production candidate differs from its recorded bytes")
                rows.append(row)
    return rows


def scheduled_arms(settings, pairs):
    rows = attempt_rows(settings["root"], pairs, settings["freeze_sha256"])
    by_arm = {(row["task_id"], row["arm"]): row for row in rows}
    retries = set(settings["retry_infra"])
    pending = []
    matched = set()
    for pair in pairs:
        task = pair["task_id"]
        for arm in pair["arm_order"]:
            key = f"{task}:{arm}"
            previous = by_arm.get((task, arm))
            if key in retries:
                matched.add(key)
                if not previous or previous["status"] != "INFRA_ERROR" or not previous["retryable_infrastructure"] or previous["attempt_number"] != 1:
                    raise ValueError(f"The recorded evidence does not authorize a replacement for {key}")
                pending.append((pair, arm, 2))
            elif not previous:
                pending.append((pair, arm, 1))
    if retries != matched:
        raise ValueError("A requested retry is outside the current wave")
    return pending


def require_prior_wave(settings):
    if settings["wave"] == 1:
        return
    path = settings.get("wave_1_zip")
    if not path:
        raise ValueError("Set WAVE_1_EVIDENCE_ZIP to the sealed first-wave evidence before running wave 2")
    verify_zip(Path(path))
    with ZipFile(path) as archive:
        report = json.loads(archive.read("Wave_Report.json"))
    if (report["freeze_sha256"] != settings["freeze_sha256"] or report["wave"] != 1
            or report["attempted_arms"] != 10 or report["status"] != "WAVE_ATTEMPTS_ARCHIVED"):
        raise ValueError("Wave 2 requires all first-wave arm records from this exact freeze")


def restore_wave(settings, pairs):
    path = settings.get("resume_wave_zip")
    if not path:
        return
    root = settings["root"]
    if (root / "attempts").exists():
        raise ValueError("The wave already has local attempts. Clear the resume ZIP override to use the saved attempts.")
    index = verify_zip(Path(path))
    prefixes = {attempt_path(Path(), p["task_id"], arm, i).as_posix() + "/"
                for p in pairs for arm in ("B", "C") for i in (1, 2)}
    with ZipFile(path) as archive, tempfile.TemporaryDirectory(prefix=".gp03_restore_", dir=root) as temporary:
        report = json.loads(archive.read("Wave_Report.json"))
        if (report["wave"] != settings["wave"] or report["freeze_sha256"] != settings["freeze_sha256"]
                or archive.read(settings["freeze"].name) != settings["freeze"].read_bytes()):
            raise ValueError("Resume evidence is outside the current freeze or wave")
        stage = Path(temporary)
        for name in index:
            if not name.startswith("attempts/"):
                continue
            if not any(name.startswith(prefix) for prefix in prefixes):
                raise ValueError("Resume evidence contains an unexpected attempt path")
            target = stage / name
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(name) as source, target.open("xb") as stream:
                shutil.copyfileobj(source, stream)
        if attempt_rows(stage, pairs, settings["freeze_sha256"]) != report["attempts"]:
            raise ValueError("Resume attempt records disagree with the sealed wave report")
        if (stage / "attempts").is_dir():
            (stage / "attempts").replace(root / "attempts")


def prepare(settings, clock):
    lock = settings["lock"]
    session = settings["session"]
    with clock.phase("Verify frozen evidence and runtime"):
        official = find_official_zip(settings["working"], settings.get("official_zip"))
        accepted, manifests = audit_official(official)
        for name in ("pairs", "hardware", "environment_sha256", "official_sources", "organizer_runtime",
                     "server_config", "effective_server_command", "model", "graph_inventory"):
            if accepted[name] != lock[name]:
                raise ValueError(f"Accepted official evidence differs from the evaluation freeze for {name}")
        competition = settings["competition"]
        load_gp03_inputs(competition, lock["pairs"], GP03_DEVELOPMENT, lock["tasks_sha256"])
        public = public_rows(competition, lock["pairs"])
        original = find_baseline(settings.get("baseline_zip"), settings["working"])
        template, serving = restore_frozen_submission(original, session / "frozen_submission")
        if serving != lock["server_config"]:
            raise ValueError("Original archived serving profile differs from the accepted configuration")
        if _require_l4() != lock["hardware"]:
            raise RuntimeError("The evaluation requires the four accepted L4 devices")
        _offline_environment()
        prepare_gpu_runtime(settings["wheelhouse"], session, settings["allow_offline_install"])
        if (read_json(session / "GPU_Runtime.json") != lock["organizer_runtime"]
                or runtime_source_pins() != lock["official_sources"]
                or environment_fingerprint(environment_profile()) != lock["environment_sha256"]):
            raise ValueError("The GPU environment, organizer wheels or worker source drifted after acceptance")
        if checkpoint_inventory(serving["model"]) != lock["model"]["files"]:
            raise ValueError("The model checkpoint bytes differ from the freeze")
        _stage_pre_server(serving, competition, public, tuple(public), session)
        expected_graph = {task: lock["graph_inventory"][task] for task in public}
        if read_json(session / "Graph_Preflight.json") != expected_graph:
            raise ValueError("The held-out graph inventory changed after official acceptance")
    return template, serving, public, manifests


def budget_stop(text):
    return any(reason.lower() in text.lower() for reason in (
        ORDINARY_STOP, MODEL_STOP, "Absolute repair deadline exhausted", "Shared agent-tool budget exhausted",
        "model-call budget exhausted", "maximum tool calls", "maximum model calls", "budget exceeded",
    ))


def capture_result(folder, state, supervisor, pair, arm, number, freeze_sha):
    task = pair["task_id"]
    records = [json.loads(line) for line in (folder / "Transport.jsonl").read_text().splitlines() if line]
    requests = [r for r in records if r["event"] == "VLLM_REQUEST"]
    responses = [r for r in records if r["event"] == "VLLM_RESPONSE"]
    denied = [r for r in records if r["event"] == "POST_DENIED"]
    request_numbers = [r["request_number"] for r in requests]
    if (any(r["forwarded_seed"] != 42 or r["max_output_tokens"] != 2048 for r in requests)
            or sorted(request_numbers) != list(range(1, len(requests) + 1)) or len(requests) > 60):
        raise ValueError("A forwarded request differs from the frozen seed, token or call budget")
    attempted = folder / "artifacts"
    identity = attempted / "Official_Worker_Attestation.json"
    attestation = read_json(identity) if identity.is_file() else {}
    latest = attempted / "capture/Latest_Checkpoint.json"
    checkpoint = read_json(latest) if latest.is_file() else {}
    stop = state.get("agent_error") or state.get("worker_error") or ""
    capture_failure = (attempted / "capture/Capture_Error.json").is_file()
    attached = (attestation.get("status") == "GP03_EVALUATION_ATTACHED"
                and attestation.get("check_manifest_sha256") == pair["official_manifest_sha256"]
                and attestation.get("visible_manifest_sha256") == pair["visible_manifest_sha256"]
                and attestation.get("tool_signatures_sha256") == pair["tool_signatures_sha256"])
    cleanup = supervisor["trusted_process_cleanup_verified"]
    upstream_failure = any(r.get("status") is not None and r["status"] >= 500 for r in responses) or any(r["event"] == "UPSTREAM_ERROR" for r in records)
    deadline = supervisor["timed_out"] or budget_stop(stop)
    # A transport timeout at the ordinary cutoff is part of the matched budget.
    if upstream_failure and not deadline and supervisor["elapsed_seconds"] < 840:
        status, reason = "INFRA_ERROR", "UPSTREAM_TRANSPORT_FAILURE"
    elif not (attached and cleanup and checkpoint and not capture_failure):
        status, reason = "INFRA_ERROR", "ATTACHMENT_CAPTURE_OR_CLEANUP_HOLD"
    elif deadline:
        status, reason = "DEADLINE", "HARD_WORKER_CAP" if supervisor["timed_out"] else "ORDINARY_OR_CALL_BUDGET"
    elif (supervisor["returncode"] == 0 and state.get("worker_complete") and not stop
          and checkpoint["source"] == "BEFORE_TEARDOWN" and state.get("sandbox_source_setup_ran")
          and state.get("reference_fields_provided_to_agent") is False
          and requests and not denied and sorted(request_numbers) == sorted(r["request_number"] for r in responses)
          and all(r["status"] == 200 and r["completed"] for r in responses)):
        status, reason = "OBSERVED", "OFFICIAL_RUN_RETURNED"
    else:
        status, reason = "INFRA_ERROR", "UNVERIFIED_OFFICIAL_TERMINATION"
    candidate = checkpoint.get("candidate", {})
    if candidate.get("bytes"):
        path = attempted / "capture" / checkpoint["patch_file"]
        if (path.stat().st_size, sha256_file(path)) != (candidate["bytes"], candidate["sha256"]):
            raise ValueError("Live checkpoint candidate bytes differ from their capture")
        shutil.copyfile(path, folder / "Production_Candidate.patch")
    audit = checkpoint.get("audit", {})
    ledger = checkpoint.get("ledger", {})
    receipts = ledger.get("receipts", [])
    events = ledger.get("events", [])
    budget = audit.get("budget", {})
    raw = attempted / "capture/Official_Returned.patch"
    if raw.is_file() and (state.get("raw_patch_bytes"), state.get("raw_patch_sha256")) != (raw.stat().st_size, sha256_file(raw)):
        raise ValueError("Official runner patch differs from its recorded identity")
    return {"task_id": task, "repo": pair["repo"], "arm": arm, "attempt_number": number,
            "attempt_id": f"{freeze_sha[:12]}_{task}_{arm}_{number:02d}", "freeze_sha256": freeze_sha,
            "status": status, "termination": reason, "agent_stop": str(stop)[:1000],
            "elapsed_seconds": round(supervisor["elapsed_seconds"], 3), "cleanup_verified": cleanup,
            "retryable_infrastructure": reason == "UPSTREAM_TRANSPORT_FAILURE" and cleanup and number == 1,
            "model_calls": len(requests), "tool_calls": budget.get("used_tool_calls"),
            "transport_defaulted_caps": sum(r.get("output_cap_source") == "PROXY_DEFAULT" for r in requests),
            "trusted_checks": len(receipts) if checkpoint else None,
            "proactive_rechecks": sum(r["provenance"] == "controller_recheck" for r in receipts) if checkpoint else None,
            "natural_exposures": sum(e["type"] == "QUALIFIED_PASS_INVALIDATED_BY_RELEVANT_EDIT" for e in events) if checkpoint else None,
            "observed_regressions": sum(e["type"] == "OBSERVED_PASS_TO_FAIL_AFTER_EDIT" for e in events) if checkpoint else None,
            "submission_status": audit.get("submission_status", "NOT_RECORDED"), "candidate_capture_source": checkpoint.get("source"),
            "candidate_bytes": candidate.get("bytes", 0), "candidate_sha256": candidate.get("sha256"),
            "raw_patch_sha256": state.get("raw_patch_sha256"), "protected_grade": "NOT_RUN"}


def run_attempt(settings, pair, arm, number, template, serving, public, raw_manifest, server):
    root, session, lock = settings["root"], settings["session"], settings["lock"]
    task = pair["task_id"]
    folder = attempt_path(root, task, arm, number)
    folder.mkdir(parents=True, exist_ok=False)
    private = session / f"{task}_{arm}_{number:02d}"
    private.mkdir()
    manifest = folder / "Public_Check_Manifest.json"
    manifest.write_bytes(raw_manifest)
    if sha256_file(manifest) != pair["official_manifest_sha256"]:
        raise ValueError("Evaluation public manifest is outside the freeze")
    public_file = private / "Public_Task.json"
    write_json(public_file, public)
    artifacts = folder / "artifacts"
    transport_path = folder / "Transport.jsonl"
    transport_path.touch(exist_ok=False)
    started = time.monotonic()
    with seed_proxy(model_ids=(*MODEL_IDS, serving["model"]), event_path=transport_path, upstream_url=server["base_url"],
                    timeout=lock["budget"]["ordinary_seconds"], max_requests=lock["budget"]["max_model_calls"]) as proxy:
        config = {key: template[key] for key in (
            "version", "model_name", "model_path", "agent_time_minutes", "max_llm_calls", "max_tool_calls", "max_output_tokens",
            "model_context_tokens", "max_file_lines", "max_file_chars", "max_stdout_chars", "command_timeout_seconds",
            "verification_timeout_seconds", "verification_source_sha256", "agent_runner_source_sha256", "verify_task_parameters",
        )}
        config.update(task_id=task, task_expected_repo=pair["repo"], task_expected_commit=pair["base_commit"],
                      snapshot_pins={task: pair["snapshot_sha256"]}, run_id=f"gp03_{task}_{arm}_{number:02d}",
                      tasks_sha256=lock["tasks_sha256"], competition_root=str(settings["competition"]), artifact_dir=str(artifacts),
                      agent_sandbox_dir=str(private / "sandboxes"), public_task_path=str(public_file),
                      submission_dir=str(session / "frozen_submission"), gp03_manifest=str(manifest),
                      gp03_manifest_sha256=pair["official_manifest_sha256"], gp03_mode="evaluation", gp03_arm=arm,
                      gp03_worker_cap=lock["budget"]["worker_seconds"], gp03_tool_signatures_sha256=pair["tool_signatures_sha256"],
                      gp03_visible_manifest_sha256=pair["visible_manifest_sha256"],
                      starter_server={"base_url": proxy.base_url, "config": {**serving, "port": proxy.server_port}})
        config_path = private / "Config.json"
        write_json(config_path, config)
        env = dict(os.environ, GP03_WORKER_STARTED_MONOTONIC=str(started), PYTHONPATH=str(SOURCE),
                   NO_PROXY="127.0.0.1,localhost", no_proxy="127.0.0.1,localhost")
        env.pop("GP02C_TRUSTED_PYTEST_REGISTRY", None)
        supervisor = supervise_worker((sys.executable, "-u", str(SOURCE / "gp03_official_worker.py"), "--config", str(config_path)),
                                      cwd=private, evidence_root=folder, overall_seconds=config["gp03_worker_cap"], env=env)
    result_path = artifacts / "agent/result.json"
    state = read_json(result_path) if result_path.is_file() else {}
    row = capture_result(folder, state, supervisor, pair, arm, number, settings["freeze_sha256"])
    write_json(folder / "Attempt_Result.json", row)
    if row["cleanup_verified"]:
        shutil.rmtree(private)
    return row


def report_wave(settings, pairs, *, hold=None):
    root = settings["root"]
    rows = attempt_rows(root, pairs, settings["freeze_sha256"])
    selected = {(row["task_id"], row["arm"]): row for row in rows}
    complete = sum(all((p["task_id"], arm) in selected for arm in ("B", "C")) for p in pairs)
    report = {"schema": "gp03.wave-evidence.v1", "freeze_sha256": settings["freeze_sha256"], "wave": settings["wave"],
              "status": "WAVE_ATTEMPTS_ARCHIVED" if complete == len(pairs) else "WAVE_INCOMPLETE",
              "planned_pairs": len(pairs), "archived_pairs": complete, "attempted_arms": len(selected), "attempts": rows,
              "infrastructure_affected_arms": sum(row["status"] == "INFRA_ERROR" for row in selected.values()),
              "held_out_model_calls": sum(row["model_calls"] for row in rows), "protected_grading": "PENDING_CPU",
              "hold": hold, "launch_wall_minutes_reported": settings["l4_wall_minutes_available"]}
    write_json(root / "Wave_Report.json", report)
    return report


def run_wave(settings):
    root, session, lock = settings["root"], settings["session"], settings["lock"]
    root.mkdir(parents=True, exist_ok=True)
    session.mkdir(parents=True, exist_ok=False)
    clock = RunClock()
    pairs = wave_pairs(settings)
    with (root / ".run.lock").open("a+") as guard:
        fcntl.flock(guard, fcntl.LOCK_EX | fcntl.LOCK_NB)
        hold = None
        try:
            require_prior_wave(settings)
            restore_wave(settings, pairs)
            pending = scheduled_arms(settings, pairs)
            if pending:
                minutes = settings["l4_wall_minutes_available"]
                required = 15 * len(pending) + lock["budget"]["reserve_minutes_per_wave"]
                if type(minutes) not in (int, float) or not math.isfinite(minutes) or minutes < required:
                    raise ValueError(f"Enter at least {required} available L4 wall minutes for these remaining arms and overhead")
                template, serving, public, manifests = prepare(settings, clock)
                with local_server(serving, session, clock, {}) as server:
                    if normalize_serving_command(server["command"]) != lock["effective_server_command"]:
                        raise ValueError("The effective vLLM launch command differs from the accepted run")
                    for model in (serving["model"], *MODEL_IDS):
                        require_ready_model(server["base_url"], model)
                    write_json(root / "Active_Runtime.json", {"freeze_sha256": settings["freeze_sha256"],
                               "effective_server_command": normalize_serving_command(server["command"]), "startup_elapsed_seconds": clock.elapsed()})
                    for pair, arm, number in pending:
                        if clock.elapsed() + lock["budget"]["worker_seconds"] + 120 > minutes * 60:
                            raise RuntimeError("Remaining reported L4 time cannot cover the next arm and export reserve")
                        task = pair["task_id"]
                        with clock.phase(f"Arm {arm} attempt {number}", task=task):
                            row = run_attempt(settings, pair, arm, number, template, serving, public[task], manifests[task], server)
                            clock.log(f"{task} {arm} | {row['status']} | calls {row['model_calls']} | candidate {row['candidate_bytes']} bytes")
                        report_wave(settings, pairs)
                        if row["status"] == "INFRA_ERROR":
                            raise RuntimeError(f"{task} {arm} is on infrastructure hold. Review the sealed evidence before resuming.")
        except Exception as exc:
            hold = str(exc)
            (root / "Latest_Hold.txt").write_text(traceback.format_exc(), encoding="utf-8")
            clock.log(f"GP03 HOLD | {exc}")
        finally:
            try:
                report = report_wave(settings, pairs, hold=hold)
            except Exception as exc:
                report = {"status": "WAVE_INCOMPLETE", "wave": settings["wave"], "freeze_sha256": settings["freeze_sha256"],
                          "hold": f"Interrupted attempt evidence needs review: {exc}"}
                write_json(root / "Wave_Report.json", report)
            shutil.copyfile(settings["freeze"], root / settings["freeze"].name)
            write_json(root / "Latest_Session_Timing.json", {"elapsed_seconds": clock.elapsed(), "phases": clock.phases})
            for name in ("gemma_server.log", "GPU_Runtime.json", "Graph_Preflight.json"):
                if (session / name).is_file():
                    shutil.copyfile(session / name, root / name)
            target = settings["working"] / f"{root.name}_{session.name}.zip"
            seal_zip(target, artifact_paths(root, excluded={".run.lock"}))
            print(report["status"], target, flush=True)
    return target


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("settings")
    args = parser.parse_args()
    run_wave(load_settings(args.settings))


## 17. gp03_eval_grade.py

In [ ]:
%%gp03_source gp03_eval_grade.py
"""Independent CPU grading and paired scoring for the sealed GP03 wave evidence."""

import argparse
from importlib import metadata
import json
from pathlib import Path
import subprocess
import sys
import tarfile
import tempfile
import time
from zipfile import ZipFile

from gp03_eval_run import attempt_path, load_settings
from gp03_eval_score import paired_results
from gp03_grade_runtime import _g15d_mounted_wheels, _g15d_offline_env, _G15D_WORKER_ENTRY, classify_candidate, delivery_gate_matches
from gp03_organizer_runtime import supervise_worker
from gp03_public_runtime import (WorkspaceFingerprinter, artifact_paths, candidate_bytes, is_production, load_gp03_inputs, GP03_DEVELOPMENT,
                                 prepare_pytest_config, run_git, seal_zip, sha256_file, stage_public_snapshot, verify_zip, write_json)

SOURCE = Path(__file__).resolve().parent
CORE = {"adk-eval-core", "adk-submission", "swegemma", "google-adk"}


class CandidateDoesNotApply(ValueError):
    pass


def load_waves(settings, folder):
    lock = settings["lock"]
    assignments = {p["task_id"]: p for p in lock["pairs"]}
    accepted = {}
    waves = set()
    for item in settings["evaluation_zips"]:
        path = Path(item)
        verify_zip(path)
        with ZipFile(path) as archive:
            report = json.loads(archive.read("Wave_Report.json"))
            if (report["freeze_sha256"] != settings["freeze_sha256"]
                    or archive.read(settings["freeze"].name) != settings["freeze"].read_bytes()):
                raise ValueError("Evaluation evidence differs from the exact freeze")
            wave = report["wave"]
            if wave not in (1, 2) or wave in waves:
                raise ValueError("Provide one latest archive per evaluation wave")
            waves.add(wave)
            numbers = lock["waves"][str(wave)]["pair_positions"]
            for row in report["attempts"]:
                task, arm, number = row["task_id"], row["arm"], row["attempt_number"]
                pair = assignments[task]
                key = task, arm
                expected_id = f"{settings['freeze_sha256'][:12]}_{task}_{arm}_{number:02d}"
                if (pair["pair_index"] not in numbers or arm not in ("B", "C") or number not in (1, 2)
                        or row["attempt_id"] != expected_id or row["freeze_sha256"] != settings["freeze_sha256"]):
                    raise ValueError("Archive contains an attempt outside its frozen assignment")
                previous = accepted.get(key)
                if ((number == 1 and previous) or (number == 2 and (not previous or not previous["retryable_infrastructure"]
                        or previous["attempt_number"] != 1 or previous["status"] != "INFRA_ERROR"))):
                    raise ValueError("Archive violates the frozen replacement-attempt policy")
                member = attempt_path(Path(), task, arm, number).as_posix()
                if json.loads(archive.read(f"{member}/Attempt_Result.json")) != row:
                    raise ValueError("Attempt summary differs from its sealed record")
                if row["candidate_bytes"] and row["status"] in ("OBSERVED", "DEADLINE"):
                    dest = folder / expected_id
                    dest.mkdir()
                    for name in ("Production_Candidate.patch", "Public_Check_Manifest.json",
                                 "artifacts/capture/Latest_Checkpoint.json", "artifacts/capture/Official_Returned.patch"):
                        if f"{member}/{name}" in archive.namelist():
                            target = dest / Path(name).name
                            target.write_bytes(archive.read(f"{member}/{name}"))
                    patch = dest / "Production_Candidate.patch"
                    if (patch.stat().st_size, sha256_file(patch)) != (row["candidate_bytes"], row["candidate_sha256"]):
                        raise ValueError("Sealed production patch identity differs")
                    if sha256_file(dest / "Public_Check_Manifest.json") != pair["official_manifest_sha256"]:
                        raise ValueError("Attempt's public definition differs from official acceptance")
                accepted[key] = row
    return list(accepted.values())


def prepare_grader(settings, private):
    from packaging.utils import parse_wheel_filename

    lock = settings["lock"]
    pin = lock["organizer_runtime"]
    expected = lock["grader"]["runtime_versions"]
    env = _g15d_offline_env(private)
    env["PYTHONPATH"] = str(SOURCE)
    versions = {}
    for name in CORE:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    record = {"runtime_source": "PREINSTALLED_ORGANIZER_RUNTIME", "versions": expected}
    if versions != {name: expected[name] for name in CORE}:
        core = {row["name"]: row["sha256"] for row in pin["wheel_pins"]
                if parse_wheel_filename(row["name"])[0] in CORE}
        wheels = _g15d_mounted_wheels(settings["competition"], settings["wheelhouse"], core)
        overlay = private / "site-packages"
        command = (sys.executable, "-m", "pip", "--isolated", "--disable-pip-version-check", "install", "--no-index",
                   "--no-deps", "--only-binary=:all:", "--target", str(overlay), *(str(p) for p in wheels))
        install = supervise_worker(command, cwd=private, evidence_root=private / "install", overall_seconds=180, env=env)
        if install["returncode"] or install["timed_out"] or not install["trusted_process_cleanup_verified"]:
            raise RuntimeError("Offline installation of the pinned CPU evaluator runtime failed")
        env["PYTHONPATH"] = f"{overlay}:{SOURCE}"
        record.update(runtime_source="TEMPORARY_MOUNTED_WHEEL_OVERLAY", runtime_wheels={p.name: sha256_file(p) for p in wheels})
    config = private / "Runtime_Check_Config.json"
    output = private / "runtime_check"
    write_json(config, {"artifact_dir": str(output), "verification_source_sha256": lock["official_sources"]["verifier"],
                        "verify_task_parameters": lock["grader"]["verify_task_parameters"]})
    command = (sys.executable, "-u", str(SOURCE / "gp03_private_grader.py"), "--config", str(config), "--preflight")
    checked = supervise_worker(command, cwd=private, evidence_root=private / "runtime_probe", overall_seconds=90, env=env)
    if checked["returncode"] or checked["timed_out"] or not checked["trusted_process_cleanup_verified"]:
        raise RuntimeError("Pinned CPU evaluator failed its source and import check")
    versions = json.loads((output / "runtime_preflight.json").read_text())["packages"]
    actual = {name.lower().replace("_", "-"): version for name, version in versions.items()}
    if any(actual.get(name) != version for name, version in expected.items()):
        raise RuntimeError("CPU evaluator versions differ from the accepted Section 15D runtime")
    return env, record


def verify_production_candidate(competition, pair, row, inputs, temporary):
    task = pair["task_id"]
    snapshot = competition / "snapshots" / f"{task}.tgz"
    if sha256_file(snapshot) != pair["snapshot_sha256"]:
        raise ValueError("CPU replay snapshot bytes differ from the freeze")
    root, _, _, _ = stage_public_snapshot(snapshot, temporary / "public", pair["repo"], time.monotonic() + 300)
    manifest = json.loads((inputs / "Public_Check_Manifest.json").read_text())
    prepare_pytest_config(root)
    if (WorkspaceFingerprinter(root).snapshot().sha256 != manifest["baseline_source_sha256"]
            or run_git(root, ["rev-parse", "HEAD"]).stdout.decode().strip() != pair["native_head"]):
        raise ValueError("CPU patch applicability checkout differs from the accepted baseline")
    candidate = inputs / "Production_Candidate.patch"
    stats = run_git(root, ["apply", "--numstat", "-z", str(candidate)]).stdout.split(b"\0")
    paths = [item.split(b"\t", 2)[-1].decode("utf-8", "surrogateescape") for item in stats if item]
    if not paths or not all(is_production(path, (manifest["production_root"],)) for path in paths):
        raise ValueError("The independently graded patch contains paths outside the production scope")
    if run_git(root, ["apply", "--check", str(candidate)], allowed=(0, 1, 128)).returncode:
        raise CandidateDoesNotApply("Patch does not apply to the pinned native checkout")
    checkpoint = json.loads((inputs / "Latest_Checkpoint.json").read_text())
    if checkpoint["candidate"]["sha256"] != row["candidate_sha256"]:
        raise ValueError("Pre-teardown capture and exported candidate disagree")
    raw = inputs / "Official_Returned.patch"
    equivalent = False
    if raw.is_file() and raw.stat().st_size and sha256_file(raw) == row["raw_patch_sha256"]:
        if run_git(root, ["apply", "--check", str(raw)], allowed=(0, 1, 128)).returncode == 0:
            run_git(root, ["apply", str(raw)])
            projected, _ = candidate_bytes(root, roots=(manifest["production_root"],), base_ref=pair["native_head"])
            equivalent = projected == candidate.read_bytes()
    if row["status"] == "OBSERVED" and not equivalent:
        raise ValueError("The captured production patch could not be matched to the official returned patch")
    return {"official_return_equivalent": equivalent,
            "gate_identity_matches": equivalent and delivery_gate_matches(checkpoint["audit"], checkpoint["ledger"], row["candidate_sha256"]),
            "capture_source": checkpoint["source"], "applicable": True}


def protected_grade(settings, pair, row, inputs, private, env):
    lock = settings["lock"]
    task = pair["task_id"]
    folder = private / row["attempt_id"]
    folder.mkdir()
    result = {"attempt_id": row["attempt_id"], "task_id": task, "arm": row["arm"],
              "candidate_sha256": row["candidate_sha256"], "protected_grade": "INFRA_ERROR", "verified_delivery": False}
    start = time.monotonic()
    try:
        identity = verify_production_candidate(settings["competition"], pair, row, inputs, folder)
        result["patch_identity"] = identity
        config = folder / "Grade_Config.json"
        write_json(config, {"competition_root": str(settings["competition"]), "artifact_dir": str(folder / "private_results"),
                            "sandbox_dir": str(folder / "sandbox"), "agent_patch_path": str(inputs / "Production_Candidate.patch"),
                            "tasks_sha256": lock["tasks_sha256"], "verification_source_sha256": lock["official_sources"]["verifier"],
                            "verify_task_parameters": lock["grader"]["verify_task_parameters"],
                            "development_task_ids": [task], "development_metadata": {task: {k: pair[k] for k in ("repo", "base_commit")}},
                            "snapshot_pins": {task: pair["snapshot_sha256"]},
                            "command_timeout_seconds": lock["grader"]["command_seconds"],
                            "verification_timeout_seconds": lock["grader"]["command_seconds"]})
        command = (sys.executable, "-u", "-c", _G15D_WORKER_ENTRY, str(SOURCE / "gp03_private_grader.py"),
                   str(lock["grader"]["command_seconds"]), "--config", str(config), "--task-id", task, "--phase", "candidate")
        worker = supervise_worker(command, cwd=folder, evidence_root=folder / "supervisor", overall_seconds=lock["grader"]["worker_seconds"], env=env)
        if worker["returncode"] or worker["timed_out"] or not worker["trusted_process_cleanup_verified"]:
            raise RuntimeError("Organizer evaluator process held or timed out")
        path = folder / "private_results" / "evaluator_only" / task / "candidate" / "result.json"
        data = json.loads(path.read_text())
        setup = data.get("workspace_setup") or {}
        if (data.get("instance_id"), data.get("phase"), data.get("snapshot_sha256"), setup.get("snapshot_head_before"),
                setup.get("snapshot_git_tree")) != (task, "candidate", pair["snapshot_sha256"], pair["native_head"], pair["native_tree"]):
            raise RuntimeError("Protected verifier result differs from the frozen native snapshot")
        verdict = classify_candidate(data)
        if verdict == "VERIFICATION_HOLD":
            raise RuntimeError("Organizer verifier produced incomplete or invalid JUnit evidence")
        counts = data["junit"]
        result.update(protected_grade="PASS" if verdict == "PATCH_VERIFIED" else "FAIL",
                      evaluator_exit_code=data["test_exit_code"],
                      protected_counts={k: counts[k] for k in ("tests", "passed", "failures", "errors", "skipped")},
                      verified_delivery=verdict == "PATCH_VERIFIED" and identity["gate_identity_matches"])
    except CandidateDoesNotApply:
        result.update(protected_grade="NOT_RUN", candidate_disposition="NONAPPLICABLE")
    except (OSError, RuntimeError, ValueError, KeyError, subprocess.SubprocessError, tarfile.TarError) as exc:
        # Export only a bounded error category. Verifier stdout and test names stay private.
        result["hold"] = type(exc).__name__
    result["elapsed_seconds"] = round(time.monotonic() - start, 3)
    return result


def run_grading(settings):
    output = settings["root"]
    output.mkdir(parents=True, exist_ok=False)
    lock = settings["lock"]
    pairs = {p["task_id"]: p for p in lock["pairs"]}
    grades = []
    with tempfile.TemporaryDirectory(prefix="gp03_cpu_grader_") as temp:
        private = Path(temp)
        (private / "inputs").mkdir()
        load_gp03_inputs(settings["competition"], lock["pairs"], GP03_DEVELOPMENT, lock["tasks_sha256"])
        attempts = load_waves(settings, private / "inputs")
        gradeable = [row for row in attempts if row["candidate_bytes"] and row["status"] in ("OBSERVED", "DEADLINE")]
        env, profile = prepare_grader(settings, private) if gradeable else (None, {"runtime_source": "NO_APPLICABLE_CANDIDATES"})
        write_json(output / "CPU_Grader_Runtime.json", profile)
        for row in gradeable:
            print(f"{row['task_id']} {row['arm']} | independent protected grade", flush=True)
            grade = protected_grade(settings, pairs[row["task_id"]], row, private / "inputs" / row["attempt_id"], private, env)
            grades.append(grade)
            write_json(output / "Protected_Grades.json", grades)
            print(f"{row['task_id']} {row['arm']} | {grade['protected_grade']} | {grade['elapsed_seconds']:.1f}s", flush=True)
    write_json(output / "Protected_Grades.json", grades)
    write_json(output / "Paired_Results.json", paired_results(lock["pairs"], attempts, grades))
    write_json(output / "Grade_Report.json", {"freeze_sha256": settings["freeze_sha256"], "input_zips": [
        {"name": Path(p).name, "sha256": sha256_file(Path(p))} for p in settings["evaluation_zips"]],
        "attempts": len(attempts), "protected_grade_counts": {status: sum(g["protected_grade"] == status for g in grades)
                                                               for status in ("PASS", "FAIL", "NOT_RUN", "INFRA_ERROR")},
        "model_calls": 0, "protected_material_exported": False})
    return seal_zip(settings["working"] / (output.name + ".zip"), artifact_paths(output))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("settings")
    args = parser.parse_args()
    run_grading(load_settings(args.settings))


## 18. gp03_eval_score.py

In [ ]:
%%gp03_source gp03_eval_score.py
"""Frozen paired analysis for independently graded GP03 repair attempts."""

from collections import defaultdict
from random import Random


BOOTSTRAP_DRAWS = 10_000
BOOTSTRAP_SEED = 20261108


def protected_endpoint(attempt, grade):
    if not attempt or attempt["status"] not in {"OBSERVED", "DEADLINE"}:
        return None
    if attempt["candidate_bytes"] == 0:
        return 0
    if (not grade or grade["attempt_id"] != attempt["attempt_id"]
            or grade["candidate_sha256"] != attempt["candidate_sha256"]):
        return None
    if grade.get("candidate_disposition") == "NONAPPLICABLE" and grade["protected_grade"] == "NOT_RUN":
        return 0
    return {"PASS": 1, "FAIL": 0}.get(grade["protected_grade"])


def paired_results(pairs, attempts, grades):
    """Use a single accepted record per arm. Missing and infrastructure grades stay unresolved."""
    grade_by_attempt = {row["attempt_id"]: row for row in grades}
    by_arm = {(row["task_id"], row["arm"]): row for row in attempts}
    if len(grade_by_attempt) != len(grades) or len(by_arm) != len(attempts):
        raise ValueError("Duplicate attempt or grade in the accepted analysis input")
    allowed = {(row["task_id"], arm) for row in pairs for arm in ("B", "C")}
    if set(by_arm) - allowed or set(grade_by_attempt) - {row["attempt_id"] for row in attempts}:
        raise ValueError("Analysis input contains an assignment outside the frozen cohort")
    strata = defaultdict(list)
    paired_counts = dict(both_pass=0, b_only=0, c_only=0, neither_pass=0)
    table = []
    lower = upper = 0
    for pair in pairs:
        task = pair["task_id"]
        arms = {}
        for arm in ("B", "C"):
            attempt = by_arm.get((task, arm))
            grade = grade_by_attempt.get(attempt["attempt_id"]) if attempt else None
            arms[arm] = {"protected_outcome": protected_endpoint(attempt, grade),
                         "protected_grade": grade["protected_grade"] if grade else "NOT_RUN",
                         "attempt_status": attempt["status"] if attempt else "MISSING",
                         "verified_delivery": bool(grade and grade.get("verified_delivery")),
                         "attempt_id": attempt["attempt_id"] if attempt else None,
                         "worker_seconds": attempt.get("elapsed_seconds") if attempt else None,
                         "model_calls": attempt.get("model_calls") if attempt else None,
                         "tool_calls": attempt.get("tool_calls") if attempt else None,
                         "trusted_checks": attempt.get("trusted_checks") if attempt else None,
                         "proactive_rechecks": attempt.get("proactive_rechecks") if attempt else None,
                         "natural_exposures": attempt.get("natural_exposures") if attempt else None,
                         "observed_regressions": attempt.get("observed_regressions") if attempt else None}
        b, c = (arms[a]["protected_outcome"] for a in ("B", "C"))
        lower += (0 if c is None else c) - (1 if b is None else b)
        upper += (1 if c is None else c) - (0 if b is None else b)
        if b is not None and c is not None:
            paired_counts[{(1, 1): "both_pass", (1, 0): "b_only", (0, 1): "c_only", (0, 0): "neither_pass"}[(b, c)]] += 1
            strata[pair["repo"]].append(c - b)
        table.append({"task_id": task, "repo": pair["repo"], "arm_order": pair["arm_order"], **arms})
    assigned = len(pairs)
    observed = sum(paired_counts.values())
    difference = interval = discordance_upper = None
    if observed:
        difference = (paired_counts["c_only"] - paired_counts["b_only"]) / observed
        rng = Random(BOOTSTRAP_SEED)
        groups = [strata[key] for key in sorted(strata)]
        draws = sorted(sum(sum(rng.choices(group, k=len(group))) for group in groups) / observed for _ in range(BOOTSTRAP_DRAWS))
        def quantile(q):
            at = q * (len(draws) - 1)
            lo = int(at)
            return draws[lo] + (at - lo) * (draws[min(lo + 1, len(draws) - 1)] - draws[lo])
        interval = [quantile(.025), quantile(.975)]
        if paired_counts["b_only"] + paired_counts["c_only"] == 0:
            discordance_upper = 1 - .05 ** (1 / observed)
    exposure = {}
    for arm in ("B", "C"):
        values = [row[arm]["natural_exposures"] for row in table if row[arm]["natural_exposures"] is not None]
        exposure[arm] = {"assigned": assigned, "observable_attempts": len(values), "arms_with_exposure": sum(v > 0 for v in values),
                         "qualifying_pass_edit_events": sum(values)}
    return {"assigned_pairs": assigned, "scored_pairs": observed, "unresolved_pairs": assigned - observed,
            "paired_counts": paired_counts, "protected_success_difference_c_minus_b": difference,
            "paired_repo_stratified_bootstrap_95": interval, "bootstrap_draws": BOOTSTRAP_DRAWS, "bootstrap_seed": BOOTSTRAP_SEED,
            "assigned_pair_bounds": [lower / assigned, upper / assigned] if assigned else None,
            "zero_discordance_upper_95": discordance_upper, "natural_exposure": exposure, "rows": table}


## Run the selected stage

In [ ]:
if RUN_MODE not in {"EVAL_GPU", "GRADE_CPU"}:
    raise ValueError("Set RUN_MODE to EVAL_GPU or GRADE_CPU")
if RUN_MODE == "GRADE_CPU" and not EVALUATION_ZIPS:
    raise ValueError("Set EVALUATION_ZIPS to the sealed wave ZIP or ZIPs you want to grade")

if RUN_MODE == "EVAL_GPU":
    root = WORK_DIR / f"GP03_Evaluation_{GP03_FREEZE_SHA256[:12]}_Wave{WAVE}"
    session = WORK_DIR / f"GP03_Session_{tag}"
    driver = "gp03_eval_run.py"
    bundle = WORK_DIR / f"{root.name}_{session.name}.zip"
else:
    root = WORK_DIR / f"GP03_Independent_CPU_Grades_{tag}"
    session = root
    driver = "gp03_eval_grade.py"
    bundle = WORK_DIR / (root.name + ".zip")

settings = GP03_SOURCE / "Settings.json"
settings.write_text(json.dumps({
    "working": str(WORK_DIR), "root": str(root), "session": str(session), "wave": WAVE,
    "freeze": str(GP03_FREEZE), "freeze_sha256": GP03_FREEZE_SHA256,
    "competition": str(COMPETITION_DIR), "wheelhouse": str(WHEELHOUSE_DIR),
    "official_zip": str(OFFICIAL_ACCEPTANCE_ZIP) if OFFICIAL_ACCEPTANCE_ZIP else None,
    "baseline_zip": str(ORIGINAL_GP02B_ZIP) if ORIGINAL_GP02B_ZIP else None,
    "wave_1_zip": str(WAVE_1_EVIDENCE_ZIP) if WAVE_1_EVIDENCE_ZIP else None,
    "resume_wave_zip": str(RESUME_WAVE_ZIP) if RESUME_WAVE_ZIP else None,
    "l4_wall_minutes_available": L4_WALL_MINUTES_AVAILABLE, "retry_infra": RETRY_INFRA,
    "allow_offline_install": ALLOW_OFFLINE_WHEEL_INSTALL,
    "evaluation_zips": [str(path) for path in EVALUATION_ZIPS],
}, indent=2))
if not bundle.is_file():
    subprocess.run([sys.executable, "-u", str(GP03_SOURCE / driver), str(settings)], check=True)


## Download the sealed evidence

In [ ]:
sys.path.insert(0, str(GP03_SOURCE))
from gp03_public_runtime import offer_zip_download
if bundle.is_file():
    offer_zip_download(bundle, auto_download=AUTO_DOWNLOAD_ZIP)
else:
    print("Review the previous execution output for the prelaunch hold.")
